# 2026 IEEE International Conference on Big Data

In [ ]:
# ═══════════════════ 판 · 코드 지문 ═══════════════════
# 이 노트북의 판 번호입니다. 파일 이름의 숫자와 반드시 같아야 합니다.
VERSION = 110
NB_PATH = 'icml_run110.ipynb'

import os as _os, re as _re, glob as _gl, json as _js, time as _tm, hashlib as _hl

# ── 1) 파일명과 판 번호가 어긋나지 않았는지
if not _os.path.exists(NB_PATH):
    print('⚠ icml_run110.ipynb 가 이 폴더에 없습니다. confieee2026_a/ 에서 여셨는지 보십시오.')

# ── 2) 나보다 새 판이 옆에 있는가 (있으면 그걸 여셔야 합니다)
_mine, _newer = VERSION, []
for _f in _gl.glob('icml_run*.ipynb'):
    _m = _re.fullmatch(r'icml_run(\d+)\.ipynb', _os.path.basename(_f))
    if _m and int(_m.group(1)) > _mine:
        _newer.append(int(_m.group(1)))

# ── 3) 코드 지문 — 코드 셀 원문만 해시합니다(실행 결과는 뺌 → 돌려도 안 바뀝니다)
def _fingerprint(p):
    try:
        _nb = _js.load(open(p, encoding='utf-8'))
    except Exception:
        return None
    _s = ''.join(''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code')
    return _hl.md5(_s.encode('utf-8')).hexdigest()[:10]

import sys as _sys0, os as _os0
_sys0.path.insert(0, _os0.path.abspath('.'))
from cdtlib import env as _cenv0
CODE_HASH, CELL_HASHES = _cenv0.nb_fingerprint(NB_PATH)
if CODE_HASH is None:
    CODE_HASH = _fingerprint(NB_PATH)
    CELL_HASHES = []
if CELL_HASHES:
    _os0.makedirs('out', exist_ok=True)
    _js0 = __import__('json')
    _js0.dump({'노트북': NB_PATH, '판': VERSION, '지문': CODE_HASH,
               '셀별': CELL_HASHES},
              open(_os0.path.join('out', '_지문.json'), 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)
STAMP = dict(notebook=NB_PATH, version=VERSION, code_hash=CODE_HASH,
             run_at=_tm.strftime('%Y-%m-%d %H:%M:%S'))

print('icml_run  v%d   코드지문 %s   %s' % (VERSION, CODE_HASH or '(못 읽음)', STAMP['run_at']))

if _newer:
    print('  ⚠ 더 새 판이 있습니다: icml_run%d.ipynb  ← 이것을 여십시오'
          % max(_newer))

_stamp_path = _os.path.join('out', '_판.json')
_prev = None
if _os.path.exists(_stamp_path):
    try:
        _prev = _js.load(open(_stamp_path, encoding='utf-8'))
    except Exception:
        _prev = None

if _prev is None:
    print('  지난 실행 기록이 없습니다.')
elif _prev.get('code_hash') == CODE_HASH:
    print('  지난 실행(%s · v%s)과 코드가 같습니다.'
          % (_prev.get('run_at'), _prev.get('version')))
else:
    print('  코드가 지난 실행 이후 바뀌었습니다  (지난 판 v%s · %s)'
          % (_prev.get('version'), _prev.get('code_hash')))
    if _prev.get('version') == VERSION:
        print('  ⚠ 그런데 판 번호가 그대로입니다. 파일명과 VERSION 을 함께 올리십시오.')

_os.makedirs(_os.path.dirname(_stamp_path), exist_ok=True)
_js.dump(STAMP, open(_stamp_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

In [ ]:
# ═══════════════════ 어느 기계에서 도는가 ═══════════════════
# 컴퓨터가 둘(윈도우 CPU · 우분투 GPU)이라 결과마다 이것을 남깁니다.
# CPU 와 GPU 는 같은 시드로도 소수점 아래가 다릅니다 — 표 하나는 한 기계에서 뽑습니다.
import sys as _sys, os as _os2
_sys.path.insert(0, _os2.path.abspath('.'))
from cdtlib import env as cenv

ENV = cenv.stamp()
print('실행 환경')
cenv.show()

# 라이브러리 지문 — 노트북 지문만으로는 부족합니다.
# 컴퓨터가 둘이면 노트북은 같은데 cdtlib 한 파일만 옛것인 일이 생깁니다.
print()
LIB_HASH, LIB_PER = cenv.show_lib('.')
ENV['라이브러리지문'] = LIB_HASH
ENV['라이브러리'] = LIB_PER
if ENV.get('장치') == 'cuda':
    import torch as _t
    _cc = _t.cuda.get_device_capability(0)
    _arch = _t.cuda.get_arch_list()
    _need = 'sm_%d%d' % _cc
    print()
    print('  연산능력 %s · 빌드된 아키에 %s %s' % (_cc, _need, '있음' if _need in _arch else '← 없음!'))
    if _need not in _arch:
        print('  ⚠ 이 torch 휠에는 이 GPU 용 커널이 없습니다. 연산이 실패합니다.')
        print('    RTX 50 계열(sm_120)은 cu128 이상 휠이 필요합니다:')
        print('    pip install torch --index-url https://download.pytorch.org/whl/cu128')
    else:
        _x = _t.randn(2000, 2000, device='cuda')
        print('  행렬곱 확인 %.1f  (여기서 안 죽으면 진짜 됩니다)' % float((_x @ _x).sum()))
        del _x; _t.cuda.empty_cache()

# [79] cdtlib 이 이 노트북과 같은 판인가 — **여기서 1초 만에 잡습니다.**
#   v78 에서 노트북만 옮기고 cdtlib 은 옛 판이 남아 CDT 125개가 전부 죽었는데,
#   나머지 다섯 모형으로 표가 멀쩡히 채워져서 한 시간 뒤에야 알았습니다.
cenv.check_lib()   # [100] models.py 고침 E·F 로 지문이 바뀌었습니다 — 위에 찍힌 새 지문을 기록하십시오

# [76] 결정적 연산 — **미리** 확인합니다. 학습 도중에 터지면 실행이 통째로 날아갑니다.
_DET = cenv.check_deterministic()
ENV['결정적연산'] = cenv.DETERMINISTIC_NOTE[0]


# ═══ [100] 시작 게이트 — 조용히 틀리게 두지 않습니다 ═══
import inspect as _ins
from cdtlib import models as _mchk
assert 'has_valid' in _ins.getsource(_mchk.TARNet.fit), \
    'models.py 에 [고침 E] 가 없습니다 — 패치판을 복사하십시오 (재적합 1에폭 동결 버그)'
assert '_uplift_order' in dir(_mchk), \
    'models.py 에 [고침 F] 가 없습니다 — 패치판을 복사하십시오 (Qini 동점 인공물)'
try:
    import econml as _ec
    print('econml %s — cforest 는 진짜 인과숲으로 돕니다' % _ec.__version__)
except ImportError:
    raise RuntimeError('econml 이 없습니다. cforest 가 T-learner 복제로 돌아 표가 오염됩니다.\n'
                       '  pip install econml   후 다시 시작하십시오.\n'
                       '  (일부러 대체로 돌리려면 이 assert 를 주석 처리 — 단, 논문에 cforest 를 싣지 마십시오)')


## 이 노트북이 하는 일

**이 노트북 하나만 위에서 아래로 실행하면 됩니다.** 터미널에서 따로 돌릴 것이 없습니다.

| 절 | 하는 일 | 이전 위치 |
|---|---|---|
| 1 | 원자료 점검 | icml61 |
| 2 | 원자료 → 표준 `.npz` (ihdp 100판 · jobs 10판 · synt · real) | icml61 |
| 3 | 왕복 테스트 · 무결성 · 처치정의 · 전처리 내역 | icml61 |
| 4 | **합성데이터 v2 생성** | `tools/gen_empservicesynt.py` |
| 5 | v2 표준화 (`.npz` + 반사실 정답 `.npz`) | 신규 |
| 6 | 기술분석 (변수·특이값·균형·누출·중복·시간·결과) | icml62 |
| 7 | 그림 2장 · 표 저장 | icml62 |
| 8 | **벤치마크 정본 대조** (IHDP·Jobs) | `tools/verify_benchmarks.py` |
| 9 | **합성 v2 검증** (수정확인·이질성·교란·양성성·대리충분성·J곡선) | `tools/verify_synt_v2.py` |
| 10 | 최종 점검표 → **`out/00_결과요약.txt` 한 개로 저장** | 신규 |

**다 돌린 뒤에는 `out/00_결과요약.txt` 파일 하나만 보내시면 됩니다.** 마지막 셀이 만듭니다.

아래 **설정 셀의 스위치**로 무거운 단계를 건너뛸 수 있습니다.
이미 만들어 둔 산출물이 있으면 기본값에서 자동으로 건너뜁니다.
처음부터 다시 만들려면 해당 스위치를 `True` 로 바꾸십시오.

**멈춘 것 같으면 화면의 마지막 `[시:분:초]` 줄을 보십시오.** 모든 단계가 시각·소요·메모리를 찍습니다.
주피터는 **대기 중인 셀도 `[*]` 로 보입니다.** 지금 실제로 도는 셀은 그 시각 줄 다음 셀입니다.

이 노트북의 최대 메모리 사용량은 약 1.5 GB 입니다. 그보다 훨씬 크게 올라간다면
다른 프로그램이 메모리를 쓰고 있어 스와핑이 일어나는 중입니다 — 그때는 `CFG['LIGHT'] = True`.

In [ ]:
# ══════════════════════════ 설정 ══════════════════════════
import os, sys, time, json, warnings
sys.path.insert(0, os.path.abspath('.'))        # confieee2026/ 에서 실행
sys.path.insert(0, os.path.abspath('tools'))    # tools/*.py 를 모듈로 부르기 위해
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from cdtlib import dataio as dio, descstat as ds, jobcode as jc
from cdtlib.config import (ensure_dirs, show, PROJECT_ROOT, CONF_ROOT,
                           DIR_RAW, DIR_STD, DIR_SYNT, DIR_OUT, RAW)

CFG = dict(
    # 표준화
    N_IHDP        = 100,          # 빠른 점검이 필요하면 5 로 줄여도 됩니다
    N_JOBS        = 10,
    FORCE_STD     = False,        # True 면 표준 npz 를 전부 다시 만듭니다

    # 합성데이터 v2  (약 3분)
    SYNT_N        = 8000,
    SYNT_SEED     = 20260821,
    SYNT_HIDDEN   = 0.35,         # 0 이면 가정 1(이력의 대리 충분성)이 정확히 성립하는 판
    SYNT_REPS     = 40,           # 반사실 몬테카를로 복제 수
    FORCE_SYNT    = False,        # True 면 v2 를 다시 생성합니다

    # 검증
    FETCH_REF     = False,        # True 면 IHDP 정본을 인터넷에서 다시 내려받습니다
    RUN_DESC      = True,         # 기술분석
    LIGHT         = False,

    # ── 단계 4·6 학습 (§7). 켜 둡니다. 이미 돌린 실행은 결과를 읽어 오므로
    #    다시 돌려도 시간이 안 듭니다(같은 run_id 는 다시 학습하지 않습니다).
    TRAIN         = True,
    TRAIN_TIMING  = True,       # 먼저 IHDP 1판 1시드로 벽시계를 잽니다
    TRAIN_MODELS  = ('naive', 'slearner', 'tlearner', 'cforest', 'tarnet', 'cdt'),
    TRAIN_DATA    = ('ihdp', 'jobs', 'empservicesyntv2', 'empservicereal'),
    # [76] 시드 하나로는 못 씁니다. v75 에서 **완전히 같은 설정을 세 번 돌려**
    #   합성 v2 √PEHE 가 29.85 / 30.88 / 34.68 이 나왔습니다(GPU 연산 순서 흔들림).
    #   재려던 효과보다 흔들림이 컸습니다. 결정적 연산을 켜고, 시드도 늘립니다.
    # [78] 자료마다 반복의 뜻이 다릅니다.
    #   ihdp·jobs — **판(realization)** 이 곧 문헌의 반복입니다. 100판·10판이 표준이고,
    #     v77 까지 1판만 돌린 √PEHE 0.48 은 공표된 TARNet 0.95·CFR 0.76 과 비교 자체가
    #     안 됩니다(쉬운 판을 뽑았을 뿐). 심사자가 첫 번째로 짚을 자리입니다.
    #   실데이터·합성 — 판이 하나뿐이라 **시드**로 반복합니다.
    TRAIN_SEEDS   = (0, 1, 2, 3, 4),
    # [82] ihdp·jobs 도 시드 3개. 지금까지 판마다 **시드 하나**라 짝지은 차 +0.207 안에
    #   학습 난수의 운이 섞여 있었습니다. 판 안에서 시드를 평균하면 그 성분이 빠집니다.
    SEEDS_BY      = {'ihdp': (0, 1, 2), 'jobs': (0, 1, 2)},
    REPS_BY       = {'ihdp': 100, 'jobs': 10},
    STRICT_DET    = True,       # 결정적 연산. 못 켜는 환경이면 자동으로 꺼지고 알려 줍니다
    TRAIN_REPS    = 1,          # REPS_BY 에 없는 자료의 기본값
    TRAIN_H       = 1,          # 관측창 0:6개월 1:12개월 2:24개월
    TRAIN_Y       = 'Y3',
    # [77] 실데이터는 결과를 하나 더 봅니다. Y3(기대소득)은 63.9%가 0 인 뾰족한 분포라
    #   valid 제곱오차가 모형선택 기준으로 잘 안 듭니다(조기종료가 1~3에폭에 걸립니다).
    #   Y1(12개월 취업 여부)은 고용정책 문헌의 표준 결과이자 훨씬 순한 분포입니다.
    TRAIN_Y_EXTRA = {'empservicereal': ('Y1',)},
    # 실행 이름은 **노트북 판 번호**로 묶입니다(v73_ihdp_cdt_s0).
    # 코드를 고치면 판이 올라가고 그때 자동으로 새로 학습합니다 — 꼬리표가 필요 없습니다.
    # 같은 판을 다시 돌리면 이미 만든 결과를 읽어 오므로 기다릴 일이 없습니다.
    CDT_EPOCHS    = 200,
    CDT_K         = 30,         # 궤적보다 길면 자동으로 T 로 줄입니다
    CDT_HIDDEN    = 128,
    CDT_LAMBDA_A  = 0.3,
    # [74] 학습률·드롭아웃은 **자료마다 valid 손실로 고릅니다**(test 는 안 봅니다).
    #   73 판은 IHDP 에서 고른 1e-2 하나를 네 자료에 썼고, 실데이터에서 조기종료가
    #   1~2에폭에 걸렸습니다(best_epoch 2). 후보를 줄이면 그만큼 빨리 끝납니다.
    CDT_LR        = (3e-4, 1e-3, 3e-3, 1e-2),
    CDT_DROPOUT   = (0.0, 0.1, 0.3),   # 실데이터(n=1345)는 더 센 규제가 필요합니다
    CDT_ACTION    = 'binary',   # 반사실 대비의 행동 부호. 'code' 는 직종별 이질성 전용
    # [78] 후보를 손으로 고른 목록으로 줍니다(격자를 다 곱하면 144개가 됩니다).
    #   head='twin' 은 처치 팔마다 결과머리를 따로 두어 τ = y1(s) − y0(s) 가
    #   **상태표현만의 매끈한 함수**가 되는 판입니다. 실데이터에서 τ 가 시드마다
    #   상쇄되던 문제(앙상블 Qini 989→222)를 겨눈 후보입니다.
    CDT_CONFIGS   = [
        dict(head='additive', hidden=128, n_layer=3, lr=3e-3, dropout=0.1),
        dict(head='additive', hidden=128, n_layer=3, lr=1e-2, dropout=0.0),
        dict(head='additive', hidden=128, n_layer=3, lr=1e-3, dropout=0.1),
        dict(head='additive', hidden=64,  n_layer=2, lr=1e-3, dropout=0.1),
        dict(head='additive', hidden=64,  n_layer=2, lr=3e-3, dropout=0.3),
        dict(head='additive', hidden=32,  n_layer=1, lr=3e-3, dropout=0.1),
        dict(head='twin',     hidden=128, n_layer=3, lr=3e-3, dropout=0.1),
        dict(head='twin',     hidden=64,  n_layer=2, lr=1e-3, dropout=0.1),
        dict(head='twin',     hidden=32,  n_layer=1, lr=3e-3, dropout=0.1),
    ],
    # 붓스트랩 — 실데이터 Qini 의 표본오차. 이것을 안 재면 순위표를 읽을 수 없습니다.
    BOOT_B        = 2000,
    # [82] 표본크기 곡선 — 참값이 있는 합성 v2 에서 **학습표본만 줄이고 시험표본은 고정**합니다.
    #   실데이터가 「아무도 구별 안 된다」로 끝난 이유를 통제된 자료에서 수치로 보이는 실험이고,
    #   기관에 「몇 명을 모아야 하는가」를 답해 주는 값이라 정부 트랙에서 바로 쓰입니다.
    NCURVE        = True,
    NCURVE_N      = (250, 500, 1000, 2000, 4000, 5600),
    # [83] 작은 n 에서 곡선이 들쭉날쭉했습니다(n=1000 이 n=500 보다 나빴습니다).
    #   그 구간은 분산이 커서 3시드로는 모자랍니다. 6개로 올립니다 — 작은 n 은 빨리 돕니다.
    NCURVE_SEEDS  = (0, 1, 2, 3, 4, 5),
    # [80] **이미 돌린 실행을 그대로 씁니다.** 79 판에서 750개를 실패 없이 돌렸으니
    #   분석만 더하는 판에서 그것을 또 학습시킬 이유가 없습니다. run_id 를 이 판 번호로
    #   찾습니다. 처음부터 다시 학습시키려면 None 으로 두십시오.
    # [85] 옛 실행을 읽어 오는 기능은 **없앴습니다**. 돌릴 때마다 전부 새로 학습합니다.
    #   시간을 아끼려고 넣었던 장치인데, 자료나 코드가 바뀌면 옛 숫자가 새 표에 조용히
    #   들어옵니다. 84판까지 두 번 겪었습니다(v78 옛 라이브러리 · v83 옛 코호트).
    #   느려도 이편이 맞습니다 — 표에 있는 숫자는 전부 이번에 나온 것입니다.

    # 실데이터 코호트.  None = 기존 2,000명 · 'r16k' = 확대 16,000명
    #   확대판은 상태열이 111개인데 기존 분석은 70개를 씁니다. 아래 STATE_KEEP 으로
    #   **기존 70개만** 골라 써서, 바뀌는 것이 사람 수 하나가 되게 합니다.
    REAL_COHORT   = 'r16k',

    # ── 단계 8 (§8). 그림 3(K 절단)과 표 5(구조 소거)
    # [81] 그림 3 은 답이 나왔습니다 — 시드 5개에서 두 곡선 모두 「읽을 수 없음」.
    #   3시드에서 「읽을 수 있음」이 나왔던 것이 우연이었습니다. 다시 돌릴 이유가 없어
    #   꺼 둡니다. 다시 보시려면 True 로.
    # [90] **스위치를 둘로 쪼갰습니다.** 89판은 `ABLATE=False` 하나로 §8-1(K 곡선)과
    #   §8-2(구조 소거 = **표 5**)를 함께 껐습니다. 그런데 85판에서 옛 실행 재사용을
    #   없앴으므로, 끄면 표 5 가 **아예 안 만들어집니다.** 논문에서 「MOT 를 빼면
    #   IHDP 0.488 → 4.205」·「이중머리를 빼면 합성 32.06 → 61.47」이 제안 구조를
    #   정당화하는 가장 강한 증거인데 그것이 통째로 빠집니다.
    ABLATE        = True,       # §8 전체 주스위치
    ABLATE_KCURVE = False,      # §8-1 K 곡선(옛 그림 3) — 81판에서 「읽을 수 없음」으로 결론. 꺼 둡니다
    ABLATE_STRUCT = True,       # §8-2 구조 소거(표 5) — **논문에 반드시 필요합니다**
    ABLATE_DATA   = 'empservicesyntv2',      # 참값이 있어야 √PEHE 를 낼 수 있습니다
    ABLATE_K      = (1, 6, 12, 30, 60, 120),
    ABLATE_SEEDS  = (0, 1, 2, 3, 4),  # 소거·K절단도 시드를 여러 개 — 한 판은 못 믿습니다
    # [77] 그림 3 을 두 곡선으로. 상태 82열 중 68열이 `상태_생애_*누적경력기간`·`효과잔존량`,
    #   곧 **이미 생애 전체를 요약해 둔 값**입니다. 그래서 K=1 도 이미 이력을 다 봅니다
    #   (v76 에서 K 곡선이 평평했던 이유). 그 열을 막으면 이력은 시퀀스로만 살아납니다.
    ABLATE_DROP   = ('상태_생애_',),
)

# ── [84] 실데이터 코호트 고르기 + 열 맞추기
from cdtlib import config as _cfg
if CFG.get('REAL_COHORT'):
    # 기존 2,000명 npz 의 상태열 70개를 그대로 가져와 화이트리스트로 씁니다.
    # [90] 89판은 여기서 실패해도 ⚠ 한 줄만 찍고 지나갔습니다. 그러면 상태열이
    #   70개가 아니라 111개가 되어 **기존 2,000명 결과와 비교가 불가능해집니다.**
    #   그런데 표는 멀쩡히 채워지므로 아무도 눈치채지 못합니다(v78 과 같은 종류의 사고).
    #   이 노트북의 설계는 「바뀌는 것은 사람 수 하나뿐」이므로, 여기서 멈춥니다.
    _base = None
    try:
        _base = dio.std_meta('empservicereal')          # 코호트 전환 **전**에 읽습니다
    except Exception as _e:
        print('⚠ 기존 npz 를 못 읽었습니다:', type(_e).__name__, _e)
    if _base and _base.get('state_cols'):
        _cfg.STATE_KEEP['empservicereal'] = list(_base['state_cols'])
        print('열 맞춤 기준: 기존 코호트의 상태열 %d개' % len(_base['state_cols']))
    else:
        raise RuntimeError(
            '기존 2,000명 코호트의 상태열 목록을 못 읽었습니다. 이대로 가면 상태열이\n'
            '111개가 되어 지금까지의 표와 비교할 수 없게 되는데, 표는 멀쩡히 나옵니다.\n'
            '  필요한 파일: confieee2026_a/datastd/empservicereal/empservicereal.npz\n'
            '               (확대 코호트 것이 아니라 **기존 2,000명** 것입니다)\n'
            '  이 기계에 없으면 윈도우에서 datastd/empservicereal/ 를 통째로 복사하십시오.\n'
            "  일부러 열 맞춤 없이 가시려면 CFG['REAL_COHORT'] 를 None 으로 두십시오.")
_cfg.use_cohort(CFG.get('REAL_COHORT'))

if not CFG.get('STRICT_DET', True):
    cenv.STRICT[0] = False          # 일부러 끄신 경우
ensure_dirs()
# [86] `show()` 는 경로만 나열했습니다. 어느 것이 원자료고 어느 것이 우리가 만든 것인지
#      구분이 안 돼 헷갈렸습니다. 크기·파일수와 함께 구분해서 찍습니다.
_cfg.show_data()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
OUT = DIR_OUT
DATASETS4 = ['ihdp', 'jobs', 'empservicesynt', 'empservicereal']

# ── 진행 표시 · 메모리 감시 ────────────────────────────────
# 어떤 셀에서 멈췄는지 알 수 있게 모든 단계가 시각과 메모리를 찍습니다.
import gc
try:
    import psutil
    _PROC = psutil.Process()
    _mem = lambda: _PROC.memory_info().rss / 1e9
    RAM_TOTAL = psutil.virtual_memory().total / 1e9
except Exception:
    _mem = lambda: float('nan'); RAM_TOTAL = None

_TSTEP = [time.time()]
def step(msg):
    now = time.time()
    print('[%s] %-46s  (직전 %5.0f초, 메모리 %.2f GB)'
          % (time.strftime('%H:%M:%S'), msg, now - _TSTEP[0], _mem()), flush=True)
    _TSTEP[0] = now

def sweep(*names):
    # 큰 객체를 지우고 메모리를 돌려줍니다
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()

# 한글 폰트 — 설치된 목록에서 골라야 합니다.
#   plt.rc('font', family='없는폰트') 는 예외를 안 냅니다. 그래서 옛 try/except 는
#   언제나 첫 후보에서 멈췄고, 리눅스에서 findfont 경고가 수백 줄 쏟아졌습니다.
FONT = cenv.setup_matplotlib()

REPORT = {}      # 마지막 점검표에 쌓아 둡니다
T0 = time.time()
print('\n설정:', {k: v for k, v in CFG.items()})
if RAM_TOTAL:
    print('\n메모리 %.1f GB' % RAM_TOTAL)
    if RAM_TOTAL < 12:
        print('  ⚠ 이 노트북의 최대 사용량은 약 1.5 GB 입니다. 브라우저·다른 프로그램까지 합치면')
        print('    12 GB 미만에서는 스와핑(디스크로 밀려남)이 일어나 몇 분짜리 셀이 몇 시간이 됩니다.')
        print("    CFG['LIGHT'] = True 로 두거나 우분투 서버에서 돌리십시오.")
step('설정 완료')

In [ ]:
rows = []
for k, p in RAW.items():
    ok = os.path.exists(p)
    rows.append({'key': k, '존재': ok,
                 'MB': round(os.path.getsize(p) / 1e6, 1) if ok else None,
                 'path': os.path.relpath(p, PROJECT_ROOT)})
df_raw = pd.DataFrame(rows)
display(df_raw)

missing = [k for k in df_raw.loc[~df_raw['존재'], 'key']
           if not k.startswith('ihdp1000') and not k.startswith('empservicesyntv2')]
REPORT['원자료 누락'] = missing or '없음'
if missing:
    print('⚠ 없는 원자료:', missing)
else:
    print('원자료 이상 없음')

In [ ]:
%%time
plan = ([('ihdp', r) for r in range(CFG['N_IHDP'])]
        + [('jobs', r) for r in range(CFG['N_JOBS'])]
        + [('empservicesynt', 0), ('empservicereal', 0)])

# ══════════════════════════════════════════════════════════════════
# [91] **원자료가 npz 보다 새로우면 다시 만듭니다.**
#   `std_is_current` 는 스키마 판만 봅니다. 코호트 parquet 을 새로 만들어도
#   파일 이름이 같으면(empservicereal_r16k.npz) 옛 npz 를 그대로 씁니다.
#   90판 실행에서 「새로 만든 것 0개 · 건너뛴 것 112개」가 찍혔는데, 그 안의
#   실데이터 npz 가 어느 코호트에서 나온 것인지 확인할 방법이 없었습니다.
#   v83 에서 똑같은 함정에 빠져 표가 다 나온 뒤에야 알았습니다.
def _src_newer(name, r):
    try:
        _np = dio.std_path(name, r)
        if not os.path.exists(_np):
            return False
        _sp = _cfg.real_path() if name == 'empservicereal' else (RAW.get(name) if isinstance(RAW, dict) else None)
        if not _sp or not os.path.exists(_sp):
            return False
        return os.path.getmtime(_sp) > os.path.getmtime(_np)
    except Exception:
        return False

log, n_skip, n_stale = [], 0, []
for name, r in plan:
    t0 = time.time()
    path = dio.std_path(name, r)
    # 파일이 있어도 **규격이 낡았으면** 다시 만든다.
    # (스키마 판 · 상태 지연 STATE_LAG 이 코드에서 바뀐 경우 — 단계 2에서 실제로 겪은 사고)
    if _src_newer(name, r):
        n_stale.append('%s r%d' % (name, r))          # [91] 원자료가 더 새것 → 다시 만듭니다
    elif os.path.exists(path) and dio.std_is_current(name, r) and not CFG['FORCE_STD']:
        n_skip += 1
        continue
    std = dio.build(name, realization=r)
    errs = dio.verify_std(std)
    if errs:
        raise RuntimeError(f'{name} r{r} 스키마 위반: {errs}')
    path = dio.save_std(std)
    log.append({'dataset': name, 'r': r, 'N': std.N, 'T': std.T, 'Ds': std.Ds,
                'MU': std.has_mu, 'RCT': std.has_rct,
                'MB': round(os.path.getsize(path) / 1e6, 2), 'sec': round(time.time() - t0, 1)})

step('표준화 완료')
print(f'새로 만든 것 {len(log)}개 · 이미 있어 건너뛴 것 {n_skip}개')
if n_stale:
    print('  [91] 원자료가 더 새것이라 다시 만든 것 %d개: %s'
          % (len(n_stale), ' · '.join(n_stale[:6])))
if log:
    df_log = pd.DataFrame(log)
    display(df_log.groupby('dataset').agg(개수=('r', 'count'), N=('N', 'first'), T=('T', 'first'),
                                          Ds=('Ds', 'first'), MU=('MU', 'first'),
                                          RCT=('RCT', 'first'), MB=('MB', 'sum')))
for name in DATASETS4:
    print(dio.describe(dio.load_std(name, 0)));  print('-' * 78)

# [90] 실데이터가 정말 바뀐 코호트인가 — 여기서 한 줄로 확인합니다.
#   v83 에서 코호트를 키웠는데 옛 npz 를 그대로 읽어 「새로 0 · 재사용 2070」이 나온 적이
#   있습니다. 그때는 표가 다 나온 뒤에야 알았습니다.
_rs = dio.load_std('empservicereal', 0)
print('실데이터 코호트 확인 — 사람 %s명 · 시점 %d · 상태열 %d   (%s)'
      % (format(_rs.N, ','), _rs.T, _rs.Ds, dio.std_path('empservicereal', 0)))
if CFG.get('REAL_COHORT') == 'r16k' and _rs.N != 16000:
    raise RuntimeError('r16k 를 골랐는데 사람이 %d명입니다. 코호트 parquet 을 확인하십시오.' % _rs.N)
# [91] **90판의 이 검사는 제가 틀렸습니다.**
#   STATE_KEEP 은 **전처리 전** 입력 열 목록(70개)이고, Ds 는 **전처리 후** 차원입니다.
#   전처리가 상수열·초희소열을 떨어뜨리므로 70 → 69 가 되는 것이 정상입니다
#   (실행 기록: "전처리 상태열 70→69 (상수 제거 0, 초희소 제거 1, log1p 50)").
#   그래서 같아야 한다고 건 것이 잘못이었습니다. 크게 어긋날 때만 멈춥니다.
_keep = _cfg.STATE_KEEP.get('empservicereal') or []
if CFG.get('REAL_COHORT') and _keep:
    _drop = len(_keep) - _rs.Ds
    print('  상태열 — 화이트리스트 %d개 → 전처리 후 %d개 (떨어진 열 %d개)'
          % (len(_keep), _rs.Ds, _drop))
    try:                                   # 어떤 열이 떨어졌는지 이름으로 남깁니다
        _m = dio.std_meta('empservicereal') or {}
        _used = _m.get('state_cols') or []
        _pp = (_m.get('preprocess') or {})
        for _k in ('dropped', 'drop', 'removed', '제거'):
            if _pp.get(_k):
                print('    떨어진 열:', _pp[_k]); break
        else:
            if _used and len(_used) != _rs.Ds:
                print('    (열 이름은 meta.preprocess 에 있습니다)')
    except Exception:
        pass
    if _drop < 0 or _drop > 3:
        raise RuntimeError(
            '상태열이 %d개입니다 — 화이트리스트 %d개에서 %d개가 사라졌습니다.\n'
            '  전처리가 떨어뜨리는 상수·초희소 열은 1~2개가 정상입니다. 이만큼이면\n'
            '  코호트나 열 맞춤이 어긋난 것입니다.' % (_rs.Ds, len(_keep), _drop))
del _rs; gc.collect()

In [ ]:
targets = [('ihdp', 0), ('ihdp', CFG['N_IHDP'] - 1), ('jobs', 0), ('jobs', CFG['N_JOBS'] - 1)]
if not CFG['LIGHT']:
    targets += [('empservicesynt', 0), ('empservicereal', 0)]

checks = []
for name, r in targets:
    step(f'왕복 테스트 {name} r{r}')
    src  = dio.build(name, realization=r)
    back = dio.load_std(name, r)
    same = all(np.allclose(getattr(src, k), getattr(back, k), equal_nan=True)
               if np.issubdtype(getattr(src, k).dtype, np.floating)
               else np.array_equal(getattr(src, k), getattr(back, k))
               for k in dio.ARRAY_KEYS)
    checks.append({'dataset': name, 'r': r, '왕복일치': same,
                   '스키마검증': '통과' if not dio.verify_std(back) else dio.verify_std(back)})
    del src, back
    gc.collect()
display(pd.DataFrame(checks))
assert all(c['왕복일치'] for c in checks), '저장/적재 왕복이 일치하지 않습니다.'
REPORT['왕복·스키마'] = '통과' + (' (LIGHT: 대용량 2종 생략)' if CFG['LIGHT'] else '')
step('왕복 테스트 완료')

In [ ]:
rows = []
for name in DATASETS4:
    s1 = dio.load_std(name, 0)
    s2 = dio.load_std(name, 0)
    tr, va, te = (set(s1.idx(x)) for x in ('train', 'valid', 'test'))
    rows.append({'dataset': name,
                 '겹침': len(tr & va) + len(tr & te) + len(va & te),
                 '합집합==N': len(tr | va | te) == s1.N,
                 '재적재 동일': bool(np.array_equal(s1.SPLIT, s2.SPLIT)),
                 'train/valid/test': f'{len(tr)}/{len(va)}/{len(te)}',
                 '처치율(tr/va/te)': '/'.join(
                     f"{s1.A[s1.idx(x)][..., 0][s1.M[s1.idx(x)].astype(bool)].mean():.3f}"
                     for x in ('train', 'valid', 'test'))})
df_int = pd.DataFrame(rows); display(df_int)
assert (df_int['겹침'] == 0).all() and df_int['합집합==N'].all() and df_int['재적재 동일'].all()
REPORT['분할 무결성'] = '통과 (겹침 0)'
print('무결성 검사 통과')

In [ ]:
# 처치 세 정의(A 흡수 / AEVT 사건 / APERSON 개인)가 데이터에서 어떻게 다른가
rows = []
for name in ['empservicesynt', 'empservicereal']:
    std = dio.load_std(name, 0)
    A, AE, AP = std.A[..., 0], std.AEVT[..., 0], std.APERSON[..., 0]
    ntr = (std.AIDX[..., 0] > 0).sum(1)
    first = [np.flatnonzero(std.AIDX[i, :, 0] > 0)[0] for i in np.flatnonzero(ntr > 0)]
    rows.append({'dataset': name,
                 'APERSON 개인내 상수': bool((AP == AP[:, :1]).all()),
                 '훈련경험자': f'{int((ntr > 0).sum())}/{std.N}',
                 '개인당 훈련횟수(중위)': int(np.median(ntr)),
                 '첫 훈련시점(중위)': int(np.median(first)) if first else None,
                 'A(흡수) 평균': round(float(A.mean()), 3),
                 'AEVT(사건) 평균': round(float(AE.mean()), 4),
                 'APERSON 평균': round(float(AP[:, 0].mean()), 3)})
display(pd.DataFrame(rows))

# 전처리 내역 — 기준은 train 에서만 계산했으므로 누수가 없습니다
rows = []
for name in ['empservicesynt', 'empservicereal']:
    pp = dio.load_std(name, 0).meta.get('preprocess', {})
    rows.append({'dataset': name, '입력열': pp.get('n_in'), '최종열': pp.get('n_out'),
                 '상수제거': pp.get('n_dropped'), '초희소제거': pp.get('n_dropped_sparse'),
                 'log1p': pp.get('n_logged'), '전처리후 |z|max': pp.get('zmax_after')})
display(pd.DataFrame(rows))

In [ ]:
%%time
import gen_empservicesynt as gen

f_obs = os.path.join(DIR_SYNT, 'empservicesynt_v2.parquet')
if os.path.exists(f_obs) and not CFG['FORCE_SYNT']:
    print('이미 있어 건너뜁니다:', os.path.relpath(f_obs, CONF_ROOT))
    print('  다시 만들려면 CFG["FORCE_SYNT"] = True')
else:
    gen.build(CFG['SYNT_N'], CFG['SYNT_SEED'], CFG['SYNT_HIDDEN'], CFG['SYNT_REPS'], DIR_SYNT)

with open(os.path.join(DIR_SYNT, 'empservicesynt_v2_meta.json'), encoding='utf-8') as fh:
    meta_v2 = json.load(fh)
print({k: meta_v2[k] for k in ('version', 'seed', 'N', 'T', 'R', 'hidden', 'horizons', 'female_zero')})
REPORT['합성 v2'] = f"N={meta_v2['N']} seed={meta_v2['seed']} hidden={meta_v2['hidden']} R={meta_v2['R']}"
gc.collect(); step('합성 v2 준비 완료')

In [ ]:
SYNTH = []
for nm in ['empservicesyntv2', 'empservicesyntsyn7', 'empservicesyntsyn8']:
    if not os.path.exists(RAW[nm]):
        print('원본 없음 — 건너뜀:', nm, '  (icml_datagen05.ipynb A부에서 만듭니다)')
        continue
    SYNTH.append(nm)
    p_std = dio.std_path(nm, 0)
    # 원본 parquet 가 표준 npz 보다 새것이면 다시 만든다.
    # (생성 노트북을 돌린 뒤 icml_run 이 옛 npz 를 그대로 쓰는 사고를 막습니다)
    stale = os.path.exists(p_std) and (
        os.path.getmtime(RAW[nm]) > os.path.getmtime(p_std)   # 원본이 더 새것
        or not dio.std_is_current(nm, 0))                     # 또는 규격이 낡음
    if os.path.exists(p_std) and not stale and not (CFG['FORCE_SYNT'] or CFG['FORCE_STD']):
        continue
    if stale:
        print('다시 표준화:', nm, '(원본이 더 새것이거나 스키마·상태지연이 바뀜)')
    _s = dio.build(nm)
    errs = dio.verify_std(_s, nm)
    if errs:
        raise RuntimeError(f'{nm} 스키마 위반: {errs}')
    dio.save_std(_s, nm, 0)
    if nm == 'empservicesyntv2':
        dio.save_truth(dio.build_truth('empservicesyntv2'), 'empservicesyntv2')
    print('저장 :', nm)
    del _s; gc.collect()
print('합성데이터', len(SYNTH), '종:', SYNTH)

std2 = dio.load_std('empservicesyntv2')
tt   = dio.load_truth('empservicesyntv2')
print(dio.describe(std2))
print('\n반사실 정답 배열:', {k: tuple(v.shape) for k, v in tt.items()
                              if getattr(v, 'ndim', 0) >= 1})
print('  MUA1=취업확률 · MUA2=로그임금(취업 조건부) · MUA3=기대소득(만원)')
print('  축: (사람, 행동 37, 관측창 3)   관측창 =', tt['HORIZONS'].tolist())

# 쓰는 법 예시 — c04 IT연 훈련의 기대소득 효과(12개월)
a  = list(tt['ACTIONS']).index('c04')
h  = list(tt['HORIZONS']).index(12)
tau = tt['MUA3'][:, a, h] - tt['MUA3'][:, 0, h]
fem = tt['GENDER'] == 1
print(f'\nc04 IT연 · 12개월 기대소득 효과   남 {tau[~fem].mean():+.2f}만원 / 여 {tau[fem].mean():+.2f}만원')

In [ ]:
%%time
LK = []
for nm in ['empservicereal', 'empservicesynt'] + SYNTH:
    if not os.path.exists(dio.std_path(nm, 0)):
        continue
    s = dio.load_std(nm)
    lag  = int(s.meta.get('state_lag', 0))
    cols = {c: j for j, c in enumerate(s.meta['state_cols'])}
    voc  = list(s.meta['action_vocab'])
    aidx = s.AIDX[..., 0]
    n_first = n_leak = 0
    for a, cd in enumerate(voc):
        if a == 0:
            continue
        j = cols.get('상태_생애_훈련직종%s효과잔존량' % cd)
        if j is None:
            continue                      # 전처리에서 버려진 열
        ev = (aidx == a)
        rows = np.flatnonzero(ev.any(1))
        if rows.size == 0:
            continue
        ts = ev[rows].argmax(1)           # 그 직종의 첫 훈련 시점
        n_first += int(rows.size)
        n_leak  += int((s.X[rows, ts, j] > 1e-9).sum())
    LK.append({'데이터': nm.replace('empservice', ''), '상태지연': lag,
               '그 직종 첫 훈련': n_first, '상태에 이미 효과 있음': n_leak,
               '비율%': round(100.0 * n_leak / max(n_first, 1), 2)})
    del s; gc.collect()

lk = pd.DataFrame(LK)
print('처치 시점 누출 검사 (단계 2 적용 후)')
print(lk.to_string(index=False))
lk.to_csv(os.path.join(DIR_OUT, '05b_처치시점_누출검사.csv'), index=False, encoding='utf-8-sig')
REPORT['처치시점 누출(지연 후)'] = ' · '.join(
    '%s %.1f%%(지연%d)' % (r['데이터'], r['비율%'], r['상태지연']) for r in LK)
step('단계 2 누출 검사')

In [ ]:
%%time
from cdtlib import outcomes as oc

OUTC, SUM3, NAI3 = {}, [], []
for nm in ['empservicereal'] + SYNTH:
    if not os.path.exists(dio.std_path(nm, 0)):
        continue
    _s = dio.load_std(nm)
    _ts = dio.load_truth(nm)['TSTAR'] if nm == 'empservicesyntv2' else None
    _o = oc.build(_s, nm, tstar=_ts)
    oc.save(_o, nm)
    OUTC[nm] = _o
    _a = oc.summary(_o);    _a.insert(0, '데이터', nm.replace('empservice', '')); SUM3.append(_a)
    _b = oc.naive_diff(_o); _b.insert(0, '데이터', nm.replace('empservice', '')); NAI3.append(_b)
    del _s; gc.collect()

sum3 = pd.concat(SUM3, ignore_index=True)
nai3 = pd.concat(NAI3, ignore_index=True)
print('── 결과변수 요약 (관측 가능 인원과 우측 절단)')
print(sum3.to_string(index=False))
sum3.to_csv(os.path.join(DIR_OUT, '05c_결과변수_요약.csv'), index=False, encoding='utf-8-sig')

print()
print('── 조정 없는 단순 차이 (훈련군 − 대조군). 조정 전 하한입니다')
print(nai3.to_string(index=False))
nai3.to_csv(os.path.join(DIR_OUT, '05c_단순차이.csv'), index=False, encoding='utf-8-sig')
print()
print('※ 합성 v2 는 생성기가 훈련받은 사람에게만 t* 와 참값을 저장합니다.')
print('  그래서 채점 표본이 훈련군뿐이고(ATT 성격) 대조군 칸이 빕니다. 정상입니다.')

_r = OUTC.get('empservicereal')
if _r is not None:
    _tr = _r['TREATED'].astype(bool)
    REPORT['단계3 결과변수'] = ' · '.join(
        '%d개월 관측 %d명(절단 %d)' % (h, int(_r['OBS'][:, k].sum()), int((~_r['OBS'][:, k]).sum()))
        for k, h in enumerate(oc.HORIZONS))
step('단계 3 결과변수 생성')

In [ ]:
# ── Y2 의 표본선택 — Lee(2009) 경계
lee = oc.lee_table(OUTC['empservicereal'])
print('── 실데이터 Y2(로그임금) Lee 경계')
print(lee.to_string(index=False))
lee.to_csv(os.path.join(DIR_OUT, '05c_Lee경계.csv'), index=False, encoding='utf-8-sig')
REPORT['Y2 Lee 경계(12개월)'] = '[%.4f, %.4f]  단순 %.4f' % (
    lee.loc[1, 'Lee 하한'], lee.loc[1, 'Lee 상한'], lee.loc[1, '단순차이(편의 있음)'])

# ── 합성 v2 : 관측 결과가 참값과 같은 척도인가
# 정의가 어긋나면 여기서 드러납니다. 채점(PEHE)의 전제 조건입니다.
if 'empservicesyntv2' in OUTC:
    _o = OUTC['empservicesyntv2']; _t = dio.load_truth('empservicesyntv2')
    _a = _t['AOBS']; _tr = _o['TREATED'].astype(bool)
    _rows = []
    for _k, _h in enumerate(oc.HORIZONS):
        _i = np.where(_o['OBS'][:, _k] & _tr)[0]
        _rows.append({'관측창': _h, '인원': len(_i),
                      'Y1 관측': round(float(np.nanmean(_o['Y1'][_i, _k])), 4),
                      'Y1 참값': round(float(np.nanmean(_t['MUA1'][_i, _a[_i], _k])), 4),
                      'Y3 관측': round(float(np.nanmean(_o['Y3'][_i, _k])), 2),
                      'Y3 참값': round(float(np.nanmean(_t['MUA3'][_i, _a[_i], _k])), 2)})
    align = pd.DataFrame(_rows)
    print()
    print('── 합성 v2 : 관측 결과 vs 참값 (같은 t* · 같은 행동)')
    print(align.to_string(index=False))
    align.to_csv(os.path.join(DIR_OUT, '05c_v2_참값대조.csv'), index=False, encoding='utf-8-sig')
    _gap = float(abs(align.loc[0, 'Y1 관측'] - align.loc[0, 'Y1 참값']))
    REPORT['v2 관측vs참값(6개월 Y1)'] = '%.4f vs %.4f' % (align.loc[0, 'Y1 관측'], align.loc[0, 'Y1 참값'])
    if _gap > 0.02:
        print()
        print('  ⚠ 짧은 관측창에서 관측과 참값의 「수준」이 벌어집니다.')
        print('    원인: gen_empservicesynt.simulate_cf 가 snapshot 에서 10개 필드만 복사하고')
        print('          res(과거 훈련 잔존) · wage · ever · in_tr · tr_end 를 0 으로 되돌립니다.')
        print('    영향: 수준은 어긋나지만 채점에 쓰는 것은 차이 τ = MU[a] − MU[0] 이고')
        print('          두 갈래가 같은 출발점을 공유(CRN)하므로 차이는 대체로 상쇄됩니다.')
        print('    조치: 합성데이터를 다시 만들 때 복사 목록에 위 필드를 추가하십시오.')
step('단계 3 Lee 경계 · 참값 대조')

In [ ]:
%%time
DESC = ['empservicereal'] + SYNTH
if CFG['LIGHT']:
    DESC = ['empservicereal']
RES = {}
if CFG['RUN_DESC']:
    for key in DESC:
        rep = ds.raw_null_report(RAW[key])
        print(f"[{key}] {rep.attrs['n_rows']:,}행 × {rep.attrs['n_cols']}열 | "
              f"결측 있는 변수 {int((rep['결측수'] > 0).sum())}개 | "
              f"최빈값 99% 이상(사실상 상수) {int((rep['최빈비율%'] >= 99).sum())}개")
    print()
    for name in DESC:
        step(f'기술분석 {name}')
        _s = dio.load_std(name, 0)
        RES[name] = ds.run_all(_s)
        del _s; gc.collect()
        p = RES[name]['profile']
        print(f'=== {name}  (train {p.attrs["n_obs"]:,}칸 × {len(p)}변수)')
        print(f'  결측 {int(p["결측수"].sum())} | 0비율 99%↑ {int((p["0비율%"] >= 99).sum())}개 '
              f'| |왜도|>5 {int((p["왜도"].abs() > 5).sum())}개 | 최대|z|>10 {int((p["최대|z|"] > 10).sum())}개')

In [ ]:
# 데이터셋 5종 개관
rows = []
for name in DATASETS4 + SYNTH:
    s = dio.load_std(name, 0); m = s.M.astype(bool)
    rows.append({'데이터셋': name, '사람수': s.N, '시점': s.T, '상태칸': s.Ds,
                 '유효칸(M=1)': int(m.sum()), '유효비율%': round(100 * float(m.mean()), 1),
                 '반사실정답': ('있음' if s.has_mu else
                             ('별도파일' if name == 'empservicesyntv2' else '없음')),
                 '처치율': round(float(s.A[..., 0][m].mean()), 3),
                 'Y평균': round(float(s.Y[..., 0][m].mean()), 3),
                 'Y표준편차': round(float(s.Y[..., 0][m].std()), 3),
                 '복제본': dio.DATASETS[name]['n_realization']})
display(pd.DataFrame(rows))

In [ ]:
if CFG['RUN_DESC']:
    print('■ 처치 균형 (SMD) — 훈련받은 사람과 안 받은 사람이 원래 달랐나')
    for name in DESC:
        b = RES[name]['balance']
        print(f"  {name:18s} 평균|SMD| {b.attrs['mean_abs_smd']} | "
              f"|SMD|>0.1 인 변수 {b.attrs['n_imbalanced']}/{len(b)}개 "
              f"(처치 {b.attrs['n_treat']:,} vs 대조 {b.attrs['n_ctrl']:,})")
    display(RES['empservicereal']['balance'].head(8).round(4))

    print('\n■ 누출 검사 — 처치 이후에 생긴 변수가 상태에 섞였나  ★')
    for name in DESC:
        lk = RES[name]['leakage']; n = int(lk['누출의심'].sum())
        print(f'  {name:18s} 누출 의심 {n}개 / {len(lk)}개')
    REPORT['누출 의심'] = ' · '.join(
        '%s %d개' % (n.replace('empservice', ''), int(RES[n]['leakage']['누출의심'].sum()))
        for n in DESC)

    print('\n■ 중복 변수 (|r| ≥ 0.999)')
    for name in DESC:
        print(f"  {name:18s} {len(RES[name]['duplicates'])}쌍")

    print('\n■ 결과 요약 — train 단순차이')
    for name in DESC:
        print(f"  {name:18s} {RES[name]['outcome'].attrs.get('naive_diff')}")

    print('\n■ 시간 교란 (train, M=1)')
    for name in DESC:
        s = dio.load_std(name, 0); idx = s.idx('train'); m = s.M[idx].astype(bool)
        ts = s.TS[idx][m].astype(float); A = s.A[idx][..., 0][m]; Y = s.Y[idx][..., 0][m]
        print(f'  {name:18s} corr(t,A)={np.corrcoef(ts, A)[0, 1]:+.3f}  '
              f'corr(t,Y)={np.corrcoef(ts, Y)[0, 1]:+.3f}  corr(A,Y)={np.corrcoef(A, Y)[0, 1]:+.3f}')

In [ ]:
# 변수 프로파일 · 특이값 · 종속변수와의 관계 · 범주별 상세 (실데이터 기준)
if CFG['RUN_DESC']:
    display(RES['empservicereal']['profile'].head(12).round(3))
    display(RES['empservicereal']['outliers'].head(8))
    display(RES['empservicereal']['corr_y'].head(10).round(4))
    for c in RES['empservicereal']['cat_cols'][:3]:
        print(f'--- {c}'); display(RES['empservicereal']['categorical'][c])

In [ ]:
if CFG['RUN_DESC']:
    fig, axes = plt.subplots(len(DESC), 3, figsize=(16, 4 * len(DESC)))
    for r, name in enumerate(DESC):
        s = dio.load_std(name, 0); idx = s.idx('train'); m = s.M[idx].astype(bool)
        y = s.Y[idx][..., 0][m]; a = s.A[idx][..., 0][m]; t = RES[name]['time']
        ax = axes[r, 0]
        ax.hist(y[a == 0], bins=50, alpha=.6, label='대조군', density=True)
        ax.hist(y[a == 1], bins=50, alpha=.6, label='처치군', density=True)
        ax.set_title(f'{name}\n결과(로그임금) 분포'); ax.legend(); ax.set_xlabel('Y')
        ax = axes[r, 1]
        ax.plot(t['t'], t['Y평균'], label='Y 평균')
        ax2 = ax.twinx(); ax2.plot(t['t'], t['처치율%'], color='tab:red', ls='--')
        ax.set_title('시점별 결과와 처치율(붉은 점선)'); ax.set_xlabel('t (개월)')
        ax = axes[r, 2]
        p = RES[name]['profile']
        ax.scatter(p['0비율%'], p['최대|z|'], s=12)
        ax.set_xlabel('0 비율 %'); ax.set_ylabel('최대 |z|'); ax.set_title('희소할수록 극단값이 커진다')
    plt.tight_layout(); plt.savefig(os.path.join(OUT, 'desc_overview.png'), dpi=140); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    t = RES['empservicereal']['time']
    axes[0].plot(t['t'], t['취업률%']);   axes[0].set_title('실데이터 시점별 취업률'); axes[0].set_xlabel('t')
    axes[1].plot(t['t'], t['훈련시작%']); axes[1].set_title('실데이터 시점별 훈련 시작 비율'); axes[1].set_xlabel('t')
    t2 = RES['empservicesyntv2']['time']
    axes[2].plot(t2['t'], t2['취업률%']); axes[2].set_title('합성 v2 시점별 취업률'); axes[2].set_xlabel('t')
    plt.tight_layout(); plt.savefig(os.path.join(OUT, 'desc_time.png'), dpi=140); plt.show()

In [ ]:
if CFG['RUN_DESC']:
    saved = []
    for name in DESC:
        saved += ds.save_tables(RES[name], OUT, name)
    for name in ['ihdp', 'jobs']:
        saved += ds.save_tables(ds.run_all(dio.load_std(name, 0)), OUT, name)
    print(f'{len(saved)}개 표 저장 → out/')

In [ ]:
%%time
import verify_benchmarks as vb
step('벤치마크 정본 대조 시작')
ref = vb.run(fetch=CFG['FETCH_REF'])
gc.collect(); step('벤치마크 정본 대조 완료')
REPORT['IHDP 정본 대조'] = f"{ref['ihdp']['정본대조_일치']}/{ref['ihdp']['정본대조_대상']}판 일치 (미확보 {ref['ihdp']['정본대조_미확보']})"
REPORT['Jobs ate=ATT'] = f"파일 {ref['jobs']['ate_file']:.6f} vs 재계산 {ref['jobs']['att']:.6f}"

In [ ]:
%%time
import verify_synt_v2 as vs
step('합성 v2 검증 시작')
SYN = vs.main()
gc.collect(); step('합성 v2 검증 완료')
REPORT['v2 수정확인'] = ('전항목 통과' if (SYN['수정확인']['판정'] == '통과').all()
                       else SYN['수정확인'].loc[SYN['수정확인']['판정'] != '통과', '번호'].tolist())
REPORT['v2 교란(Y3)'] = ('단순 %.2f vs 참 ATT %.2f'
                        % (SYN['교란'].loc[2, '단순차이'], SYN['교란'].loc[2, '참 ATT']))
REPORT['v2 양성성'] = SYN['양성성']['판정'].value_counts().to_dict()

In [ ]:
# 그림 3의 정답 곡선 — 이력이 길수록 U 는 드러나고 V 는 안 드러난다
tab = SYN['대리충분성']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(tab['이력길이K'], tab['U(흔적 남김) R2'], 'o-', label='U — 이력에 흔적을 남김')
ax[0].plot(tab['이력길이K'], tab['V(은닉) R2'], 's--', label='V — 흔적 없음')
ax[0].set_xscale('log'); ax[0].set_xlabel('이력 길이 K (개월)'); ax[0].set_ylabel('복원 R²')
ax[0].set_title('가정 1(이력의 대리 충분성) 검증'); ax[0].legend(); ax[0].grid(alpha=.3)

j = SYN['J곡선']
ax[1].plot(j['t*로부터의 개월'], j['취업률'], 'o-')
ax[1].axvline(0, color='r', ls=':', label='훈련 시작 t*')
ax[1].set_xlabel('t* 로부터의 개월'); ax[1].set_ylabel('취업률')
ax[1].set_title('J곡선 — 선택 · lock-in · 회복'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'syn_proxy_jcurve.png'), dpi=140); plt.show()

In [ ]:
# 선생님이 예로 드신 대조 — 용접 훈련의 성별 효과
het = SYN['이질성']
key = het[het.훈련직종.isin(['c27', 'c26', 'c28', 'c24', 'c14', 'c17', 'c21', 'c04'])]
display(key.style.format({'τ_취업': '{:.4f}', 'τ_로그임금': '{:.4f}', 'τ_기대소득(만원)': '{:+.2f}'})
        if hasattr(key, 'style') else key)
print('\n괄호 없이 읽으십시오. 「관측수강자수 = 0」 인 줄은 참값이 존재하지만')
print('데이터로는 도달할 수 없는 칸입니다. 모델이 여기에 숫자를 내놓으면 오답으로 채점합니다.')

In [ ]:
%%time
from cdtlib import models as mdl, runs as R

CDT_HP = {}          # 자료별로 valid 가 고른 초매개변수 — §8 이 이어 씁니다
HP_PICK = {}         # (자료, 모형) → **첫 시드**에서 valid 가 고른 값. 나머지 시드는 이것을 씁니다
#   시드마다 격자를 다 돌면 시간이 시드 수만큼 곱해지고, 시드마다 다른 초매개변수를 쓰면
#   「시드 간 흔들림」에 「조율의 흔들림」이 섞여 무엇을 재는지 알 수 없게 됩니다.

def CDT_KW(**over):
    kw = dict(epochs=CFG['CDT_EPOCHS'], K=CFG['CDT_K'], hidden=CFG['CDT_HIDDEN'],
              lr=CFG['CDT_LR'], dropout=CFG['CDT_DROPOUT'],
              lambda_action=CFG['CDT_LAMBDA_A'], action=CFG['CDT_ACTION'])
    kw.update(over)
    return kw

def _load_pair(nm, rep=0):
    _s = dio.load_std(nm, rep)
    _o = oc.for_dataset(_s, nm, rep)
    _t = dio.load_truth(nm) if nm == 'empservicesyntv2' else None
    return _s, _o, _t

if not CFG['TRAIN']:
    _n = len(CFG['TRAIN_DATA']) * len(CFG['TRAIN_MODELS']) * len(CFG['TRAIN_SEEDS'])
    print('§7 학습은 꺼져 있습니다.  CFG["TRAIN"] = True 로 두면 돕니다.')
    print('  켜면 돌 것: 데이터 %d × 모델 %d × 시드 %d = 실행 %d개'
          % (len(CFG['TRAIN_DATA']), len(CFG['TRAIN_MODELS']), len(CFG['TRAIN_SEEDS']), _n))
    print('  ', ' · '.join(CFG['TRAIN_DATA']))
    print('  ', ' · '.join(CFG['TRAIN_MODELS']))
    SCORES = None
else:
    # ── 벽시계 재기 (계획서 13.4절 1단계). 예산을 모르면 계획을 세울 수 없습니다.
    if CFG['TRAIN_TIMING']:
        _s, _o, _ = _load_pair('ihdp', 0)
        _tim = []
        for _m in CFG['TRAIN_MODELS']:
            _t0 = time.time()
            try:
                _kw = CDT_KW() if _m == 'cdt' else {}
                _f0 = mdl.MODELS[_m](seed=0)
                _f0.fit(_s, _o, h=0, y='Y3', split=0, **_kw)
                _grid_sec = round(time.time() - _t0, 1)
                # [100] 표 8(비용)은 「1회 적합」이어야 합니다. 위 시간은 격자(후보 여러 개)
                #   포함이라, CDT 15.7초 × 300판 > 총소요 라는 검산 모순을 만들었습니다.
                #   고른 초매개변수 하나로 다시 적합해 단일적합 시간을 따로 잽니다.
                _pk = getattr(_f0, 'hp_chosen', None) or {}
                _kw1 = dict(_kw); _kw1.update(_pk)
                for _gk in ('lr', 'dropout'):
                    if isinstance(_kw1.get(_gk), (tuple, list)):
                        _kw1[_gk] = _kw1[_gk][0]
                _kw1.pop('configs', None)
                _t1 = time.time()
                mdl.MODELS[_m](seed=0).fit(_s, _o, h=0, y='Y3', split=0, **_kw1)
                _tim.append({'모델': _m, '초(단일적합)': round(time.time() - _t1, 1),
                             '초(격자포함)': _grid_sec,
                             '후보수': len(getattr(_f0, 'hp_table', []) or []) or 1})
            except Exception as _e:
                import traceback as _tb
                _tim.append({'모델': _m, '초(단일적합)': None,
                             '오류': '%s: %s' % (type(_e).__name__, _e)})
                print('  ⚠ 벽시계 %s 실패' % _m)
                print(_tb.format_exc())
        tim = pd.DataFrame(_tim)
        print('── IHDP 1판 1시드 벽시계')
        print(tim.to_string(index=False))
        tim.to_csv(os.path.join(DIR_OUT, '07_벽시계.csv'), index=False, encoding='utf-8-sig')
        # [71] 죽은 모델의 시간을 0 으로 더하면 「빨리 끝났다」로 보입니다. 실제로 겪었습니다.
        _ok = tim['초(단일적합)'].notna()
        _bad = tim[~_ok]
        _tot = float(tim.loc[_ok, '초(단일적합)'].sum())
        print()
        if len(_bad):
            print('  ⚠ %d개 모델이 실패했습니다 — 아래 시간은 성공한 %d개만 더한 값입니다.'
                  % (len(_bad), int(_ok.sum())))
            for _, _r in _bad.iterrows():
                print('     %-10s %s' % (_r['모델'], _r.get('오류', '')))
            REPORT['벽시계(ihdp 1판)'] = '성공 %d/%d · %.0f초 (실패: %s)' % (
                int(_ok.sum()), len(tim), _tot, ' · '.join(_bad['모델']))
        else:
            print('  성공 %d/%d · 합계 %.0f초' % (int(_ok.sum()), len(tim), _tot))
            _plan = sum(len(CFG.get('SEEDS_BY', {}).get(_d, CFG['TRAIN_SEEDS']))
                        * CFG.get('REPS_BY', {}).get(_d, CFG['TRAIN_REPS'])
                        * len(CFG['TRAIN_MODELS'])
                        * (1 + len(CFG.get('TRAIN_Y_EXTRA', {}).get(_d, ())))
                        for _d in CFG['TRAIN_DATA'])
            print('  이번에 돌 실행: %d개' % _plan)
            REPORT['벽시계(ihdp 1판 6모델)'] = '단일적합 합계 %.0f초 (표 8 은 이 열로)' % _tot
        del _s, _o; gc.collect()
        step('벽시계 측정')

    # ── 본 학습
    ROWS, FAILS = [], []
    for _nm in CFG['TRAIN_DATA']:
        _reps = CFG.get('REPS_BY', {}).get(_nm, CFG['TRAIN_REPS'])
        for _rep in range(_reps):
            _s, _o, _tr = _load_pair(_nm, _rep)
            _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
            _YS = (CFG['TRAIN_Y'],) + tuple(CFG.get('TRAIN_Y_EXTRA', {}).get(_nm, ()))
            for _y in _YS:
              _ytag = None if _y == CFG['TRAIN_Y'] else _y   # 기본 결과변수는 꼬리표 없이
              for _seed in CFG.get('SEEDS_BY', {}).get(_nm, CFG['TRAIN_SEEDS']):
                for _m in CFG['TRAIN_MODELS']:
                    _cfg = dict(h=_h, y=_y, seed=_seed)
                    _pick = HP_PICK.get((_nm, _m, _y))
                    if _m == 'cdt':
                        _cfg.update(CDT_KW(**(_pick or {})))
                        if _pick is None and CFG.get('CDT_CONFIGS'):
                            _cfg['configs'] = CFG['CDT_CONFIGS']   # 첫 판에서만 고릅니다
                    elif _pick:
                        _cfg.update(_pick)
                    try:
                        # [85] overwrite=True — 같은 이름의 옛 실행이 있으면 덮어씁니다.
                        #      읽어 오지 않습니다. 표의 숫자는 전부 이번에 나온 것입니다.
                        with R.Run(_nm, _m, _seed,
                                   realization=(_rep if _reps > 1 else None),
                                   tag=_ytag, ver=VERSION, config=_cfg,
                                   overwrite=True) as _run:
                            _t0 = time.time()
                            _kw = {k: v for k, v in _cfg.items() if k not in ('h', 'y', 'seed')}
                            _fit = mdl.MODELS[_m](seed=_seed)
                            _fit.fit(_s, _o, h=_h, y=_y, split=0, **_kw)
                            _tau = _fit.predict_tau(_s, _o, h=_h)
                            # [77] τ 를 남깁니다 — §7c 가 시드 평균(앙상블)을 만들 때 씁니다.
                            np.save(os.path.join(_run.dir, 'tau.npy'), _tau)
                            _sc = mdl.score(_nm, _tau, _s, _o, h=_h, y=_y,
                                            split=2, truth=_tr)
                            _sc.update(모델=_m, 시드=_seed, 판=_rep, 결과=_y,
                                       초=round(time.time() - _t0, 1),
                                       best_epoch=getattr(_fit, 'best_epoch', None),
                                       돈에폭=getattr(_fit, 'epochs_run', None),
                                       에폭한계=getattr(_fit, 'hit_limit', None),
                                       선택=getattr(_fit, 'hp_chosen', None))
                            # [74] §8 소거연구는 §7 이 valid 로 고른 초매개변수를 그대로 씁니다.
                            #      (K 만 바꾸고 나머지를 같이 흔들면 K 의 효과가 아닙니다)
                            if getattr(_fit, 'hp_chosen', None):
                                HP_PICK.setdefault((_nm, _m, _y), dict(_fit.hp_chosen))
                                if _m == 'cdt':
                                    CDT_HP.setdefault(_nm, dict(_fit.hp_chosen))
                            _run.result(**_sc)
                            ROWS.append(_sc)
                            print('  %-18s %-9s s%d r%d  %5.1f초  %s'
                                  % (_nm, _m, _seed, _rep, _sc['초'],
                                     {k: v for k, v in _sc.items()
                                      if k in ('√PEHE', 'ε_ATE', 'ε_ATT', '정책위험', 'AUUC', 'Qini')}))
                    except Exception as _e:
                        import traceback as _tb
                        print('  ⚠ %s %s 실패: %s: %s' % (_nm, _m, type(_e).__name__, _e))
                        print(_tb.format_exc())
                        FAILS.append((_nm, _m, '%s: %s' % (type(_e).__name__, _e)))
            del _s, _o, _tr; gc.collect()
        step('학습 %s' % _nm)

    SCORES = pd.DataFrame(ROWS)
    # [71] 성공이 0 이어도 반드시 남깁니다. 줄이 없으면 「안 돌렸나?」로 오해합니다.
    REPORT['학습 실행'] = '%d개 전부 새로 학습 · 실패 %d%s' % (
        len(SCORES), len(FAILS),
        ('  (' + ' · '.join('%s/%s' % (a, b) for a, b, _ in FAILS[:6]) + ')') if FAILS else '')
    if FAILS:
        print()
        print('── 실패 목록 (%d건)' % len(FAILS))
        for _a, _b, _c in FAILS:
            print('  %-18s %-9s %s' % (_a, _b, _c[:110]))
    if len(SCORES):
        _front = ['데이터', '모델', '시드', '판', '분할', 'n', '초', 'best_epoch']
        SCORES = SCORES[[c for c in _front if c in SCORES.columns] +
                        [c for c in SCORES.columns if c not in _front]]
        print()
        print(SCORES.to_string(index=False))
        SCORES.to_csv(os.path.join(DIR_OUT, '07_모델비교.csv'), index=False, encoding='utf-8-sig')
        REPORT['학습 실행'] += '  [%s]' % ' · '.join(sorted(set(SCORES['데이터'])))
    step('§7 학습 완료')

In [ ]:
# ── 7b. 시드 평균 ± 표준편차 (표 2·3·4 의 실제 내용)
# 한 시드 숫자로 순위를 매기면 안 됩니다. v75 에서 같은 설정을 세 번 돌려
# 합성 v2 √PEHE 가 29.85 / 30.88 / 34.68 이 나왔습니다 — 모형 간 차이보다 컸습니다.
PRIMARY = {'ihdp': ['√PEHE', 'ε_ATE'], 'jobs': ['ε_ATT', '정책위험'],
           'empservicesyntv2': ['√PEHE', 'ε_ATE'], 'empservicereal': ['Qini', 'AUUC']}
if SCORES is None or not len(SCORES):
    print('§7 을 먼저 돌리십시오.')
    SUMMARY = None
else:
    _rows = []
    _KEY = ['데이터', '결과'] if '결과' in SCORES.columns else ['데이터']
    for _k, _g in SCORES.groupby(_KEY):
        _nm = _k[0] if isinstance(_k, tuple) else _k
        for _m, _gm in _g.groupby('모델'):
            _r = {'데이터': _nm, '모델': _m, '실행수': len(_gm)}
            # [83] **안정성** — 같은 판 안에서 시드끼리 얼마나 흔들리는가.
            #   판을 넘나드는 표준편차(자료 난이도)와는 전혀 다른 값입니다. 실무자가
            #   「한 번 돌려서 나온 숫자를 믿어도 되나」를 판단하는 데 쓰는 것은 이쪽입니다.
            #   IHDP 에서 tarnet 은 시드가 바뀌어도 거의 안 변했고 cdt 는 크게 변했습니다.
            if _gm['판'].nunique() > 1 and _gm['시드'].nunique() > 1:
                for _k0 in PRIMARY.get(_nm, []):
                    if _k0 in _gm.columns and _gm[_k0].notna().any():
                        _r['시드흔들림(%s)' % _k0] = round(
                            float(_gm.groupby('판')[_k0].std().mean()), 4)
                        break
            if '결과' in SCORES.columns:
                _r['결과'] = _gm['결과'].iloc[0]
            for _k in PRIMARY.get(_nm, []):
                if _k in _gm.columns and _gm[_k].notna().any():
                    _v = _gm[_k].astype(float)
                    _r[_k] = '%.4f ± %.4f' % (_v.mean(), _v.std(ddof=1) if len(_v) > 1 else 0.0)
                    _r['_%s' % _k] = float(_v.mean())
            _rows.append(_r)
    SUMMARY = pd.DataFrame(_rows)
    _order = list(CFG['TRAIN_MODELS'])
    SUMMARY['_o'] = SUMMARY['모델'].map({m: i for i, m in enumerate(_order)})
    _sk = ['데이터', '결과', '_o'] if '결과' in SUMMARY.columns else ['데이터', '_o']
    SUMMARY = SUMMARY.sort_values(_sk).drop(columns=['_o'])
    SUMMARY = SUMMARY[[c for c in ['데이터', '결과', '모델', '실행수'] if c in SUMMARY.columns]
                      + [c for c in SUMMARY.columns
                         if c not in ('데이터', '결과', '모델', '실행수')]]
    _show = SUMMARY[[c for c in SUMMARY.columns if not c.startswith('_')]]
    print(_show.to_string(index=False))
    _show.to_csv(os.path.join(DIR_OUT, '07_모델비교_요약.csv'), index=False, encoding='utf-8-sig')

    # 자료마다 1차 지표의 승자 (낮을수록 좋은 지표 / 높을수록 좋은 지표를 구분)
    _HIGH = {'Qini', 'AUUC'}
    print()
    for _key, _sub0 in SUMMARY.groupby(_KEY):
        _nm = _key[0] if isinstance(_key, tuple) else _key
        _yl = (' %s' % _key[1]) if isinstance(_key, tuple) and len(_key) > 1 else ''
        _k = PRIMARY[_nm][0]; _c = '_%s' % _k
        if _c not in _sub0:
            continue
        _sub = _sub0[_sub0[_c].notna()]
        if not len(_sub):
            continue
        _b = _sub.loc[_sub[_c].idxmax() if _k in _HIGH else _sub[_c].idxmin()]
        print('  %-18s%-4s %-8s 1위: %-9s %s' % (_nm, _yl, _k, _b['모델'], _b[_k]))
        REPORT['표 %s%s (%s)' % (_nm, _yl, _k)] = '%s %s' % (_b['모델'], _b[_k])
    step('§7b 시드 요약')


In [ ]:
%%time
# ── 7c. 시드 앙상블 — τ 를 시드 평균한 뒤 **한 번** 채점합니다
# 실데이터에서 CDT 의 Qini 가 989 ± 2354 로 흔들렸습니다(시드별 2571·1809·604·-2918·2880).
# n=1345 에 3층 트랜스포머라 시드마다 다른 국소해로 갑니다. 평균은 이 분산을 줄입니다
# (deep ensemble). **표에는 앙상블과 단일 시드를 함께** 싣습니다 — 감추면 안 됩니다.
if SCORES is None or not len(SCORES):
    print('§7 을 먼저 돌리십시오.')
    ENSEMBLE = None
else:
    _rows = []
    _KEY2 = ['데이터', '결과'] if '결과' in SCORES.columns else ['데이터']
    for _key, _g in SCORES.groupby(_KEY2):
        _nm = _key[0] if isinstance(_key, tuple) else _key
        _y = _key[1] if isinstance(_key, tuple) and len(_key) > 1 else CFG['TRAIN_Y']
        _ytag = None if _y == CFG['TRAIN_Y'] else _y
        _s, _o, _tr = _load_pair(_nm, 0)
        _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
        for _m in CFG['TRAIN_MODELS']:
            _taus = []
            for _seed in CFG['TRAIN_SEEDS']:
                _fp = os.path.join(R.DIR_RUNS, R.run_id(_nm, _m, _seed, None, _ytag, VERSION),
                                   'tau.npy')
                if os.path.exists(_fp):
                    _taus.append(np.load(_fp))
            if len(_taus) < 2:
                continue
            _tau = np.mean(_taus, 0)
            _sc = mdl.score(_nm, _tau, _s, _o, h=_h, y=_y, split=2, truth=_tr)
            _sc.update(데이터=_nm, 결과=_y, 모델=_m, 시드수=len(_taus),
                       τ평균=float(_tau.mean()), τsd=float(_tau.std()))
            _rows.append(_sc)
        del _s, _o, _tr; gc.collect()
    ENSEMBLE = pd.DataFrame(_rows)
    if len(ENSEMBLE):
        _front = ['데이터', '결과', '모델', '시드수']
        ENSEMBLE = ENSEMBLE[[c for c in _front if c in ENSEMBLE.columns]
                            + [c for c in ENSEMBLE.columns if c not in _front]]
        _drop = [c for c in ('분할', 'n', '초', 'best_epoch', '돈에폭', '에폭한계', '선택') 
                 if c in ENSEMBLE.columns]
        print(ENSEMBLE.drop(columns=_drop).to_string(index=False))
        ENSEMBLE.to_csv(os.path.join(DIR_OUT, '07_앙상블.csv'), index=False, encoding='utf-8-sig')
        # 단일 시드 평균과 견줍니다 — 앙상블이 정말 나은가
        _HIGH2 = {'Qini', 'AUUC'}
        for _key, _e in ENSEMBLE.groupby(_KEY2):
            _nm = _key[0] if isinstance(_key, tuple) else _key
            _k = PRIMARY[_nm][0]
            if _k not in _e.columns:
                continue
            _b = _e.loc[_e[_k].astype(float).idxmax() if _k in _HIGH2
                        else _e[_k].astype(float).idxmin()]
            _lab = '%s%s' % (_nm, (' ' + _key[1]) if isinstance(_key, tuple) else '')
            print('  앙상블 %-20s %-8s 1위: %-9s %.4f' % (_lab, _k, _b['모델'], float(_b[_k])))
            REPORT['앙상블 %s (%s)' % (_lab, _k)] = '%s %.4f' % (_b['모델'], float(_b[_k]))
    step('§7c 앙상블')


In [ ]:
%%time
# ── 7d. 짝지은 붓스트랩 — **모형 간 차이가 표본오차 안인가**
#
# 왜 이것이 없으면 안 되는가. 실데이터 시험표본은 293명(처치 151 · 대조 142)이고
# Y3(기대소득)은 63.9%가 0, sd 123 인 긴 꼬리 분포입니다. 이런 표본에서 Qini 의
# 95% 신뢰구간은 ±2,500 규모입니다 — v77 에서 우리가 순위로 읽던 모형 간 격차와
# 같은 크기입니다. 구간을 안 그리면 「cforest 가 CDT 를 이겼다」를 **말할 수 없습니다.**
#
# 짝짓기(paired): 같은 재표본에서 두 모형의 Qini 를 함께 계산해 **차이**의 구간을 냅니다.
# 사람 표본의 흔들림이 두 모형에 똑같이 걸리므로, 따로 구간을 그릴 때보다 훨씬 예민합니다.
from cdtlib.models import _fit_mask, _yh, qini as _qini_f, auuc as _auuc_f

if SCORES is None or not len(SCORES):
    print('§7 을 먼저 돌리십시오.')
    BOOT = None
else:
    _B = int(CFG.get('BOOT_B', 2000))
    _rows = []
    for _nm in [d for d in CFG['TRAIN_DATA'] if d.startswith('empservicereal')]:
        _s, _o, _ = _load_pair(_nm, 0)
        _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
        for _y in (CFG['TRAIN_Y'],) + tuple(CFG.get('TRAIN_Y_EXTRA', {}).get(_nm, ())):
            _ytag = None if _y == CFG['TRAIN_Y'] else _y
            _m = _fit_mask(_s, _o, _h, 2, _y)
            _idx = np.where(_m)[0]
            _a = np.asarray(_o['TREATED'], bool)[_idx]
            _yv = _yh(_o, _y, _h)[_idx]
            # τ̂ — 시드 앙상블(§7c 와 같은 값)을 씁니다
            _TAU = {}
            for _mo in CFG['TRAIN_MODELS']:
                _ts = []
                for _sd in CFG.get('SEEDS_BY', {}).get(_nm, CFG['TRAIN_SEEDS']):
                    _fp = os.path.join(R.DIR_RUNS,
                                       R.run_id(_nm, _mo, _sd, None, _ytag, VERSION), 'tau.npy')
                    if os.path.exists(_fp):
                        _ts.append(np.load(_fp)[_idx])
                if _ts:
                    _TAU[_mo] = np.mean(_ts, 0)
            if len(_TAU) < 2:
                continue
            _rng = np.random.default_rng(20260821)
            _J = [_rng.choice(len(_idx), len(_idx), replace=True) for _ in range(_B)]
            _Q = {k: np.array([_qini_f(v[j], _a[j], _yv[j]) for j in _J])
                  for k, v in _TAU.items()}
            _pt = {k: _qini_f(v, _a, _yv) for k, v in _TAU.items()}
            _best = max(_pt, key=lambda k: _pt[k])
            print('── %s / %s   시험 %d명 (처치 %d · 대조 %d) · 재표본 %d회'
                  % (_nm, _y, len(_idx), _a.sum(), (~_a).sum(), _B))
            for _k in CFG['TRAIN_MODELS']:
                if _k not in _pt:
                    continue
                _b = _Q[_k][np.isfinite(_Q[_k])]
                _r = {'데이터': _nm, '결과': _y, '모델': _k, 'Qini': _pt[_k],
                      '하한': float(np.percentile(_b, 2.5)),
                      '상한': float(np.percentile(_b, 97.5))}
                if _k != _best:
                    _d = _Q[_best] - _Q[_k]
                    _d = _d[np.isfinite(_d)]
                    _r.update({'1위와의차': float(_d.mean()),
                               '차_하한': float(np.percentile(_d, 2.5)),
                               '차_상한': float(np.percentile(_d, 97.5)),
                               'P(1위가더나음)': float((_d > 0).mean())})
                    _r['판정'] = ('1위가 유의하게 낫다' if _r['차_하한'] > 0
                                else '구별 못 함 (표본오차 안)')
                    # 몇 명이면 이 격차를 잴 수 있는가 — 기관이 실제로 쓸 수 있는 숫자입니다.
                    #   구간 반폭은 대략 1/√n 로 줄어듭니다. 지금 반폭이 h, 재려는 격차가 Δ 면
                    #   필요한 시험표본 ≈ n × (h/Δ)².
                    _hw = (_r['차_상한'] - _r['차_하한']) / 2
                    if _r['차_하한'] <= 0 < _r['1위와의차']:
                        _need = len(_idx) * (_hw / _r['1위와의차']) ** 2
                        # 격차가 구간에 견주어 너무 작으면 필요표본이 폭발합니다.
                        # 그건 「엄청난 표본이 필요하다」가 아니라 **사실상 동률**이라는 뜻입니다.
                        if _need > 40 * len(_idx):
                            _r['판정'] = '사실상 동률 (격차가 구간의 1/6 미만)'
                        else:
                            _r['필요_시험표본'] = int(round(_need))
                            _r['필요_전체표본'] = int(round(_need * _s.N / max(len(_idx), 1)))
                else:
                    _r['판정'] = '기준(1위)'
                _rows.append(_r)
                print('   %-9s Qini %9.1f  95%%CI [%9.1f, %9.1f]   %s%s'
                      % (_k, _r['Qini'], _r['하한'], _r['상한'], _r['판정'],
                         ('  · 이 격차를 재려면 시험 %d명(전체 %d명)'
                          % (_r['필요_시험표본'], _r['필요_전체표본']))
                         if _r.get('필요_시험표본') else ''))
            print()
        del _s, _o; gc.collect()
    BOOT = pd.DataFrame(_rows)
    if len(BOOT):
        BOOT.to_csv(os.path.join(DIR_OUT, '07_붓스트랩.csv'), index=False, encoding='utf-8-sig')
        for (_nm, _y), _g in BOOT.groupby(['데이터', '결과']):
            _n_sig = int((_g['판정'] == '1위가 유의하게 낫다').sum())
            _b = _g.loc[_g['판정'] == '기준(1위)', '모델'].iloc[0]
            _nd = _g['필요_전체표본'].dropna() if '필요_전체표본' in _g else []
            REPORT['붓스트랩 %s %s' % (_nm, _y)] = (
                '1위 %s · 유의하게 뒤진 모형 %d/%d (나머지는 표본오차 안)%s'
                % (_b, _n_sig, len(_g) - 1,
                   ('  · 구별하려면 전체 %d명쯤 필요' % int(_nd.max())) if len(_nd) else ''))
            print('  %s %s → %s' % (_nm, _y, REPORT['붓스트랩 %s %s' % (_nm, _y)]))
    step('§7d 붓스트랩 신뢰구간')


In [ ]:
%%time
# ── 7e. 짝지은 비교 — **판을 짝지어야 표가 읽힙니다**
#
# IHDP 100판에서 √PEHE 는 tarnet 1.4952 ± 1.4730, cdt 1.7021 ± 2.3466 입니다.
# 이 ± 는 **판 사이의 흔들림**이지 모형 간 차이의 오차가 아닙니다. IHDP 는 판마다
# 난이도가 크게 달라(가장 쉬운 판 0.3, 가장 어려운 판 30 이상) 수준의 표준편차가
# 커 보일 뿐이고, **같은 판에서 두 모형을 견주면** 그 난이도가 통째로 상쇄됩니다.
# 실데이터에서 짝지은 붓스트랩이 필요했던 것과 똑같은 이유입니다.
#
#   · 짝지은 평균차 + 95% 신뢰구간 (판을 재표본)
#   · 윌콕슨 부호순위 검정 — 몇 판에서 이겼나 (평균은 한 판의 이상치에 끌려갑니다)
#   · 중앙값 — IHDP 는 꼬리가 길어 평균만 싣는 것은 정직하지 않습니다
from scipy import stats as _st

def _wilcoxon_floor(n):
    """
    **윌콕슨 양측검정이 n 개 표본에서 낼 수 있는 가장 작은 p 값.**

    부호순위 검정은 모든 부호가 같아도 2/2^n 보다 작은 p 를 낼 수 없습니다.
    n=5 면 0.0625, n=6 이면 0.03125 입니다. 그래서 v80 에서 **시드 5개로 5/5 전승에
    신뢰구간 [+2.41, +3.14] 인 결과를 「구별 못 함」으로 찍었습니다** — p<0.05 를
    요구했는데 그 조건이 원리상 만족될 수 없었습니다. 검정의 한계를 결론으로
    착각한 것이라 반드시 고쳐야 합니다.
    """
    return 2.0 / (2 ** n) if n < 10 else 0.0

_LOWER = {'√PEHE', 'ε_ATE', 'ε_ATT', '정책위험'}      # 낮을수록 좋은 지표
if SCORES is None or not len(SCORES):
    print('§7 을 먼저 돌리십시오.')
    PAIRED = None
else:
    _rows = []
    _KEY3 = ['데이터', '결과'] if '결과' in SCORES.columns else ['데이터']
    for _key, _g in SCORES.groupby(_KEY3):
        _nm = _key[0] if isinstance(_key, tuple) else _key
        _y = _key[1] if isinstance(_key, tuple) and len(_key) > 1 else CFG['TRAIN_Y']
        _unit = '판' if _g['판'].nunique() > 1 else '시드'
        # **두 짝짓기는 뜻이 다릅니다.**
        #   판 짝짓기  — 다른 자료를 뽑았을 때. 일반화 오차. 논문 표에 그대로 씁니다.
        #   시드 짝짓기 — 같은 사람들, 학습 난수만 다름. **사람 표본의 불확실성은
        #                 전혀 안 들어 있습니다.** 실데이터의 정책 결론은 여기가 아니라
        #                 §7d 의 붓스트랩(사람을 재표본)으로 내야 합니다.
        _seed_only = (_unit == '시드')
        for _met in PRIMARY.get(_nm, []):
            if _met not in _g.columns or not _g[_met].notna().any():
                continue
            _w = _g.pivot_table(index=_unit, columns='모델', values=_met)
            _w = _w.dropna(axis=1, how='all').dropna()
            if _w.shape[0] < 3 or _w.shape[1] < 2:
                continue
            _low = _met in _LOWER
            _best = _w.mean().idxmin() if _low else _w.mean().idxmax()
            print('── %s / %s / %s   %s %d개 · 기준(1위) %s%s'
                  % (_nm, _y, _met, _unit, _w.shape[0], _best,
                     '   ※ 학습 난수만 다름 — 사람 표본 불확실성 아님(§7d 를 보십시오)'
                     if _seed_only else ''))
            _rng2 = np.random.default_rng(20260821)
            _J2 = [_rng2.choice(_w.shape[0], _w.shape[0], replace=True) for _ in range(1000)]
            for _mo in [c for c in CFG['TRAIN_MODELS'] if c in _w.columns]:
                _r = {'데이터': _nm, '결과': _y, '지표': _met, '모델': _mo, '단위': _unit,
                      'n': int(_w.shape[0]), '평균': float(_w[_mo].mean()),
                      '중앙값': float(_w[_mo].median())}
                if _mo != _best:
                    _d = (_w[_mo] - _w[_best]).values         # 낮을수록 좋으면 +면 1위가 낫다
                    if not _low:
                        _d = -_d
                    _bs = np.array([_d[j].mean() for j in _J2])
                    _wil = _st.wilcoxon(_d, alternative='two-sided') if np.any(_d != 0) else None
                    _r.update({'1위와의차': float(_d.mean()),
                               '차_하한': float(np.percentile(_bs, 2.5)),
                               '차_상한': float(np.percentile(_bs, 97.5)),
                               '1위가_이긴_판': int((_d > 0).sum()),
                               'p(윌콕슨)': float(_wil.pvalue) if _wil is not None else np.nan})
                    # 판정: 신뢰구간이 0 을 안 물고 + (윌콕슨 유의 **또는** 전승)
                    #   표본이 작아 윌콕슨이 0.05 밑으로 못 내려가는 경우를 구제합니다.
                    _floor = _wilcoxon_floor(_r['n'])
                    _sweep = _r['1위가_이긴_판'] == _r['n']
                    _r['윌콕슨_최소가능p'] = _floor
                    _r['판정'] = ('1위가 유의하게 낫다'
                                 if _r['차_하한'] > 0 and
                                 (_r['p(윌콕슨)'] < 0.05 or (_floor >= 0.05 and _sweep))
                                 else '구별 못 함')
                else:
                    _r['판정'] = '기준(1위)'
                _r['짝짓기'] = '시드(학습 난수)' if _seed_only else '판(자료 재추출)'
                _rows.append(_r)
                print('   %-9s 평균 %8.4f  중앙값 %8.4f   %s'
                      % (_mo, _r['평균'], _r['중앙값'],
                         (_r['판정'] if _mo == _best else
                          '%s  (짝지은 차 %+.4f  95%%CI [%+.4f, %+.4f] · %d/%d판 · p=%.3g)'
                          % (_r['판정'], _r['1위와의차'], _r['차_하한'], _r['차_상한'],
                             _r['1위가_이긴_판'], _r['n'], _r['p(윌콕슨)']))))
            print()
    PAIRED = pd.DataFrame(_rows)
    if len(PAIRED):
        PAIRED.to_csv(os.path.join(DIR_OUT, '07_짝지은비교.csv'),
                      index=False, encoding='utf-8-sig')
        for (_nm, _y, _met), _g2 in PAIRED.groupby(['데이터', '결과', '지표']):
            _b = _g2.loc[_g2['판정'] == '기준(1위)', '모델']
            if not len(_b):
                continue
            _tie = _g2[_g2['판정'] == '구별 못 함']['모델'].tolist()
            _sd_only = _g2['짝짓기'].iloc[0].startswith('시드')
            _cdt = _g2[_g2['모델'] == 'cdt']
            _extra = ''
            if len(_cdt) and _cdt['판정'].iloc[0] != '기준(1위)':
                _c0 = _cdt.iloc[0]
                _extra = '  · cdt %d/%d판·중앙값 %.4g(1위 %.4g)' % (
                    _c0['n'] - _c0['1위가_이긴_판'], _c0['n'], _c0['중앙값'],
                    _g2.loc[_g2['판정'] == '기준(1위)', '중앙값'].iloc[0])
            REPORT['짝지은%s %s %s (%s)' % ('(시드)' if _sd_only else '', _nm, _y, _met)] = (
                '1위 %s%s%s' % (_b.iloc[0],
                               ('  · 동률: ' + ' '.join(_tie)) if _tie else '  · 모두 유의하게 뒤짐',
                               _extra))
    step('§7e 짝지은 비교')


In [ ]:
%%time
# ── 7f. 표본크기 곡선 — **몇 명부터 복잡한 모형이 값을 하는가** (새 그림 3)
#
# 실데이터에서 상위 네 모형이 전부 구별되지 않았습니다(§7d). 학습표본이 1,345명뿐이기
# 때문인지, 아니면 방법이 원래 비슷한 것인지는 **참값이 있는 자료에서 표본만 줄여 보면**
# 갈립니다. 합성 v2 는 참 ATT 를 알고 있으니 이 실험이 됩니다.
#
#   · 학습·검증만 비율대로 줄이고 **시험표본 1,200명은 고정** — 채점 잣대를 안 바꿉니다
#   · 초매개변수는 §7 이 고른 값 그대로 — n 마다 다시 고르면 「조율의 효과」가 섞입니다
#   · 시드 3개 → 평균 ± 표준편차
import dataclasses as _dc

if not CFG.get('NCURVE') or SCORES is None:
    print('§7f 는 건너뜁니다.')
    NCUR = None
else:
    _nm = CFG.get('ABLATE_DATA', 'empservicesyntv2')
    _s, _o, _tr = _load_pair(_nm, 0)
    _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
    _hp = CDT_HP.get(_nm, {'lr': 1e-2, 'dropout': 0.0})
    _n_full = int((_s.SPLIT == 0).sum())
    print('%s · 전체 학습 %d명 · 시험 %d명 고정 · 초매개변수 %s'
          % (_nm, _n_full, int((_s.SPLIT == 2).sum()), _hp))
    _raw = []
    for _n in CFG['NCURVE_N']:
        if _n > _n_full:
            continue
        for _sd in CFG['NCURVE_SEEDS']:
            # 학습·검증을 같은 비율로 줄입니다. 검증만 크게 두면 작은 n 이 부당하게 유리해집니다.
            _rg = np.random.default_rng(10_000 + _sd)
            # SPLIT 은 uint8 이라 -1 을 못 담습니다. 부호 있는 형으로 바꿔서 뺍니다.
            _sp = _s.SPLIT.astype(np.int16).copy()
            _frac = _n / _n_full
            for _c in (0, 1):
                _ix = np.where(_s.SPLIT == _c)[0]
                _keep = _rg.choice(_ix, max(20, int(round(len(_ix) * _frac))), replace=False)
                _drop = np.setdiff1d(_ix, _keep)
                _sp[_drop] = -1                      # 어느 분할에도 안 들어감
            _s2 = _dc.replace(_s, SPLIT=_sp)
            for _m in CFG['TRAIN_MODELS']:
                _t0 = time.time()
                try:
                    _kw = CDT_KW(**_hp) if _m == 'cdt' else {}
                    _f = mdl.MODELS[_m](seed=_sd).fit(_s2, _o, h=_h, y=CFG['TRAIN_Y'],
                                                      split=0, **_kw)
                    _tau = _f.predict_tau(_s2, _o, h=_h)
                    _sc = mdl.score(_nm, _tau, _s2, _o, h=_h, y=CFG['TRAIN_Y'],
                                    split=2, truth=_tr)
                    _raw.append({'n학습': int((_sp == 0).sum()), '요청n': _n, '시드': _sd,
                                 '모델': _m, '√PEHE': _sc.get('√PEHE'),
                                 'ε_ATE': _sc.get('ε_ATE'), '초': round(time.time() - _t0, 1)})
                except Exception as _e:
                    print('  ⚠ n=%d %s 실패: %s' % (_n, _m, str(_e)[:80]))
        print('  n=%-5d 끝 (%d행)' % (_n, len(_raw)), flush=True)
    NCUR_RAW = pd.DataFrame(_raw)
    NCUR_RAW.to_csv(os.path.join(DIR_OUT, '07_표본크기_원자료.csv'),
                    index=False, encoding='utf-8-sig')
    NCUR = (NCUR_RAW.groupby(['요청n', '모델'], sort=False)
            .agg(**{'√PEHE': ('√PEHE', 'mean'), 'sd': ('√PEHE', 'std'),
                    'ε_ATE': ('ε_ATE', 'mean'), '실행수': ('시드', 'count')})
            .reset_index())
    print()
    print(NCUR.pivot(index='요청n', columns='모델', values='√PEHE')
          [list(CFG['TRAIN_MODELS'])].round(2).to_string())

    # ── 몇 명부터 cdt 가 최고 기준모형과 갈리는가 (시드를 짝지어 비교)
    _W = NCUR_RAW.pivot_table(index=['요청n', '시드'], columns='모델', values='√PEHE')
    _lines = []
    for _n, _g in _W.groupby(level=0):
        _bs = [c for c in _g.columns if c != 'cdt']
        _best = _g[_bs].mean().idxmin()
        _d = (_g[_best] - _g['cdt']).values          # +면 cdt 가 낫다
        _lo, _hi = np.percentile(
            [np.mean(_d[np.random.default_rng(_i).choice(len(_d), len(_d))])
             for _i in range(500)], [2.5, 97.5])
        _v = ('cdt 가 유의하게 낫다' if _lo > 0 else
              ('%s 가 유의하게 낫다' % _best if _hi < 0 else '구별 못 함'))
        _lines.append({'n학습': _n, '최고 기준모형': _best, 'cdt−기준 차': float(_d.mean()),
                       '하한': float(_lo), '상한': float(_hi), '판정': _v})
    NCUR_CMP = pd.DataFrame(_lines)
    NCUR_CMP.to_csv(os.path.join(DIR_OUT, '07_표본크기_판정.csv'),
                    index=False, encoding='utf-8-sig')
    print()
    print(NCUR_CMP.to_string(index=False))
    _win = NCUR_CMP[NCUR_CMP['판정'] == 'cdt 가 유의하게 낫다']
    REPORT['표본크기 문턱'] = ('학습 %d명부터 cdt 가 유의하게 나음' % _win['n학습'].min()
                          if len(_win) else '어떤 크기에서도 유의한 우위 없음')
    print('  →', REPORT['표본크기 문턱'])

    # ── 그림 3 (새 판)
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    _EN = {'naive': 'naive', 'slearner': 'S-learner', 'tlearner': 'T-learner',
           'cforest': 'causal forest', 'tarnet': 'TARNet', 'cdt': 'CDT (ours)'}
    for _m in CFG['TRAIN_MODELS']:
        _d = NCUR[NCUR['모델'] == _m]
        if not len(_d) or _d['√PEHE'].isna().all():
            continue
        ax.errorbar(_d['요청n'], _d['√PEHE'], yerr=_d['sd'].fillna(0),
                    fmt='o-' if _m == 'cdt' else 's--', capsize=2, alpha=1 if _m == 'cdt' else .6,
                    linewidth=2 if _m == 'cdt' else 1, label=_EN.get(_m, _m))
    ax.set_xscale('log'); ax.set_xticks(list(NCUR['요청n'].unique()))
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('training sample size'); ax.set_ylabel('sqrt(PEHE)')
    # [100] 1,345 는 옛 2,000명 코호트의 학습표본이었습니다. 실제 값을 읽어 씁니다.
    try:
        _n_real = len(dio.load_std('empservicereal', 0).idx('train'))
    except Exception:
        _n_real = None
    if _n_real:
        _xs = min(_n_real, int(max(NCUR['요청n'])))
        ax.axvline(_xs, color='k', ls=':', lw=1)
        ax.text(_xs, ax.get_ylim()[1], ' real data (n=%s%s)'
                % (format(_n_real, ','), '' if _n_real <= max(NCUR['요청n']) else ', 축 밖'),
                fontsize=7, va='top', ha='right')
    ax.legend(fontsize=7, ncol=2)
    ax.set_title('Fig 3.  How much data before the sequence model pays off?', fontsize=9)
    fig.tight_layout()
    _fp = os.path.join(CONF_ROOT, 'paper', 'fig', 'fig3_n.png')
    os.makedirs(os.path.dirname(_fp), exist_ok=True)
    fig.savefig(_fp, dpi=300); plt.show()
    print('그림 3 저장: %s' % _fp)
    del _s, _s2, _o, _tr; gc.collect()
    step('§7f 표본크기 곡선')


In [ ]:
%%time
# ── 8-1. K 절단 — 그림 3 (두 곡선)
#
# v76 에서 K 를 1→120 으로 늘려도 √PEHE 가 평평했습니다(31.8 / 30.5 / 31.9 / 31.8 / 31.4 / 31.5,
# 흔들림 ±0.93). 원인을 찾았습니다: **상태 82열 중 68열이 이미 생애 누적 요약**입니다
# (`상태_생애_직종c**누적경력기간` 48열 · `효과잔존량` 20열). K=1 도 이미 이력을 다 봅니다.
#
# 그래서 곡선을 둘 그립니다.
#   ① 전체 상태      — 누적 요약이 있는 판. 평평하면 「요약이 충분하다」는 뜻
#   ② 누적요약 제거   — 그 68열을 막은 판. 이력은 오직 시퀀스로만 살아납니다
# ②가 K 를 따라 내려가면 「시퀀스가 요약을 복원한다」가 증명됩니다. 그것이 이 논문의 주장입니다.
# 안 내려가면 주장을 바꿔야 합니다 — 그 결과도 그대로 싣습니다.
if not (CFG.get('ABLATE') and CFG.get('ABLATE_KCURVE', True)) or SCORES is None:
    print('§8-1 K 곡선은 건너뜁니다 (ABLATE_KCURVE=False 이거나 §7 을 안 돌렸습니다).')
    ABL_K = None
else:
    _nm = CFG['ABLATE_DATA']
    _s, _o, _tr = _load_pair(_nm, 0)
    _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
    _hp = CDT_HP.get(_nm, {'lr': 1e-2, 'dropout': 0.1})
    _ARMS = [('전체 상태', None), ('누적요약 제거', CFG['ABLATE_DROP'])]
    print('K 절단 — %s · 고정 초매개변수 %s' % (_nm, _hp))
    _raw = []
    for _arm, _drop in _ARMS:
        for _K in CFG['ABLATE_K']:
            if _K > _s.T:
                continue
            for _sd in CFG['ABLATE_SEEDS']:
                _t0 = time.time()
                _f = mdl.MODELS['cdt'](seed=_sd).fit(
                    _s, _o, h=_h, y=CFG['TRAIN_Y'], split=0, state_drop=_drop,
                    **CDT_KW(K=_K, lr=_hp['lr'], dropout=_hp.get('dropout', 0.0)))
                _tau = _f.predict_tau(_s, _o, h=_h)
                _sc = mdl.score(_nm, _tau, _s, _o, h=_h, y=CFG['TRAIN_Y'], split=2, truth=_tr)
                _raw.append({'판': _arm, 'K': _K, '시드': _sd, '상태열': _f.state_kept,
                             '√PEHE': _sc.get('√PEHE'), 'ε_ATE': _sc.get('ε_ATE'),
                             'τ평균': float(_tau.mean()), 'τsd': float(_tau.std()),
                             'best_epoch': _f.best_epoch, '초': round(time.time() - _t0, 1)})
                print('  %-8s K=%-4d s%d  상태열 %d  √PEHE %s  ε_ATE %s  (%.0f초)'
                      % (_arm, _K, _sd, _f.state_kept, _raw[-1]['√PEHE'],
                         _raw[-1]['ε_ATE'], _raw[-1]['초']), flush=True)
    ABL_K_RAW = pd.DataFrame(_raw)
    ABL_K_RAW.to_csv(os.path.join(DIR_OUT, '08_K절단_원자료.csv'),
                     index=False, encoding='utf-8-sig')
    ABL_K = (ABL_K_RAW.groupby(['판', 'K'], sort=False)
             .agg(**{'√PEHE': ('√PEHE', 'mean'), '√PEHE_sd': ('√PEHE', 'std'),
                     'ε_ATE': ('ε_ATE', 'mean'), 'ε_ATE_sd': ('ε_ATE', 'std'),
                     '상태열': ('상태열', 'first'), '실행수': ('시드', 'count')})
             .reset_index())
    print()
    print(ABL_K.to_string(index=False))

    # ── 그림 3
    _p = os.path.join(DIR_OUT, 'syn05_대리충분성.csv')
    _r2 = pd.read_csv(_p) if os.path.exists(_p) else None
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    _STY = {'전체 상태': ('o-', '#1f77b4', 'full state (cumulative summaries kept)'),
            '누적요약 제거': ('s-', '#2ca02c', 'cumulative summaries removed')}
    for _arm in ABL_K['판'].unique():
        _d = ABL_K[ABL_K['판'] == _arm]
        _f2, _c2, _lb = _STY.get(_arm, ('o-', '#333333', _arm))
        ax.errorbar(_d['K'], _d['√PEHE'], yerr=_d['√PEHE_sd'].fillna(0),
                    fmt=_f2, capsize=3, color=_c2, label=_lb)
    ax.set_xscale('log'); ax.set_xticks(sorted(ABL_K['K'].unique()))
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('history length K (months)'); ax.set_ylabel('sqrt(PEHE)')
    if _r2 is not None and len(_r2):
        _c = [c for c in _r2.columns if c.startswith('U(')]
        if _c:
            ax2 = ax.twinx()
            ax2.plot(_r2.iloc[:, 0], _r2[_c[0]], '^--', color='#d62728', alpha=.6,
                     label='R2 of U recovered from history')
            ax2.set_ylabel('R2 (hidden confounder U)')
            _h1, _l1 = ax.get_legend_handles_labels()
            _h2, _l2 = ax2.get_legend_handles_labels()
            ax.legend(_h1 + _h2, _l1 + _l2, fontsize=7, loc='best')
        else:
            ax.legend(fontsize=7)
    else:
        ax.legend(fontsize=7)
    ax.set_title('Fig 3.  Does the sequence recover what the summary already knows?',
                 fontsize=8.5)
    fig.tight_layout()
    _fp = os.path.join(CONF_ROOT, 'paper', 'fig', 'fig3_K.png')
    os.makedirs(os.path.dirname(_fp), exist_ok=True)
    fig.savefig(_fp, dpi=300); plt.show()
    print('그림 3 저장: %s' % _fp)

    # ── 판정 — 팔마다 「K 효과가 흔들림보다 큰가」
    for _arm in ABL_K['판'].unique():
        _d = ABL_K[ABL_K['판'] == _arm]
        _rng = float(_d['√PEHE'].max() - _d['√PEHE'].min())
        _noise = float(_d['√PEHE_sd'].mean(skipna=True) or 0)
        _msg = ('K 효과 %.2f > 흔들림 %.2f×2 — 읽을 수 있음' % (_rng, _noise)
                if _rng > 2 * _noise else
                'K 효과 %.2f ≤ 흔들림 %.2f×2 — **읽을 수 없음**' % (_rng, _noise))
        REPORT['그림3 %s' % _arm] = '%s  (%s)' % (
            ' · '.join('K%d %.2f' % (r['K'], r['√PEHE']) for _, r in _d.iterrows()), _msg)
        print('  %-8s %s' % (_arm, _msg))
    del _s, _o, _tr; gc.collect()
    step('§8-1 K 절단')


In [ ]:
%%time
# ── 8-2. 구조 소거 — 표 5
if not (CFG.get('ABLATE') and CFG.get('ABLATE_STRUCT', True)) or SCORES is None:
    print('§8-2 구조 소거(표 5)는 건너뜁니다 — ABLATE_STRUCT=False 이거나 §7 을 안 돌렸습니다.')
    ABL_S = None
else:
    _VAR = [('전체(제안)',        {}),
            ('MOT 제거',          dict(mot=False)),
            ('이중머리 제거',      dict(dual=False)),
            ('인과→양방향 어텐션',  dict(causal=False)),
            ('K=1',              dict(K=1)),
            ('행동손실 제거',      dict(lambda_action=0.0))]
    _rows = []
    _DATA8 = list(dict.fromkeys([CFG['ABLATE_DATA'], 'ihdp']))   # 같은 자료를 두 번 돌지 않게
    for _nm in _DATA8:
        _s, _o, _tr = _load_pair(_nm, 0)
        _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
        _hp = CDT_HP.get(_nm, {'lr': 1e-2, 'dropout': 0.1})
        for _lab, _ov in _VAR:
          if _lab == 'K=1' and _s.T <= 1:
              continue          # 시점이 하나면 K=1 이 곧 전체입니다 — 같은 줄을 두 번 싣지 않습니다
          for _sd in CFG['ABLATE_SEEDS']:
            _t0 = time.time()
            try:
                _f = mdl.MODELS['cdt'](seed=_sd).fit(
                    _s, _o, h=_h, y=CFG['TRAIN_Y'], split=0,
                    **CDT_KW(lr=_hp['lr'], dropout=_hp.get('dropout', 0.0), **_ov))
                _tau = _f.predict_tau(_s, _o, h=_h)
                _sc = mdl.score(_nm, _tau, _s, _o, h=_h, y=CFG['TRAIN_Y'],
                                split=2, truth=_tr)
                _rows.append({'데이터': _nm, '판': _lab, '시드': _sd,
                              '√PEHE': _sc.get('√PEHE'), 'ε_ATE': _sc.get('ε_ATE'),
                              'ε_ATT': _sc.get('ε_ATT'),
                              'τ평균': float(_tau.mean()), 'τsd': float(_tau.std()),
                              'best_epoch': _f.best_epoch, '초': round(time.time() - _t0, 1)})
            except Exception as _e:
                import traceback as _tb
                print('  ⚠ %s / %s 실패: %s' % (_nm, _lab, _e)); print(_tb.format_exc())
                _rows.append({'데이터': _nm, '판': _lab, '시드': _sd, '오류': str(_e)[:80]})
            print('  %-16s %-14s s%d  %s' % (_nm, _lab, _sd, _rows[-1]), flush=True)
        del _s, _o, _tr; gc.collect()
    ABL_S_RAW = pd.DataFrame(_rows)
    ABL_S_RAW.to_csv(os.path.join(DIR_OUT, '08_구조소거_원자료.csv'),
                     index=False, encoding='utf-8-sig')
    # 표 5 — 평균 ± 표준편차. 흔들림 안에 들어오는 차이는 「차이 없음」으로 읽어야 합니다.
    _M = '√PEHE'
    _ok = ABL_S_RAW[ABL_S_RAW.get('오류').isna()] if '오류' in ABL_S_RAW else ABL_S_RAW
    ABL_S = (_ok.groupby(['데이터', '판'], sort=False)
             .agg(**{'평균': (_M, 'mean'), '표준편차': (_M, 'std'),
                     'ε_ATE': ('ε_ATE', 'mean'), 'τ평균': ('τ평균', 'mean'),
                     'τsd': ('τsd', 'mean'), '실행수': ('시드', 'count')})
             .reset_index())
    # 전체(제안) 대비 차이가 흔들림의 2배를 넘는가
    _base = {d: g.loc[g['판'] == '전체(제안)', '평균'].mean()
             for d, g in ABL_S.groupby('데이터')}
    ABL_S['전체대비'] = [r['평균'] - _base.get(r['데이터'], float('nan'))
                       for _, r in ABL_S.iterrows()]
    ABL_S['판정'] = ['기준' if r['판'] == '전체(제안)' else
                    ('의미 있음' if abs(r['전체대비']) > 2 * (r['표준편차'] or 0)
                     else '흔들림 안')
                    for _, r in ABL_S.iterrows()]
    ABL_S.to_csv(os.path.join(DIR_OUT, '08_구조소거.csv'), index=False, encoding='utf-8-sig')
    print()
    print(ABL_S.to_string(index=False))
    REPORT['표5 구조소거'] = '%d개 판 × %d자료 × 시드 %d' % (
        len(_VAR), len(_DATA8), len(CFG['ABLATE_SEEDS']))
    step('§8-2 구조 소거')


In [ ]:
# 실행 목록 — 어떤 것이 끝났고 어떤 것이 죽었는지
_rt = R.list_runs()
if len(_rt):
    print(_rt.tail(30).to_string(index=False))
    _rt.to_csv(os.path.join(DIR_OUT, '07_실행목록.csv'), index=False, encoding='utf-8-sig')
else:
    print('runs/ 가 비어 있습니다. 아직 학습을 안 돌리셨습니다.')

In [ ]:
# ═══ §93. 오라클 상수 판별(표 2) · PEHE 분해 — run93 을 흡수 ═══
# ═══ 설정 ═══
import os, sys, glob, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.')); sys.path.insert(0, os.path.abspath('tools'))
import numpy as np, pandas as pd
from cdtlib import dataio as dio, outcomes as oc, models as mdl, env as cenv
from cdtlib import config as _cfg
from cdtlib.config import DIR_OUT, DIR_RUNS

REAL_COHORT='r16k'; TRAIN_H=1; VER='v%d' % VERSION   # [100] 이번 실행의 tau 를 읽습니다
_cfg.use_cohort(REAL_COHORT); cenv.setup_matplotlib()
cenv.check_lib()
MODELS=('naive','slearner','tlearner','cforest','tarnet','cdt')
# 판이 여럿인 자료는 판마다 재야 합니다. ihdp 100판 · jobs 는 참값이 없어 제외.
TASKS=[('empservicesyntv2','Y3',(0,1,2,3,4),[None]),
       ('ihdp','Y3',(0,1,2),list(range(100)))]

def h_of(s): return 0 if s.T<=1 else TRAIN_H
def load_tau(nm,m,sd,rep=None,ytag=None):
    p='%s_%s_%s_s%d'%(VER,nm,m,sd)
    if ytag: p+='_'+ytag
    if rep is not None: p+='_r%03d'%rep
    f=os.path.join(DIR_RUNS,p,'tau.npy')
    return np.load(f) if os.path.exists(f) else None
print('준비 완료')

In [ ]:
%%time
# ═══ A′. 판마다 오라클 상수 vs 모형 ═══
# τ̂=0 으로 채점하면  √PEHE = sqrt(E[τ²]) · ε_ATE = |E[τ]|  이므로
#   sd(τ) = sqrt(E[τ²] − E[τ]²)  = 「참 평균효과를 모두에게 주는 답」의 √PEHE.
# 이보다 잘하는 상수는 존재하지 않습니다.
rows=[]
for nm,y,seeds,reps in TASKS:
    for rep in reps:
        r0 = 0 if rep is None else rep
        s=dio.load_std(nm,r0); o=oc.for_dataset(s,nm,r0)
        t=dio.load_truth(nm) if nm=='empservicesyntv2' else None
        z=np.zeros(s.N,float)
        s0=mdl.score(nm,z,s,o,h=h_of(s),y=y,split=2,truth=t)
        rms=float(s0.get('√PEHE',np.nan)); ate=float(s0.get('ε_ATE',np.nan))
        sdt=float(np.sqrt(max(rms**2-ate**2,0.0)))
        for m in MODELS:
            vs=[]
            for sd_ in seeds:
                tau=load_tau(nm,m,sd_,rep)
                if tau is None: continue
                sc=mdl.score(nm,np.asarray(tau,float).reshape(-1),s,o,h=h_of(s),y=y,split=2,truth=t)
                if sc.get('√PEHE') is not None: vs.append(dict(pehe=float(sc['√PEHE']),ate=float(sc['ε_ATE'])))
            if not vs: continue
            pe=float(np.mean([v['pehe'] for v in vs])); ae=float(np.mean([v['ate'] for v in vs]))
            rows.append({'데이터':nm,'판':r0,'모델':m,'오라클상수 sd(τ)':sdt,'참 |ATE|':ate,
                         '모형 √PEHE':pe,'모형 ε_ATE':ae,'모형이 더 나음':pe<sdt})
P=pd.DataFrame(rows)
P.to_csv(os.path.join(DIR_OUT,'93A_오라클상수_판별.csv'),index=False,encoding='utf-8-sig')

print('── 판마다 짝지어 비교: 모형이 오라클 상수를 이긴 판의 비율')
g=P.groupby(['데이터','모델']).agg(**{'판수':('판','nunique'),
        '오라클 sd(τ) 평균':('오라클상수 sd(τ)','mean'),
        '모형 √PEHE 평균':('모형 √PEHE','mean'),
        '이긴 판 비율':('모형이 더 나음','mean')}).round(4)
g['오라클 대비'] = (g['오라클 sd(τ) 평균']/g['모형 √PEHE 평균']).round(3)
print(g.to_string())

In [ ]:
%%time
# ═══ E. PEHE 분해 — 수준 오차 vs 형태 오차 ═══
#   PEHE = E[(τ̂−τ)²] = (E[τ̂]−E[τ])² + Var(τ̂−τ)
#   앞이 **수준**(평균을 맞췄나), 뒤가 **형태**(개인차를 맞췄나).
#   형태 오차가 sd(τ) 보다 크면 → 그 모형이 만든 이질성은 **도움이 아니라 해**입니다
#   (개인차를 아예 예측하지 않는 편이 나았다는 뜻).
E=P.copy()
E['수준오차²']=E['모형 ε_ATE']**2
E['형태오차']=np.sqrt(np.maximum(E['모형 √PEHE']**2-E['수준오차²'],0.0))
E['형태오차 / sd(τ)']=E['형태오차']/E['오라클상수 sd(τ)']
E['이질성이 도움됨']=E['형태오차 / sd(τ)']<1.0
E.to_csv(os.path.join(DIR_OUT,'93E_PEHE분해.csv'),index=False,encoding='utf-8-sig')
q=E.groupby(['데이터','모델']).agg(**{'√PEHE':('모형 √PEHE','mean'),
      '수준오차':('모형 ε_ATE','mean'),'형태오차':('형태오차','mean'),
      'sd(τ)':('오라클상수 sd(τ)','mean'),
      '형태/sd(τ)':('형태오차 / sd(τ)','mean'),
      '이질성 도움된 판%':('이질성이 도움됨','mean')}).round(4)
q['이질성 도움된 판%']=(q['이질성 도움된 판%']*100).round(1)
print('── 형태오차가 sd(τ) 보다 크면, 그 모형의 개인화는 오히려 해입니다')
print(q.to_string())
print()
print('해석: 형태/sd(τ) < 1 이면 개인차 예측이 순이득, > 1 이면 순손실입니다.')

In [ ]:
# ═══ §92. 수축(α)과 지표 — run92 를 흡수. 92A(판0 상수를 100판 평균과 비교)는 폐기 ═══
# ═══════════════════ 설정 ═══════════════════

import os, sys, json, time, glob, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('tools'))
import numpy as np, pandas as pd

from cdtlib import dataio as dio, outcomes as oc, models as mdl, env as cenv
from cdtlib import config as _cfg
from cdtlib.config import DIR_OUT, DIR_RUNS

REAL_COHORT = 'r16k'      # v91 과 같아야 합니다
TRAIN_H     = 1
VER         = 'v%d' % VERSION   # [100] 이번 실행의 tau
ALPHAS      = np.round(np.concatenate([np.arange(0.0, 2.01, 0.1), [2.5, 3.0, 4.0]]), 2)

_cfg.use_cohort(REAL_COHORT)
FONT = cenv.setup_matplotlib()
os.makedirs(DIR_OUT, exist_ok=True)
print('라이브러리 지문 확인'); cenv.check_lib()
print('실행 폴더:', DIR_RUNS, '·', len(glob.glob(os.path.join(DIR_RUNS, VER + '_*'))), '개')
print('α 격자:', list(ALPHAS))

In [ ]:
# ═══════ 공통 도구 ═══════
_CACHE = {}
def load_pair(nm, rep=0):
    k = (nm, rep)
    if k not in _CACHE:
        s = dio.load_std(nm, rep)
        o = oc.for_dataset(s, nm, rep)
        t = dio.load_truth(nm) if nm == 'empservicesyntv2' else None
        _CACHE[k] = (s, o, t)
    return _CACHE[k]

def person_split(s):
    """사람 단위 분할 벡터 (0 train · 1 valid · 2 test)."""
    sp = np.asarray(s.SPLIT)
    if sp.ndim > 1:
        sp = sp[:, 0]
    return sp.reshape(-1)

def h_of(s):
    return 0 if s.T <= 1 else TRAIN_H

def run_dir(nm, model, seed, rep=None, ytag=None):
    p = '%s_%s_%s_s%d' % (VER, nm, model, seed)
    if ytag:  p += '_' + ytag
    if rep is not None: p += '_r%03d' % rep
    return os.path.join(DIR_RUNS, p)

def load_tau(nm, model, seed, rep=None, ytag=None):
    f = os.path.join(run_dir(nm, model, seed, rep, ytag), 'tau.npy')
    return np.load(f) if os.path.exists(f) else None

def shrink(tau, alpha, base):
    """평균(base) 쪽으로 α배. α=0 상수 · α=1 원래 · α>1 팽창."""
    return base + alpha * (np.asarray(tau, float) - base)

def score1(nm, tau, s, o, t, y, split):
    try:
        return mdl.score(nm, tau, s, o, h=h_of(s), y=y, split=split, truth=t)
    except Exception as e:
        return {'오류': '%s: %s' % (type(e).__name__, e)}

MODELS = ('naive', 'slearner', 'tlearner', 'cforest', 'tarnet', 'cdt')
# (자료, 결과, 시드들, 판) — ihdp·jobs 는 판이 많아 대표 20판만 씁니다(α 주사는 곡선이라 충분)
TASKS = [('empservicesyntv2', 'Y3', (0,1,2,3,4), [None], None),
         ('empservicereal',   'Y3', (0,1,2,3,4), [None], None),
         ('empservicereal',   'Y1', (0,1,2,3,4), [None], 'Y1'),
         ('ihdp',             'Y3', (0,1,2),     list(range(20)), None)]
print('과제:', [(a,b,len(d)) for a,b,c,d,e in TASKS])

In [ ]:
%%time
# ═══════════════════ B · C. α 주사 ═══════════════════
# τ_α = τ̄ + α(τ̂ − τ̄).  τ̄ 는 **학습분할**에서 구합니다(시험 정보를 안 씁니다).
rowsB, chk = [], []
for nm, y, seeds, reps, ytag in TASKS:
    for rep in reps:
        s, o, t = load_pair(nm, rep or 0)
        sp = person_split(s)
        tr_mask = (sp == 0)
        for m in MODELS:
            for sd_ in seeds:
                tau = load_tau(nm, m, sd_, rep, ytag)
                if tau is None:
                    continue
                tau = np.asarray(tau, float).reshape(-1)
                base = float(np.nanmean(tau[tr_mask])) if tr_mask.any() else float(np.nanmean(tau))
                for a in ALPHAS:
                    sc = score1(nm, shrink(tau, a, base), s, o, t, y, 2)
                    if '오류' in sc:
                        continue
                    rowsB.append({'데이터': nm, '결과': y, '모델': m, '시드': sd_, '판': rep, 'α': a,
                                  '√PEHE': sc.get('√PEHE'), 'ε_ATE': sc.get('ε_ATE'),
                                  'ε_ATT': sc.get('ε_ATT'), '정책위험': sc.get('정책위험'),
                                  'AUUC': sc.get('AUUC'), 'Qini': sc.get('Qini'),
                                  'τ평균': float(np.nanmean(shrink(tau, a, base))),
                                  'τsd': float(np.nanstd(shrink(tau, a, base)))})
                    if a == 1.0:
                        chk.append(rowsB[-1])
B = pd.DataFrame(rowsB)
B.to_csv(os.path.join(DIR_OUT, '92B_알파주사_원자료.csv'), index=False, encoding='utf-8-sig')
print('행', len(B))

# ── 자체검사: α=1 이 v91 원본과 같은가
C0 = pd.DataFrame(chk)
orig = pd.read_csv(os.path.join(DIR_OUT, '07_모델비교.csv'))
bad = []
for _, r in C0.iterrows():
    q = orig[(orig['데이터'] == r['데이터']) & (orig['모델'] == r['모델']) &
             (orig['시드'] == r['시드']) & (orig['결과'] == r['결과'])]
    if r['판'] is not None and not pd.isna(r['판']):
        q = q[q['판'] == r['판']]
    if not len(q):
        continue
    for k in ('√PEHE', 'ε_ATE', 'Qini'):
        a_, b_ = r.get(k), q.iloc[0].get(k)
        if pd.notna(a_) and pd.notna(b_) and abs(float(a_) - float(b_)) > 1e-6 * max(1, abs(float(b_))):
            bad.append((r['데이터'], r['모델'], r['시드'], k, float(a_), float(b_)))
print('α=1 대조 — 어긋난 항목 %d개' % len(bad))
for x in bad[:8]:
    print('   ', x)
if len(bad) > max(3, 0.01 * len(C0)):
    raise RuntimeError('α=1 이 원본과 다릅니다. 수축 코드나 분할 처리를 다시 보십시오.')
print('자체검사 통과 — 아래 곡선은 v91 과 같은 채점 경로입니다.')

In [ ]:
%%time
# ═══════ B 요약: 지표별 최적 α ═══════
LOW  = ('√PEHE', 'ε_ATE', 'ε_ATT', '정책위험')     # 작을수록 좋음
HIGH = ('AUUC', 'Qini')                            # 클수록 좋음
rows = []
for (nm, y, m), g in B.groupby(['데이터', '결과', '모델']):
    a = g.groupby('α').mean(numeric_only=True)
    for k in LOW + HIGH:
        if k not in a or a[k].isna().all():
            continue
        best = a[k].idxmin() if k in LOW else a[k].idxmax()
        rows.append({'데이터': nm, '결과': y, '모델': m, '지표': k, '최적 α': best,
                     'α=1 값': round(float(a.loc[1.0, k]), 4) if 1.0 in a.index else np.nan,
                     '최적 값': round(float(a.loc[best, k]), 4),
                     '개선율%': round(abs(float(a.loc[best, k]) - float(a.loc[1.0, k]))
                                    / max(abs(float(a.loc[1.0, k])), 1e-9) * 100, 1) if 1.0 in a.index else np.nan})
OPT = pd.DataFrame(rows)
OPT.to_csv(os.path.join(DIR_OUT, '92B_최적알파.csv'), index=False, encoding='utf-8-sig')
print('── 지표별 최적 α (모형 평균)')
print(OPT.groupby(['데이터', '결과', '지표'])['최적 α'].mean().round(2).to_string())
print()
print('── 모형별 상세')
print(OPT.to_string(index=False))

# ═══════ C. CDT 를 팽창시키면 Qini 가 오르는가 ═══════
print()
print('══ C. CDT 팽창 시험 ══')
for (nm, y), g in B[B['모델'] == 'cdt'].groupby(['데이터', '결과']):
    k = 'Qini' if g['Qini'].notna().any() else '√PEHE'
    a = g.groupby('α')[k].mean()
    if k not in HIGH:
        continue
    top = B[(B['데이터'] == nm) & (B['결과'] == y) & (B['α'] == 1.0)].groupby('모델')[k].mean()
    best_other = top.drop('cdt', errors='ignore').max()
    print('  %s %s : cdt α=1 %.1f · 최고 α=%.1f 에서 %.1f · 다른 모형 최고(α=1) %.1f  → %s'
          % (nm, y, a.get(1.0, np.nan), a.idxmax(), a.max(), best_other,
             '순서는 맞았고 크기만 작았음' if a.max() >= best_other else '순서 자체를 못 잡음'))

In [ ]:
%%time
# ═══════════════════ D. 처방 — 검증분할에서 α 를 고릅니다 ═══════════════════
# 시험 정보를 안 쓰고 α 를 고를 수 있는가, 그리고 그것이 도움이 되는가.
rowsD = []
for nm, y, seeds, reps, ytag in TASKS:
    for rep in reps:
        s, o, t = load_pair(nm, rep or 0)
        sp = person_split(s)
        tr_mask = (sp == 0)
        for m in MODELS:
            for sd_ in seeds:
                tau = load_tau(nm, m, sd_, rep, ytag)
                if tau is None:
                    continue
                tau = np.asarray(tau, float).reshape(-1)
                base = float(np.nanmean(tau[tr_mask])) if tr_mask.any() else float(np.nanmean(tau))
                for k in ('√PEHE', 'Qini'):
                    va, te = {}, {}
                    for a in ALPHAS:
                        v1 = score1(nm, shrink(tau, a, base), s, o, t, y, 1).get(k)
                        v2 = score1(nm, shrink(tau, a, base), s, o, t, y, 2).get(k)
                        if v1 is not None and v1 == v1: va[a] = float(v1)
                        if v2 is not None and v2 == v2: te[a] = float(v2)
                    if not va or not te:
                        continue
                    a_star = (min(va, key=va.get) if k in LOW else max(va, key=va.get))
                    rowsD.append({'데이터': nm, '결과': y, '모델': m, '시드': sd_, '판': rep, '지표': k,
                                  'α*(검증)': a_star, '시험 α=1': te.get(1.0),
                                  '시험 α*': te.get(a_star),
                                  '시험 최선': (min(te.values()) if k in LOW else max(te.values()))})
D = pd.DataFrame(rowsD)
D.to_csv(os.path.join(DIR_OUT, '92D_알파처방.csv'), index=False, encoding='utf-8-sig')
g = D.groupby(['데이터', '결과', '지표']).agg(**{
        'α*평균': ('α*(검증)', 'mean'),
        '시험 α=1': ('시험 α=1', 'mean'),
        '시험 α*': ('시험 α*', 'mean'),
        '시험 최선': ('시험 최선', 'mean')}).round(4)
g['처방 이득%'] = ((g['시험 α=1'] - g['시험 α*']).abs() / g['시험 α=1'].abs() * 100).round(1)
print(g.to_string())
print()
print('α* 가 시험 최선에 얼마나 가까운가 = 검증으로 고르는 것이 되는가')

In [ ]:
# ═══════════════════ 그림 ═══════════════════
import matplotlib.pyplot as plt
os.makedirs(os.path.join(DIR_OUT, '..', 'paper', 'fig'), exist_ok=True)
FIG = os.path.join(DIR_OUT, '..', 'paper', 'fig')

# Fig — 같은 예측을 α로 늘리고 줄일 때 두 지표가 반대로 움직인다
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5))
sub = B[(B['데이터'] == 'empservicereal') & (B['결과'] == 'Y3') & (B['모델'] == 'cdt')]
if len(sub):
    a = sub.groupby('α').mean(numeric_only=True)
    axes[0].plot(a.index, a['Qini'], 'o-', color='#1f77b4')
    axes[0].axvline(1.0, ls='--', c='gray'); axes[0].set_xlabel('shrinkage alpha')
    axes[0].set_ylabel('Qini (targeting)'); axes[0].set_title('real data, CDT')
sub2 = B[(B['데이터'] == 'empservicesyntv2') & (B['모델'] == 'cdt')]
if len(sub2):
    a2 = sub2.groupby('α').mean(numeric_only=True)
    axes[1].plot(a2.index, a2['√PEHE'], 's-', color='#d62728')
    axes[1].axvline(1.0, ls='--', c='gray'); axes[1].set_xlabel('shrinkage alpha')
    axes[1].set_ylabel('sqrt(PEHE)'); axes[1].set_title('synthetic, CDT')
for ax in axes: ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'fig_alpha.png'), dpi=200)
print('그림 저장:', os.path.join(FIG, 'fig_alpha.png'))
plt.show()

In [ ]:
%%time
# ═══ [100A] 학습가능성 오라클 — 표 3 의 근거 ═══
# IHDP: 참 τ = MU1 − MU0 (판 20개).
# 합성 v2 [101]: parquet 대응 추정을 **폐기**합니다 — AIDX0 가 전부 0 이라 100판에서
#   후보가 전부 실패했습니다. 채점 경로가 쓰는 배열(load_truth 의 MUA3·AOBS)에서
#   참 τ 를 직접 꺼냅니다. score() 와 같은 원천이므로 자체검사는 항등 수준이어야 합니다.
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
def _psplit99(s):
    sp = np.asarray(s.SPLIT); return (sp[:, 0] if sp.ndim > 1 else sp).reshape(-1)

_rowsA = []
for _rep in range(20):
    _s = dio.load_std('ihdp', _rep); _o = oc.for_dataset(_s, 'ihdp', _rep)
    _tt = (_s.MU1[:, 0, 0] - _s.MU0[:, 0, 0]).astype(float)
    _F = mdl.flatten_features(_s, _o)
    _g = HistGradientBoostingRegressor(max_depth=6, max_iter=300, random_state=0)
    _pd_ = cross_val_predict(_g, _F, _tt, cv=5)
    _r2 = 1 - float(np.sum((_tt - _pd_) ** 2)) / float(np.sum((_tt - _tt.mean()) ** 2))
    _rowsA.append({'데이터': 'ihdp', '판': _rep, 'R2_oracle': round(_r2, 4),
                   'sd(τ)': round(float(np.std(_tt)), 4), 'ATE': round(float(np.mean(_tt)), 4),
                   'n': int(len(_tt)), '비고': 'std.MU'})
print('IHDP 20판 완료')

try:
    _nm = 'empservicesyntv2'
    _s = dio.load_std(_nm, 0); _o = oc.for_dataset(_s, _nm, 0)
    _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
    _tr99 = dio.load_truth(_nm)
    _aT = np.asarray(_tr99['AOBS'], int).reshape(-1)
    _mu1 = np.asarray(_tr99['MUA3'])[np.arange(_s.N), _aT, _h]
    _mu0 = np.asarray(_tr99['MUA3'])[:, 0, _h]
    _tau_true = (_mu1 - _mu0).astype(float)
    _tre = np.asarray(_o['TREATED'], bool).reshape(-1)
    _te = (_psplit99(_s) == 2)

    # ── 자체검사: score(τ̂=0) 재현 — 같은 배열·같은 마스크이므로 사실상 0 이어야 합니다
    _z = mdl.score(_nm, np.zeros(_s.N), _s, _o, h=_h, y=CFG['TRAIN_Y'], split=2, truth=_tr99)
    _rms = float(_z['√PEHE'])
    _mm = _te & _tre & np.isfinite(_tau_true) & np.isfinite(_mu1) & np.isfinite(_mu0)
    _r_ = float(np.sqrt(np.mean(_tau_true[_mm] ** 2)))
    _d = abs(_r_ - _rms) / max(_rms, 1e-9)
    print('자체검사  √E[τ²] 내 계산 %.4f vs score %.4f (상대오차 %.2e) · %s'
          % (_r_, _rms, _d, _z.get('채점표본', '')))
    # [103] score() 는 소수 4자리 반올림 값을 돌려줍니다 → 상대오차 ~1.6e-6 까지는 반올림 잔차.
    assert _d < 1e-3, '참 τ 추출이 채점 경로와 다릅니다 — load_truth 배열·마스크를 보십시오'

    # ── 오라클 회귀: 논문 표 3 정의 — 훈련군 전체(참값 존재)
    _ok = np.isfinite(_tau_true) & _tre
    _F = mdl.flatten_features(_s, _o)
    _g = HistGradientBoostingRegressor(max_depth=6, max_iter=300, random_state=0)
    _pd_ = cross_val_predict(_g, _F[_ok], _tau_true[_ok], cv=5)
    _r2 = 1 - float(np.sum((_tau_true[_ok] - _pd_) ** 2)) / \
              float(np.sum((_tau_true[_ok] - _tau_true[_ok].mean()) ** 2))
    _rowsA.append({'데이터': _nm, '판': 0, 'R2_oracle': round(_r2, 4),
                   'sd(τ)': round(float(np.std(_tau_true[_ok])), 4),
                   'ATE': round(float(np.mean(_tau_true[_ok])), 4),
                   'n': int(_ok.sum()), '비고': '관측 특징(기관이 가진 것)'})

    # [103] 학습가능성의 세 층 — 「관측 특징 R² 이 낮은 것이 설계 탓인가」에 대한 답.
    #   ① 참 구동변수(성별·연령·교육·U·V·직종): 효과가 구조적으로 실재하는가의 상한
    #   ② 직종×성별 칸 평균: 규칙표 수준의 거친 구조
    #   ③ 관측 특징(위): 기관이 실제로 쓸 수 있는 것
    _drv = np.column_stack([_tr99['GENDER'], _tr99['AGE0'], _tr99['EDU'],
                            _tr99['U'], _tr99['V'], _aT]).astype(float)[_ok]
    _g2 = HistGradientBoostingRegressor(max_depth=6, max_iter=300, random_state=0)
    _pd2 = cross_val_predict(_g2, _drv, _tau_true[_ok], cv=5)
    _r2_drv = 1 - float(np.sum((_tau_true[_ok] - _pd2) ** 2)) /                   float(np.sum((_tau_true[_ok] - _tau_true[_ok].mean()) ** 2))
    _dfc = pd.DataFrame({'a': _aT[_ok], 'g': np.asarray(_tr99['GENDER'])[_ok],
                         'tau': _tau_true[_ok]})
    _cell = _dfc.groupby(['a', 'g'])['tau'].transform('mean')
    _r2_cell = 1 - float(((_dfc['tau'] - _cell) ** 2).sum()) /                    float(((_dfc['tau'] - _dfc['tau'].mean()) ** 2).sum())
    _rowsA.append({'데이터': _nm, '판': 0, 'R2_oracle': round(_r2_drv, 4),
                   'sd(τ)': round(float(np.std(_tau_true[_ok])), 4),
                   'ATE': round(float(np.mean(_tau_true[_ok])), 4),
                   'n': int(_ok.sum()), '비고': '참 구동변수(성별·연령·교육·U·V·직종) — 상한'})
    _rowsA.append({'데이터': _nm, '판': 0, 'R2_oracle': round(_r2_cell, 4),
                   'sd(τ)': round(float(_cell.std()), 4), 'ATE': round(float(np.mean(_tau_true[_ok])), 4),
                   'n': int(_ok.sum()), '비고': '직종×성별 칸 평균'})
    print()
    print('── [103] 학습가능성 세 층 (합성 v2 Y3, 훈련군 %d명)' % int(_ok.sum()))
    print('  참 구동변수(U·V 포함)  R² = %.3f   ← 이질성이 실재함의 증거' % _r2_drv)
    print('  직종×성별 칸 평균      R² = %.3f' % _r2_cell)
    print('  관측 특징(이력 요약)    R² = %.3f   ← 기관이 배울 수 있는 몫' % _r2)
    print('  → 논문 표 3 의 0.425 는 재현되지 않음. 위 세 값으로 표 3 을 다시 쓰십시오.')
    REPORT['학습가능성(표3)'] = ('ihdp %.3f · v2 관측특징 %.3f / 참구동 %.3f / 직종×성별 %.3f' % (
        np.mean([r['R2_oracle'] for r in _rowsA if r['데이터'] == 'ihdp']),
        _r2, _r2_drv, _r2_cell))
except Exception as _e:
    import traceback; traceback.print_exc()
    REPORT['학습가능성(표3)'] = '합성 v2 실패: %s — 표 3 을 논문에 싣지 마십시오' % str(_e)[:60]

A99 = pd.DataFrame(_rowsA)
A99.to_csv(os.path.join(DIR_OUT, '100A_학습가능성.csv'), index=False, encoding='utf-8-sig')
_ih = A99[A99['데이터'] == 'ihdp']
print()
print('── 표 3 대조용 요약')
print('IHDP        R2_oracle %.3f · CV(판별 비의 평균) %.3f · CV(평균의 비) %.3f'
      % (_ih['R2_oracle'].mean(), (_ih['sd(τ)'] / _ih['ATE'].abs()).mean(),
         _ih['sd(τ)'].mean() / _ih['ATE'].abs().mean()))
_sv = A99[A99['데이터'] == 'empservicesyntv2']
if len(_sv):
    print('합성 v2     R2_oracle %.3f · sd(τ) %.2f · CV %.3f — 논문 표 3(0.425)과 대조. 다르면 초록 42.5%%·35%% 교체'
          % (_sv['R2_oracle'].iloc[0], _sv['sd(τ)'].iloc[0],
             _sv['sd(τ)'].iloc[0] / max(abs(_sv['ATE'].iloc[0]), 1e-9)))
print('※ 논문에 CV 를 실을 때 어느 정의인지 명시하십시오 (본문 1.308 은 「평균의 비」).')
step('§100A 학습가능성')

In [ ]:
%%time
# ═══ [104S] 은닉교란 민감도 — 「실패하도록 설계했다」는 반론에 대한 답 ═══
# hidden ∈ {0.0, 0.35, 0.7} 세 판을 같은 시드로 생성해 같은 간소 파이프라인으로 진단합니다.
#   · hidden=0.0 → V 없음: 가정 1(이력의 대리 충분성)이 **정확히 성립**하는 판
#   · 읽는 것은 절대수치가 아니라 hidden 에 따른 **변화**입니다 (간소 파이프라인:
#     t*−1 상태 + 이전 취업률·평균임금, HGB T-러너, 70/30 분할 × 3시드)
# 약 8분 (판당 생성 ~2분 + 진단 ~30초). 생성은 datasynt/_sens/ 아래 — 본판을 안 건드립니다.
import gen_empservicesynt as _gen104
from sklearn.ensemble import HistGradientBoostingRegressor as _HGB104
from sklearn.model_selection import cross_val_predict as _cvp104

_SENS_DIR = os.path.join(DIR_SYNT, '_sens')
_HIDDENS = (0.0, 0.35, 0.7)

def _sens_analyze(_pq, _tq, _eq, _horizon=12, _seeds=(0, 1, 2)):
    _df = pd.read_parquet(_pq); _tr = pd.read_parquet(_tq); _per = pd.read_parquet(_eq)
    _N = len(_per); _T = int(_df['메모리'].max())
    _per = _per.reset_index(drop=True)
    _trt = (_per['훈련직종'] != 'c00').to_numpy()
    _ts = _per['tstar'].to_numpy().astype(int)
    _t12 = _tr[_tr['horizon'] == _horizon]
    _piv = _t12.pivot(index='persona', columns='action', values='MU_Y3').reindex(_per['persona'])
    _cols = list(_piv.columns); _M = _piv.to_numpy(float)
    _idx = np.array([_cols.index(a) if a in _cols else 0 for a in _per['훈련직종']])
    _tau = _M[np.arange(_N), _idx] - _piv['c00'].to_numpy(float)
    _wg = _df['보상_로그임금'].to_numpy().reshape(_N, _T); _em = (_wg > 0)
    _m12 = np.clip(_ts + _horizon, 0, _T - 1)
    _y3 = np.where(_em[np.arange(_N), _m12], np.exp(_wg[np.arange(_N), _m12]), 0.0)
    _naive = _y3[_trt].mean() - _y3[~_trt].mean()
    _att = float(np.nanmean(_tau[_trt]))
    _num = _df.select_dtypes(include=[np.number]).to_numpy().reshape(_N, _T, -1)
    _t0 = np.clip(_ts - 1, 0, _T - 1)
    _pe = np.array([_em[i, :max(_ts[i], 1)].mean() for i in range(_N)])
    _pw = np.array([_wg[i, :max(_ts[i], 1)].mean() for i in range(_N)])
    _F = np.column_stack([_num[np.arange(_N), _t0, :], _pe, _pw, _ts])
    _ok = np.isfinite(_tau) & _trt
    _g = _HGB104(max_depth=6, max_iter=300, random_state=0)
    _pr = _cvp104(_g, _F[_ok], _tau[_ok], cv=5)
    _r2 = 1 - float(np.sum((_tau[_ok] - _pr) ** 2)) / float(np.sum((_tau[_ok] - _tau[_ok].mean()) ** 2))
    _pes, _sds = [], []
    for _sd in _seeds:
        _rg = np.random.default_rng(_sd)
        _od = _rg.permutation(_N); _tn = np.zeros(_N, bool); _tn[_od[:int(_N * 0.7)]] = True
        _m1 = _HGB104(random_state=0).fit(_F[_tn & _trt], _y3[_tn & _trt])
        _m0 = _HGB104(random_state=0).fit(_F[_tn & ~_trt], _y3[_tn & ~_trt])
        _th = _m1.predict(_F) - _m0.predict(_F)
        _teok = (~_tn) & _ok
        _pes.append(float(np.sqrt(np.nanmean((_th[_teok] - _tau[_teok]) ** 2))))
        _sds.append(float(np.nanstd(_tau[_teok])))
    return dict(**{'sd(τ)': round(float(np.nanstd(_tau[_ok])), 3), '참ATT': round(_att, 3),
                   '단순차이': round(float(_naive), 3), '편향': round(float(_naive - _att), 3),
                   'R2_oracle': round(_r2, 4),
                   'T러너 √PEHE': round(float(np.mean(_pes)), 3),
                   '√PEHE_sd': round(float(np.std(_pes)), 3),
                   '오라클상수': round(float(np.mean(_sds)), 3),
                   '상수를 이김': bool(np.mean(_pes) < np.mean(_sds))})

_rowsS = []
for _hv in _HIDDENS:
    _tg = 'sens_h%03d' % int(round(_hv * 100))
    _od104 = os.path.join(_SENS_DIR, _tg)
    _f1 = os.path.join(_od104, 'empservicesynt_%s.parquet' % _tg)
    if not os.path.exists(_f1):
        print('  생성 hidden=%.2f ...' % _hv, flush=True)
        _gen104.build(8000, 20260821, _hv, 40, _od104, tag=_tg)
    else:
        print('  이미 있음(같은 시드·결정적): hidden=%.2f' % _hv)
    _r = _sens_analyze(_f1, os.path.join(_od104, 'empservicesynt_%s_truth.parquet' % _tg),
                       os.path.join(_od104, 'empservicesynt_%s_person.parquet' % _tg))
    _r = dict(hidden=_hv, **_r); _rowsS.append(_r)
    print('  hidden=%.2f →' % _hv, _r, flush=True)
S104 = pd.DataFrame(_rowsS)
S104.to_csv(os.path.join(DIR_OUT, '104S_민감도.csv'), index=False, encoding='utf-8-sig')
print(); print(S104.to_string(index=False))
print()
print('읽는 법: hidden=0.0 은 은닉교란 V 가 없어 가정 1 이 정확히 성립하는 판입니다.')
print('여기서도 「상수를 이김」이 False 면, 오라클 상수 실패는 hidden=0.35 라는')
print('설계 선택의 산물이 아니라 이 자료구조 자체의 성질입니다 — 순환성 반론에 대한 답.')
REPORT['민감도(hidden 스윕)'] = ' · '.join(
    'h=%.2f: 편향 %.0f · R² %.3f · 상수이김 %s' % (r['hidden'], r['편향'], r['R2_oracle'], r['상수를 이김'])
    for r in _rowsS)
step('§104S 민감도')

In [ ]:
%%time
# ═══ [105N] 국가 규모 검증 — §7d 의 검정력 예측을 스스로 시험한다 ═══
# n ∈ {16k, 60k, 120k, 전체} 중첩 부표본. 「n 명만 모았다면」을 같은 코호트 안에서 재현합니다.
# 모형: slearner·tlearner·cforest·tarnet ×3시드 (CDT 는 비용 문제로 n≤60k 에서 2시드).
# 긴 셀입니다 — (n, 모형) 단위로 방어하고 중간 저장하므로 어디서 끊겨도 결과가 남습니다.
import dataclasses as _dc105
from cdtlib.models import _fit_mask as _fm105, _yh as _yh105, qini as _qini105

_TAG105 = None
for _t105 in ('r240k', 'r160k'):
    _p105 = _cfg.RAW.get('empservicereal_%s' % _t105)
    if _p105 and os.path.exists(_p105):
        _TAG105 = _t105; break

if _TAG105 is None:
    print('[105N] 확대 코호트가 없습니다 — 건너뜁니다.')
    print('  윈도우에서 icml_datagen31.ipynb 를 돌려 datareal/empservicereal_r240k.parquet 를')
    print('  만들어 복사한 뒤 이 노트북을 다시 여십시오. 다른 셀 결과는 이 셀과 무관합니다.')
    REPORT['규모검증(105N)'] = '건너뜀 — r240k/r160k 없음'
else:
  try:
    print('[105N] 확대 코호트:', _TAG105)
    _cfg.use_cohort(_TAG105)
    # ── 표준화 (결정적 · 있으면 재사용: 이 산출물은 코호트 태그로 분리 저장됩니다)
    if not (os.path.exists(dio.std_path('empservicereal', 0))
            and dio.std_is_current('empservicereal', 0)):
        print('  표준화 중 — 사람 수에 비례해 오래 걸립니다...', flush=True)
        _sN = dio.build('empservicereal', realization=0)
        _e105 = dio.verify_std(_sN)
        if _e105: raise RuntimeError('스키마 위반: %s' % _e105)
        dio.save_std(_sN); del _sN; gc.collect()
    _sF = dio.load_std('empservicereal', 0)
    print('  코호트 %s명 · 상태열 %d' % (format(_sF.N, ','), _sF.Ds))
    # [고침 G] v107 은 oc.for_dataset() 를 불렀는데, 결과묶음 캐시 경로가 코호트
    #   꼬리표를 몰라 r16k 것(16,000명)을 조용히 돌려주었습니다. 240,000명 코호트와
    #   길이가 어긋나 105N 의 모든 적합이 shape mismatch 로 죽었습니다(107판 로그).
    #   코호트 꼬리표를 붙인 캐시를 따로 두고, 길이를 확인한 뒤에만 씁니다.
    #   (무꼬리표 파일은 r16k 본 파이프라인 것이므로 건드리지 않습니다)
    _pOF = os.path.join(DIR_STD, 'empservicereal',
                        'empservicereal_outcomes_%s.npz' % _TAG105)
    if os.path.exists(_pOF):
        _zOF = np.load(_pOF, allow_pickle=False)
        _oF = {k: _zOF[k] for k in _zOF.files}
    else:
        _oF = oc.build(_sF, 'empservicereal')
        np.savez_compressed(_pOF, **_oF)
    _len_oF = len(np.asarray(_oF['TREATED']).reshape(-1))
    if _len_oF != _sF.N:
        raise RuntimeError('결과묶음 길이 %d ≠ 코호트 %d — 캐시 %s 를 지우고 다시 도십시오'
                           % (_len_oF, _sF.N, _pOF))
    _hF = CFG['TRAIN_H']
    _NS = [n for n in (16000, 60000, 120000, _sF.N) if n <= _sF.N]
    _NS = sorted(set(_NS))
    _MODELS_N = ('slearner', 'tlearner', 'cforest', 'tarnet')
    _CDT_MAX = 60000; _SEEDS_N = (0, 1, 2)
    _sp_full = np.asarray(_sF.SPLIT)
    _sp_full = (_sp_full[:, 0] if _sp_full.ndim > 1 else _sp_full).reshape(-1).astype(np.int16)
    _rng105 = np.random.default_rng(20260821)
    _perm = _rng105.permutation(_sF.N)              # 중첩 부표본: 앞 n 명

    _rowsN, _rowsV = [], []
    for _n in _NS:
        _keep = np.zeros(_sF.N, bool); _keep[_perm[:_n]] = True
        _spn = _sp_full.copy(); _spn[~_keep] = -1
        _s2 = _dc105.replace(_sF, SPLIT=_spn)
        _TAUn = {}
        for _y in ('Y3', 'Y1'):
            _ytag = None if _y == CFG['TRAIN_Y'] else _y
            for _m in _MODELS_N + (('cdt',) if _n <= _CDT_MAX else ()):
                # [108] 큰 n 은 시드 2개 — 표본이 커질수록 시드 분산은 줄고 적합 시간은 늡니다.
                _sds = (0, 1) if (_m == 'cdt' or _n > 60000) else _SEEDS_N
                _ts105 = []
                for _sd in _sds:
                    try:
                        _t0105 = time.time()
                        _kw = {}
                        _pk = HP_PICK.get(('empservicereal', _m, _y)) if 'HP_PICK' in dir() else None
                        if _m == 'cdt':
                            _kw = CDT_KW(**(_pk or {}))
                        elif _pk:
                            _kw = dict(_pk)
                        _f = mdl.MODELS[_m](seed=_sd)
                        _f.fit(_s2, _oF, h=_hF, y=_y, split=0, **_kw)
                        _tau = np.asarray(_f.predict_tau(_s2, _oF, h=_hF), float).reshape(-1)
                        _ts105.append(_tau)
                        print('    n=%-7d %-9s %s s%d  %5.0f초' %
                              (_n, _m, _y, _sd, time.time() - _t0105), flush=True)
                    except Exception as _e:
                        print('    ⚠ n=%d %s %s s%d: %s' % (_n, _m, _y, _sd, str(_e)[:80]))
                if not _ts105:
                    continue
                _tauE = np.mean(_ts105, 0)
                _TAUn[(_y, _m)] = _tauE
                if _n == _sF.N:                      # [106] 최대 규모 τ̂ 는 파일로 — 106F 가 씁니다
                    np.save(os.path.join(DIR_OUT, '105N_tau_%s_%s.npy' % (_y, _m)), _tauE)
                _mm = _fm105(_s2, _oF, _hF, 2, _y)
                _idx = np.where(_mm)[0]
                _a = np.asarray(_oF['TREATED'], bool)[_idx]
                _yv = _yh105(_oF, _y, _hF)[_idx]
                _q = _qini105(_tauE[_idx], _a, _yv)
                _rowsN.append({'n': _n, '결과': _y, '모델': _m, '시드수': len(_ts105),
                               'Qini': round(float(_q), 2), '시험n': int(_mm.sum())})
            # ── 상위 2개(tarnet vs cforest) 짝지은 사람-붓스트랩
            if (_y, 'tarnet') in _TAUn and (_y, 'cforest') in _TAUn:
                _mm = _fm105(_s2, _oF, _hF, 2, _y)
                _idx = np.where(_mm)[0]
                _a = np.asarray(_oF['TREATED'], bool)[_idx]
                _yv = _yh105(_oF, _y, _hF)[_idx]
                _t1 = _TAUn[(_y, 'tarnet')][_idx]; _t2 = _TAUn[(_y, 'cforest')][_idx]
                _rb = np.random.default_rng(20260821)
                _ds = []
                for _ in range(500):
                    _j = _rb.integers(0, len(_idx), len(_idx))
                    _ds.append(_qini105(_t1[_j], _a[_j], _yv[_j])
                               - _qini105(_t2[_j], _a[_j], _yv[_j]))
                _ds = np.array(_ds); _lo, _hi = np.percentile(_ds, [2.5, 97.5])
                _v = ('구별됨 (tarnet 우위)' if _lo > 0 else
                      ('구별됨 (cforest 우위)' if _hi < 0 else '구별 못 함'))
                _rowsV.append({'n': _n, '결과': _y, '차(tarnet−cforest)': round(float(_ds.mean()), 2),
                               '하한': round(float(_lo), 2), '상한': round(float(_hi), 2), '판정': _v})
                print('  n=%-7d %s  tarnet−cforest %+.1f [%.1f, %.1f] → %s'
                      % (_n, _y, _ds.mean(), _lo, _hi, _v), flush=True)
        pd.DataFrame(_rowsN).to_csv(os.path.join(DIR_OUT, '105N_규모곡선.csv'),
                                    index=False, encoding='utf-8-sig')
        pd.DataFrame(_rowsV).to_csv(os.path.join(DIR_OUT, '105N_판정.csv'),
                                    index=False, encoding='utf-8-sig')
        gc.collect()
    if not _rowsN:
        raise RuntimeError('105N: 성공한 적합이 하나도 없습니다 — 위 ⚠ 줄을 보십시오')
    N105, V105 = pd.DataFrame(_rowsN), pd.DataFrame(_rowsV)
    print(); print(N105.pivot_table(index=['결과', '모델'], columns='n', values='Qini').round(1).to_string())
    print(); print(V105.to_string(index=False))
    _res = {}
    for _y in ('Y1', 'Y3'):
        _g = V105[V105['결과'] == _y]
        _first = _g[_g['판정'] != '구별 못 함']
        _res[_y] = ('n=%d 부터 구별' % _first['n'].min()) if len(_first) else '최대 규모에서도 동률'
    REPORT['규모검증(105N)'] = '%s · Y1 %s · Y3 %s (예측: Y1 53,319 · Y3 145,374)' % (
        _TAG105, _res['Y1'], _res['Y3'])
    print(); print('§7d 의 예측(전체 Y1 53,319명 · Y3 145,374명)과 위 판정을 대조하십시오.')
    print('예측이 맞으면: 검정력 분석 방법 자체가 검증됨 — 정규 트랙의 새 기여.')
    print('틀리면: 그것대로 「시뮬레이션·붓스트랩 기반 표본설계의 한계」라는 발견입니다.')
  except Exception:
    import traceback; traceback.print_exc()
    REPORT['규모검증(105N)'] = '실패 — 트레이스백 참조 (다른 결과는 유효)'
  finally:
    _cfg.use_cohort(CFG.get('REAL_COHORT'))          # 본 파이프라인 코호트로 복귀
step('§105N 규모 검증')

In [ ]:
# ═══════ [109] 공통 도구 — 규모 검증 보강 절들이 함께 씁니다 ═══════
from cdtlib.models import _fit_mask as _fm9, _yh as _yh9, qini as _qini9
import dataclasses as _dc9

def _load_scale9(tag):
    """확대 코호트 표준화본 + 코호트 꼬리표 결과묶음 ([고침 G] 와 같은 경로)."""
    _cfg.use_cohort(tag)
    s = dio.load_std('empservicereal', 0)
    p = os.path.join(DIR_STD, 'empservicereal', 'empservicereal_outcomes_%s.npz' % tag)
    if os.path.exists(p):
        z = np.load(p, allow_pickle=False); o = {k: z[k] for k in z.files}
    else:
        o = oc.build(s, 'empservicereal'); np.savez_compressed(p, **o)
    assert len(np.asarray(o['TREATED']).reshape(-1)) == s.N, '결과묶음 길이 ≠ 코호트'
    return s, o

def _subsplit9(s, perm, n):
    sp = np.asarray(s.SPLIT); sp = (sp[:, 0] if sp.ndim > 1 else sp).reshape(-1).astype(np.int16)
    keep = np.zeros(s.N, bool); keep[perm[:n]] = True
    spn = sp.copy(); spn[~keep] = -1
    return _dc9.replace(s, SPLIT=spn)

def _fit_tau9(s2, oF, m, sd, y):
    pk = HP_PICK.get(('empservicereal', m, y)) if 'HP_PICK' in dir() else None
    f = mdl.MODELS[m](seed=sd)
    f.fit(s2, oF, h=CFG['TRAIN_H'], y=y, split=0, **(dict(pk) if pk else {}))
    return np.asarray(f.predict_tau(s2, oF, h=CFG['TRAIN_H']), float).reshape(-1)

def _qini_of9(s2, oF, tau, y):
    mm = _fm9(s2, oF, CFG['TRAIN_H'], 2, y)
    idx = np.where(mm)[0]
    a = np.asarray(oF['TREATED'], bool)[idx]
    yv = _yh9(oF, y, CFG['TRAIN_H'])[idx]
    return float(_qini9(tau[idx], a, yv)), idx, a, yv

def _verdict9(t1, t2, idx, a, yv, B=500, seed=20260821):
    rb = np.random.default_rng(seed); ds = []
    for _ in range(B):
        j = rb.integers(0, len(idx), len(idx))
        ds.append(_qini9(t1[idx][j], a[j], yv[j]) - _qini9(t2[idx][j], a[j], yv[j]))
    ds = np.array(ds); lo, hi = np.percentile(ds, [2.5, 97.5])
    v = ('구별됨 (tarnet 우위)' if lo > 0 else ('구별됨 (cforest 우위)' if hi < 0 else '구별 못 함'))
    return float(ds.mean()), float(lo), float(hi), v


In [ ]:
%%time
# ═══ [109A] 규모 판정의 시드 보강 — 120k·240k 를 시드 3개로 ═══
# v108 [105N] 은 n>60k 에서 시드 2개였습니다. 「120k 의 무의미가 시드 부족 탓인가」에
# 답하기 위해 tarnet·cforest 만 시드 3개로 재적합해 판정을 다시 냅니다.
# 같은 순열(seed 20260821)·같은 초매개변수(v108 valid 선택)·같은 붓스트랩(500회) — 시드 수만 다릅니다.
try:
    _tag = 'r240k' if os.path.exists(_cfg.RAW.get('empservicereal_r240k', '')) else None
    assert _tag, 'datareal/empservicereal_r240k.parquet 가 없습니다'
    _sF, _oF = _load_scale9(_tag)
    print('코호트 %s명 · 상태열 %d' % (format(_sF.N, ','), _sF.Ds))
    _perm = np.random.default_rng(20260821).permutation(_sF.N)   # 105N 과 동일

    _rows = []
    for _n in (120000, 240000):
        _s2 = _subsplit9(_sF, _perm, _n)
        for _y in ('Y3', 'Y1'):
            _taus = {}
            for _m in ('cforest', 'tarnet'):
                _ts = []
                for _sd in (0, 1, 2):
                    _t0 = time.time()
                    try:
                        _tau = _fit_tau9(_s2, _oF, _m, _sd, _y)
                        _ts.append(_tau)
                        np.save(os.path.join(DIR_OUT, '109A_tau_n%d_%s_%s_s%d.npy' % (_n, _y, _m, _sd)), _tau)
                        print('  n=%-7d %-8s %s s%d  %5.0f초' % (_n, _m, _y, _sd, time.time() - _t0), flush=True)
                    except Exception as _e:
                        print('  ⚠ n=%d %s %s s%d: %s' % (_n, _m, _y, _sd, str(_e)[:80]))
                assert len(_ts) == 3, '109A: %s %s 시드 3개 확보 실패' % (_m, _y)
                _taus[_m] = _ts
            _, _idx, _a, _yv = _qini_of9(_s2, _oF, np.mean(_taus['tarnet'], 0), _y)
            for _lbl, _k in (('3시드', 3), ('2시드(105N 재현)', 2)):
                _t1 = np.mean(_taus['tarnet'][:_k], 0); _t2 = np.mean(_taus['cforest'][:_k], 0)
                _d, _lo, _hi, _v = _verdict9(_t1, _t2, _idx, _a, _yv)
                _rows.append({'n': _n, '결과': _y, '시드': _lbl,
                              'Qini(tarnet)': round(_qini_of9(_s2, _oF, _t1, _y)[0], 2),
                              'Qini(cforest)': round(_qini_of9(_s2, _oF, _t2, _y)[0], 2),
                              '차(tarnet−cforest)': round(_d, 2), '하한': round(_lo, 2),
                              '상한': round(_hi, 2), '판정': _v})
                print('  n=%-7d %s [%s]  차 %+.1f [%.1f, %.1f] → %s' % (_n, _y, _lbl, _d, _lo, _hi, _v), flush=True)
            pd.DataFrame(_rows).to_csv(os.path.join(DIR_OUT, '109A_판정.csv'),
                                       index=False, encoding='utf-8-sig')
        gc.collect()
    A109 = pd.DataFrame(_rows)
    print(); print(A109.to_string(index=False))
    _g3 = A109[A109['시드'] == '3시드']
    REPORT['109A 시드보강'] = ' · '.join('n=%dk %s: %s' % (r['n'] // 1000, r['결과'], r['판정'])
                                      for _, r in _g3.iterrows())
    print(); print('읽는 법: 3시드 판정이 2시드와 같으면 「시드 예산」 위협은 소멸 —')
    print('원고 Threats 의 seed-budget 항목을 「3시드로도 동일(run109)」로 바꿔 쓰십시오.')
except Exception:
    import traceback; traceback.print_exc(); REPORT['109A 시드보강'] = '실패 — 트레이스백 참조'
finally:
    _cfg.use_cohort(CFG.get('REAL_COHORT'))
step('§109A 시드 보강')


In [ ]:
%%time
# ═══ [109B] 부표본 추첨 안정성 — 다른 순열로 16k·60k 순위 재현 ═══
# 「16k 에서 숲이 Y1 1위인 것은 중첩 부표본 추첨 운 아닌가」(순위 뒤집힘 주장의 출발점)에
# 답합니다. 순열 시드만 20260822 로 바꾸고 나머지는 전부 동일. 시드 2개 평균.
try:
    _tag = 'r240k' if os.path.exists(_cfg.RAW.get('empservicereal_r240k', '')) else None
    assert _tag
    _sF, _oF = _load_scale9(_tag)
    _perm2 = np.random.default_rng(20260822).permutation(_sF.N)   # ← 여기만 다름

    _rowsB = []
    for _n in (16000, 60000):
        _s2 = _subsplit9(_sF, _perm2, _n)
        for _y in ('Y1', 'Y3'):
            for _m in ('tlearner', 'cforest', 'tarnet'):
                _ts = []
                for _sd in (0, 1):
                    try:
                        _ts.append(_fit_tau9(_s2, _oF, _m, _sd, _y))
                    except Exception as _e:
                        print('  ⚠ n=%d %s %s s%d: %s' % (_n, _m, _y, _sd, str(_e)[:80]))
                if not _ts:
                    continue
                _q, _, _, _ = _qini_of9(_s2, _oF, np.mean(_ts, 0), _y)
                _rowsB.append({'n': _n, '결과': _y, '모델': _m, '시드수': len(_ts),
                               'Qini(순열2)': round(_q, 2)})
                print('  n=%-6d %-8s %s  Qini %.2f' % (_n, _m, _y, _q), flush=True)
        pd.DataFrame(_rowsB).to_csv(os.path.join(DIR_OUT, '109B_순위재현.csv'),
                                    index=False, encoding='utf-8-sig')
        gc.collect()
    B109 = pd.DataFrame(_rowsB)
    print(); print(B109.pivot_table(index=['결과', '모델'], columns='n', values='Qini(순열2)').round(1).to_string())
    _y1 = B109[(B109['결과'] == 'Y1') & (B109['n'] == 16000)].set_index('모델')['Qini(순열2)']
    _top = _y1.idxmax() if len(_y1) else '(없음)'
    REPORT['109B 추첨안정성'] = '순열2 · 16k Y1 1위 = %s (105N 순열1: cforest)' % _top
    print(); print('읽는 법: 16k Y1 1위가 순열을 바꿔도 cforest 이면 「순위가 규모에 따라 뒤집힌다」의')
    print('출발점이 추첨 운이 아님이 확인된 것입니다.')
except Exception:
    import traceback; traceback.print_exc(); REPORT['109B 추첨안정성'] = '실패 — 트레이스백 참조'
finally:
    _cfg.use_cohort(CFG.get('REAL_COHORT'))
step('§109B 추첨 안정성')


In [ ]:
%%time
# ═══ [106T] 시간외(out-of-time) 검증 — 과거로 학습해 미래를 겨눈다 ═══
# 무작위 분할은 「같은 시기의 다른 사람」을 채점합니다. 기관의 실제 배치는 「미래」입니다.
# t* 를 시간순으로 갈라: 앞 85% 시기 → 학습·검증(80/20) · 뒤 15% 시기 → 시험.
# r16k 로 돌립니다(비용 통제). 무작위 분할 Qini(§7)와 나란히 보십시오.
import dataclasses as _dc106
try:
    _s, _o, _ = _load_pair('empservicereal', 0)
    _hT = CFG['TRAIN_H']
    _ts = np.asarray(_o['TSTAR'], int)
    _cut = int(np.quantile(_ts, 0.85))
    _pre = _ts < _cut; _post = ~_pre
    _rg = np.random.default_rng(20260821)
    _spT = np.full(_s.N, 2, np.int16)                 # 기본: 시험(미래)
    _idx_pre = np.where(_pre)[0]; _rg.shuffle(_idx_pre)
    _nv = int(len(_idx_pre) * 0.2)
    _spT[_idx_pre[_nv:]] = 0; _spT[_idx_pre[:_nv]] = 1
    _s2 = _dc106.replace(_s, SPLIT=_spT)
    print('시간 절단: t* < %d(월지수) 학습·검증 %d명 · 이후 시험 %d명 (2015+%d/12≈%d년 기준)'
          % (_cut, int(_pre.sum()), int(_post.sum()), _cut, 2015 + _cut // 12))
    from cdtlib.models import _fit_mask as _fmT, _yh as _yhT, qini as _qT
    _rowsT = []
    for _y in ('Y3', 'Y1'):
        for _m in ('slearner', 'tlearner', 'cforest', 'tarnet'):
            _tsE = []
            for _sd in (0, 1, 2):
                try:
                    _pk = HP_PICK.get(('empservicereal', _m, _y)) if 'HP_PICK' in dir() else None
                    _f = mdl.MODELS[_m](seed=_sd)
                    _f.fit(_s2, _o, h=_hT, y=_y, split=0, **(_pk or {}))
                    _tsE.append(np.asarray(_f.predict_tau(_s2, _o, h=_hT), float).reshape(-1))
                except Exception as _e:
                    print('  ⚠ %s %s s%d: %s' % (_m, _y, _sd, str(_e)[:70]))
            if not _tsE:
                continue
            _tauE = np.mean(_tsE, 0)
            _mm = _fmT(_s2, _o, _hT, 2, _y)
            _i2 = np.where(_mm)[0]
            _q = _qT(_tauE[_i2], np.asarray(_o['TREATED'], bool)[_i2], _yhT(_o, _y, _hT)[_i2])
            _rowsT.append({'결과': _y, '모델': _m, 'Qini(시간외)': round(float(_q), 2),
                           '시험n(미래)': int(_mm.sum())})
            print('  %-9s %s  시간외 Qini %10.2f' % (_m, _y, _q), flush=True)
    T106 = pd.DataFrame(_rowsT)
    T106.to_csv(os.path.join(DIR_OUT, '106T_시간외검증.csv'), index=False, encoding='utf-8-sig')
    print(); print(T106.to_string(index=False))
    print(); print('§7(무작위 분할)의 순위와 대조하십시오 — 순위가 유지되면 「미래 배치에도 선다」는 증거,')
    print('붕괴하면 그 자체가 발견입니다(시간 이동 아래의 uplift 취약성).')
    _b = {y: g.loc[g['Qini(시간외)'].idxmax(), '모델'] for y, g in T106.groupby('결과')}
    REPORT['시간외 검증(106T)'] = ' · '.join('%s 1위 %s' % (y, m) for y, m in _b.items())
except Exception:
    import traceback; traceback.print_exc()
    REPORT['시간외 검증(106T)'] = '실패 — 트레이스백 참조'
step('§106T 시간외 검증')

In [ ]:
%%time
# ═══ [106F] 타게팅 구성 감사 — τ̂ 상위 30% 는 누구인가 (공정성) ═══
# 개인화가 여성·고연령을 체계적으로 배제하면, Qini 가 높아도 기관은 쓸 수 없습니다.
# 최대 규모(105N)의 τ̂ 가 있으면 그것을, 없으면 §7 의 r16k 앙상블 τ̂ 를 씁니다.
try:
    _useN = os.path.exists(os.path.join(DIR_OUT, '105N_tau_Y3_tarnet.npy'))
    if _useN:
        _tagF = 'r240k' if os.path.exists(_cfg.RAW.get('empservicereal_r240k', '')) else 'r160k'
        _cfg.use_cohort(_tagF)
        _s = dio.load_std('empservicereal', 0)
        # [고침 G] 코호트 꼬리표 결과묶음 — 105N 이 만든 캐시를 씁니다 (길이 강제 확인)
        _pFG = os.path.join(DIR_STD, 'empservicereal',
                            'empservicereal_outcomes_%s.npz' % _tagF)
        if os.path.exists(_pFG):
            _zFG = np.load(_pFG, allow_pickle=False)
            _o = {k: _zFG[k] for k in _zFG.files}
        else:
            _o = oc.build(_s, 'empservicereal')
        if len(np.asarray(_o['TREATED']).reshape(-1)) != _s.N:
            raise RuntimeError('106F: 결과묶음 길이가 코호트와 다릅니다 — %s 를 지우고 다시' % _pFG)
    else:
        _s, _o, _ = _load_pair('empservicereal', 0)
    _cols = list((_s.meta or {}).get('state_cols') or [])
    _jf = next((j for j, c in enumerate(_cols) if '성별_여' in c), None)
    _ja = next((j for j, c in enumerate(_cols) if '나이' in c), None)
    _ts = np.asarray(_o['TSTAR'], int)
    _t0 = np.clip(_ts - 1, 0, _s.T - 1)
    _rows_ = np.arange(_s.N)
    _fem = _s.X[_rows_, _t0, _jf] > 0.5 if _jf is not None else None
    _age = _s.X[_rows_, _t0, _ja] * 100 if _ja is not None else None
    _rowsF = []
    for _y in ('Y3', 'Y1'):
        _ytag = None if _y == CFG['TRAIN_Y'] else _y
        for _m in ('tarnet', 'cforest', 'tlearner', 'cdt'):
            _tau = None
            _fpN = os.path.join(DIR_OUT, '105N_tau_%s_%s.npy' % (_y, _m))
            if _useN and os.path.exists(_fpN):
                _tau = np.load(_fpN); _src = '105N(최대규모)'
            else:
                _tsl = []
                for _sd in CFG['TRAIN_SEEDS']:
                    _fp = os.path.join(R.DIR_RUNS, R.run_id('empservicereal', _m, _sd, None, _ytag, VERSION), 'tau.npy')
                    if os.path.exists(_fp): _tsl.append(np.load(_fp))
                if _tsl: _tau = np.mean(_tsl, 0).reshape(-1); _src = 'r16k(§7)'
            if _tau is None or len(_tau) != _s.N:
                continue
            _thr = np.quantile(_tau, 0.70)
            _top = _tau >= _thr
            _r = {'결과': _y, '모델': _m, 'τ̂출처': _src, '상위30% n': int(_top.sum())}
            if _fem is not None:
                _r['여성비율(전체)'] = round(float(_fem.mean()), 3)
                _r['여성비율(상위30%)'] = round(float(_fem[_top].mean()), 3)
                _r['여성 대표성비'] = round(float(_fem[_top].mean() / max(_fem.mean(), 1e-9)), 3)
            if _age is not None:
                _r['평균나이(전체)'] = round(float(_age.mean()), 1)
                _r['평균나이(상위30%)'] = round(float(_age[_top].mean()), 1)
            _rowsF.append(_r)
    F106 = pd.DataFrame(_rowsF)
    F106.to_csv(os.path.join(DIR_OUT, '106F_타게팅구성.csv'), index=False, encoding='utf-8-sig')
    print(F106.to_string(index=False))
    print(); print('여성 대표성비 < 1 이면 개인화 타게팅이 여성을 과소 선발한다는 뜻입니다 — 값 자체가 발견.')
    if len(F106) and '여성 대표성비' in F106:
        REPORT['타게팅 구성(106F)'] = 'Y3 여성 대표성비 ' + ' · '.join(
            '%s %.2f' % (r['모델'], r['여성 대표성비'])
            for _, r in F106[F106['결과'] == 'Y3'].iterrows() if '여성 대표성비' in r)
except Exception:
    import traceback; traceback.print_exc()
    REPORT['타게팅 구성(106F)'] = '실패 — 트레이스백 참조'
finally:
    _cfg.use_cohort(CFG.get('REAL_COHORT'))
step('§106F 타게팅 구성')

In [ ]:
%%time
# ═══ [106R] 오라클 상수의 몬테카를로 잡음 보정 — R=40 vs R=400 ═══
# 참값 MU 는 R회 몬테카를로 평균이라 sd(τ)=27.46 에는 시뮬레이션 잡음이 섞여 있습니다.
# 같은 시드·hidden=0.35 로 R=400 판을 만들어 분산을 분해합니다:
#   Var_R = Var_true + c/R  →  두 R 에서 연립으로 Var_true 와 잡음 몫을 풉니다.
# 결과는 표 2 를 바꾸지 않습니다(채점은 R=40 정의 그대로) — 위협요인 절의 정직성 소재입니다.
try:
    import gen_empservicesynt as _gen106
    _od = os.path.join(DIR_SYNT, '_sens', 'r400')
    _f1 = os.path.join(_od, 'empservicesynt_r400_truth.parquet')
    if not os.path.exists(_f1):
        print('  생성 중 R=400 (약 15~20분)...', flush=True)
        _gen106.build(8000, 20260821, 0.35, 400, _od, tag='r400')
    _per4 = pd.read_parquet(os.path.join(_od, 'empservicesynt_r400_person.parquet'))
    _tr4 = pd.read_parquet(_f1); _tr4 = _tr4[_tr4['horizon'] == 12]
    _piv4 = _tr4.pivot(index='persona', columns='action', values='MU_Y3').reindex(_per4['persona'])
    _c4 = list(_piv4.columns); _M4 = _piv4.to_numpy(float)
    _i4 = np.array([_c4.index(a) if a in _c4 else 0 for a in _per4['훈련직종']])
    _trt4 = (_per4['훈련직종'] != 'c00').to_numpy()
    _tau400 = (_M4[np.arange(len(_per4)), _i4] - _piv4['c00'].to_numpy(float))[_trt4]
    _tr40 = dio.load_truth('empservicesyntv2')
    _a40 = np.asarray(_tr40['AOBS'], int)
    _tau40 = (np.asarray(_tr40['MUA3'])[np.arange(len(_a40)), _a40, 1]
              - np.asarray(_tr40['MUA3'])[:, 0, 1])
    _trt40 = (_a40 > 0)
    _tau40 = _tau40[_trt40 & np.isfinite(_tau40)]
    _V40, _V400 = float(np.var(_tau40)), float(np.var(_tau400[np.isfinite(_tau400)]))
    _c_noise = max(0.0, (_V40 - _V400) * 400.0 * 40.0 / (400.0 - 40.0))   # c = ΔV·R1R2/(R2−R1)
    _Vt = max(0.0, _V400 - _c_noise / 400.0)
    R106 = pd.DataFrame([{'sd(τ)@R=40': round(np.sqrt(_V40), 3),
                          'sd(τ)@R=400': round(np.sqrt(_V400), 3),
                          'sd(τ)_참(잡음 보정)': round(np.sqrt(_Vt), 3),
                          'MC잡음 분산비율@R=40': round((_c_noise / 40.0) / max(_V40, 1e-12), 3)}])
    R106.to_csv(os.path.join(DIR_OUT, '106R_잡음보정.csv'), index=False, encoding='utf-8-sig')
    print(R106.to_string(index=False))
    print(); print('해석: 오라클 상수(=sd(τ)@R=40)의 일부는 시뮬레이션 잡음입니다. 보정 sd(τ)_참이')
    print('개인화가 원리상 노릴 수 있는 이질성의 상한이며, 위협요인 절에 이 표를 인용하십시오.')
    REPORT['MC잡음 보정(106R)'] = 'sd 27.46→참 %.1f · 잡음몫 %.0f%%' % (
        np.sqrt(_Vt), 100 * (_c_noise / 40.0) / max(_V40, 1e-12))
except Exception:
    import traceback; traceback.print_exc()
    REPORT['MC잡음 보정(106R)'] = '실패 — 트레이스백 참조'
step('§106R MC 잡음 보정')

In [ ]:
%%time
# ═══ [100D] IHDP 재적합 — 고정 에폭 (IX-D 의 근거. 95·97판의 깨진 재적합을 대체) ═══
#  · models.py [고침 E] 전제 — 빈 검증에서 1에폭 가중치가 복원되던 사고 차단
#  · be 는 0-기준 색인 → be+1 에폭 · `be=0` falsy 버그 수정 · backend 기록
import copy as _copy
_REPS = list(range(100)); _SEEDS = (0, 1, 2)
_GRID_T = dict(lr=(1e-4, 3e-4, 1e-3, 3e-3), hidden=200)
_GRID_C = dict(lr=(3e-4, 1e-3, 3e-3, 1e-2), dropout=(0.0, 0.1, 0.3), epochs=200, K=1, hidden=128)
_HP99 = {}
_rowsD = []
for _rep in _REPS:
    _s = dio.load_std('ihdp', _rep); _o = oc.for_dataset(_s, 'ihdp', _rep)
    _h = 0 if _s.T <= 1 else CFG['TRAIN_H']
    _sp = np.asarray(_s.SPLIT)
    _sp1 = (_sp[:, 0] if _sp.ndim > 1 else _sp).reshape(-1)
    _s2 = _copy.copy(_s); _sp2 = _sp1.copy(); _sp2[_sp2 == 1] = 0
    _s2.SPLIT = _sp2.astype(_sp.dtype)
    for _m in CFG['TRAIN_MODELS']:
        for _sd in _SEEDS:
            try:
                _kw = {}
                if _m == 'tarnet': _kw = dict(_GRID_T)
                if _m == 'cdt':    _kw = dict(_GRID_C)
                if _m in _HP99:
                    _kw = {k: v for k, v in _kw.items() if k not in ('lr', 'dropout')}
                    _kw.update(_HP99[_m])
                _f1 = mdl.MODELS[_m](seed=_sd); _f1.fit(_s, _o, h=_h, y=CFG['TRAIN_Y'], split=0, **_kw)
                _pick = getattr(_f1, 'hp_chosen', None) or {}
                if _pick and _m not in _HP99:
                    _HP99[_m] = dict(_pick); print('  [%s] 초매개변수 고정 %s' % (_m, _pick), flush=True)
                _be = getattr(_f1, 'best_epoch', None)
                _t1 = np.asarray(_f1.predict_tau(_s, _o, h=_h), float).reshape(-1)
                _s1c = mdl.score('ihdp', _t1, _s, _o, h=_h, y=CFG['TRAIN_Y'], split=2)
                _rowsD.append({'판': _rep, '모델': _m, '시드': _sd, '절차': '기존(538)',
                               '√PEHE': float(_s1c['√PEHE']), 'ε_ATE': float(_s1c['ε_ATE']),
                               'best_epoch': _be, 'backend': getattr(_f1, 'backend', '')})
                _kw2 = dict(_kw); _kw2.update(_pick)
                if _m in ('tarnet', 'cdt') and _be is not None:
                    _kw2['epochs'] = int(_be) + 1; _kw2['patience'] = 10 ** 6
                _f2 = mdl.MODELS[_m](seed=_sd); _f2.fit(_s2, _o, h=_h, y=CFG['TRAIN_Y'], split=0, **_kw2)
                if _m in ('tarnet', 'cdt'):
                    assert getattr(_f2, 'used_final_weights', False), \
                        '재적합인데 최고복원이 작동 — 검증 병합 실패거나 models.py 옛 판'
                _t2 = np.asarray(_f2.predict_tau(_s2, _o, h=_h), float).reshape(-1)
                _s2c = mdl.score('ihdp', _t2, _s, _o, h=_h, y=CFG['TRAIN_Y'], split=2)
                _rowsD.append({'판': _rep, '모델': _m, '시드': _sd, '절차': '재적합(672,고정에폭)',
                               '√PEHE': float(_s2c['√PEHE']), 'ε_ATE': float(_s2c['ε_ATE']),
                               'best_epoch': _be, 'backend': getattr(_f2, 'backend', '')})
            except Exception as _e:
                print('  ⚠ %s r%d s%d: %s' % (_m, _rep, _sd, str(_e)[:80]))
    if _rep % 20 == 0:
        pd.DataFrame(_rowsD).to_csv(os.path.join(DIR_OUT, '100D_ihdp재적합.csv'),
                                    index=False, encoding='utf-8-sig')
        print('  판 %d 까지' % _rep, flush=True)
D99 = pd.DataFrame(_rowsD)
D99.to_csv(os.path.join(DIR_OUT, '100D_ihdp재적합.csv'), index=False, encoding='utf-8-sig')
_p99 = D99.pivot_table(index='모델', columns='절차', values='√PEHE', aggfunc='mean').round(4)
_p99['개선%'] = ((_p99['기존(538)'] - _p99['재적합(672,고정에폭)']) / _p99['기존(538)'] * 100).round(1)
print(); print(_p99.to_string())
REPORT['IHDP 재적합(IX-D)'] = ' · '.join('%s %+.1f%%' % (i, v) for i, v in _p99['개선%'].items())
print(); print('논문 IX-D 를 이 표로 다시 쓰십시오 — 95·97판의 -196%/-347% 는 구현 사고였습니다.')
step('§100D IHDP 재적합')

In [ ]:
%%time
# ═══ [102C] 실데이터 정책가치 — 반사실 없이 원화(만원/월)로 (run95 C 를 흡수) ═══
# 「τ̂ 상위 k% 만 훈련」 규칙의 가치를 이중강건(DR)으로 추정해 상수 규칙(전원/아무도)과
# 비교합니다. 성향·결과모형은 학습분할에서만 적합, 채점은 시험분할. τ̂ 는 시드 앙상블.
# Qini 는 순위 품질만 말하지만 이 표는 「얼마 버는가」를 만원/월로 말합니다 — I&G 트랙용.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor as _HGB102

_nm = 'empservicereal'
_s, _o, _ = _load_pair(_nm, 0)
_h = 0 if _s.T <= 1 else CFG['TRAIN_H']
_F = mdl.flatten_features(_s, _o)
_sp102 = np.asarray(_s.SPLIT); _sp102 = (_sp102[:, 0] if _sp102.ndim > 1 else _sp102).reshape(-1)
_W = np.asarray(_o['TREATED']).reshape(-1).astype(float)
_rowsC = []
for _yn in ('Y3', 'Y1'):
    _Y = np.asarray(_o[_yn])
    _Y = _Y[:, min(_h, _Y.shape[1] - 1)].astype(float) if _Y.ndim > 1 else _Y.astype(float)
    _okY = np.isfinite(_Y)
    _tr = (_sp102 == 0) & _okY; _te = (_sp102 == 2) & _okY
    _ps = LogisticRegression(max_iter=2000).fit(_F[_tr], _W[_tr])
    _e = np.clip(_ps.predict_proba(_F)[:, 1], 0.02, 0.98)
    _m1 = _HGB102(random_state=0).fit(_F[_tr & (_W == 1)], _Y[_tr & (_W == 1)])
    _m0 = _HGB102(random_state=0).fit(_F[_tr & (_W == 0)], _Y[_tr & (_W == 0)])
    _g1, _g0 = _m1.predict(_F), _m0.predict(_F)
    def _drv_vec(_pi, _idx):
        _a = _pi[_idx]; _w = _W[_idx]; _y = _Y[_idx]; _ee = _e[_idx]
        return _a * (_g1[_idx] + _w * (_y - _g1[_idx]) / _ee) \
               + (1 - _a) * (_g0[_idx] + (1 - _w) * (_y - _g0[_idx]) / (1 - _ee))
    def _drv(_pi, _idx):
        _v = _drv_vec(_pi, _idx)
        return float(np.mean(_v)), float(np.std(_v) / np.sqrt(max(len(_v), 1)))
    _b1, _ = _drv(np.ones(_s.N), _te); _b0, _ = _drv(np.zeros(_s.N), _te)
    _bc = max(_b1, _b0)
    _ytag = None if _yn == CFG['TRAIN_Y'] else _yn
    for _mo in CFG['TRAIN_MODELS']:
        _ts_ = []
        for _sd in CFG['TRAIN_SEEDS']:
            _fp = os.path.join(R.DIR_RUNS, R.run_id(_nm, _mo, _sd, None, _ytag, VERSION), 'tau.npy')
            if os.path.exists(_fp):
                _ts_.append(np.load(_fp))
        if not _ts_:
            continue
        _tau = np.mean(_ts_, 0).reshape(-1)
        for _k in (10, 20, 30, 50):
            _thr = np.quantile(_tau[_te], 1 - _k / 100.0)
            _pi = (_tau >= _thr).astype(float)
            _v, _se = _drv(_pi, _te)
            _row = {'결과': _yn, '모델': _mo, '상위%': _k, '정책가치': round(_v, 4),
                    '표준오차': round(_se, 4), '전원': round(_b1, 4),
                    '아무도': round(_b0, 4), '개인화 이득': round(_v - _bc, 4)}
            # [104] 상위 30% (본문 표) 는 사람 재표집 붓스트랩 CI 와 판정을 붙입니다
            if _k == 30:
                _idxte = np.where(_te)[0]
                _pi_c = np.ones(_s.N) if _b1 >= _b0 else np.zeros(_s.N)
                _dvec = _drv_vec(_pi, _idxte) - _drv_vec(_pi_c, _idxte)
                _rngb = np.random.default_rng(20260821)
                _bs = np.array([_dvec[_rngb.integers(0, len(_dvec), len(_dvec))].mean()
                                for _ in range(1000)])
                _row['이득_하한'] = round(float(np.percentile(_bs, 2.5)), 4)
                _row['이득_상한'] = round(float(np.percentile(_bs, 97.5)), 4)
                _row['판정'] = ('개인화가 유의하게 낫다' if _row['이득_하한'] > 0 else
                              ('상수가 유의하게 낫다' if _row['이득_상한'] < 0 else '구별 못 함'))
            _rowsC.append(_row)
    print('[%s] 전원 %.3f · 아무도 %.3f · 최선 상수 %.3f' % (_yn, _b1, _b0, _bc))
C102 = pd.DataFrame(_rowsC)
C102.to_csv(os.path.join(DIR_OUT, '102_정책가치.csv'), index=False, encoding='utf-8-sig')
print()
print(C102.pivot_table(index=['결과', '모델'], columns='상위%', values='개인화 이득').round(3).to_string())
print()
print('Y3 단위는 만원/월 · 양수 = 개인화가 최선의 상수 규칙보다 낫다는 뜻입니다.')
_c3 = C102[(C102['결과'] == 'Y3') & (C102['상위%'] == 30)]
if len(_c3):
    _bst = _c3.loc[_c3['개인화 이득'].idxmax()]
    REPORT['정책가치(원화)'] = 'Y3 상위30%% 최고 이득 %.2f만원/월 (%s · CI [%.2f, %.2f] · %s)' % (
        _bst['개인화 이득'], _bst['모델'], _bst.get('이득_하한', float('nan')),
        _bst.get('이득_상한', float('nan')), _bst.get('판정', ''))
del _s, _o; gc.collect()
step('§102C 정책가치')

In [ ]:
%%time
# ═══ [100T] 논문 표 자동 내보내기 — paper/tab (규칙 7: 손으로 옮겨 적지 않는다) ═══
import shutil
TAB = os.path.join(CONF_ROOT, 'paper', 'tab'); os.makedirs(TAB, exist_ok=True)

# [102] 폐기된 셀·깨진 판의 옛 산출물 정리 — 새 표와 뒤섞이지 않게 (모두 재생성 가능한 파일)
for _stale in ('92A_상수대비.csv', '92A_최선의상수.csv', '95A_학습가능성.csv',
               '95C_정책가치.csv', '95D_ihdp재적합.csv', '97D_ihdp재적합.csv',
               '98A_후보대응.csv', '98A_학습가능성_syntv2.csv', '98D_ihdp재적합.csv',
               '100A_후보대응.csv'):
    _p102 = os.path.join(DIR_OUT, _stale)
    if os.path.exists(_p102):
        os.remove(_p102); print('  옛 산출물 삭제:', _stale)
def _psplit99(s):
    sp = np.asarray(s.SPLIT); return (sp[:, 0] if sp.ndim > 1 else sp).reshape(-1)

# ── 표 4 (분해 · ρ) — 시드별로 계산해 평균. 채점과 같은 마스크(시험∧훈련군∧유한).
_nm = 'empservicesyntv2'
_s, _o, _t = _load_pair(_nm, 0)
_h = 0 if _s.T <= 1 else CFG['TRAIN_H']
_aT = _t['AOBS']
_mu1 = _t['MUA3'][np.arange(_s.N), _aT, _h]; _mu0 = _t['MUA3'][:, 0, _h]
_tauT = (_mu1 - _mu0).astype(float)
_mm = (_psplit99(_s) == 2) & np.asarray(_o['TREATED'], bool) & np.isfinite(_tauT)
_tt = _tauT[_mm]; _sdT = float(_tt.std())
_rows4 = []
for _m in CFG['TRAIN_MODELS']:
    for _sd in CFG['TRAIN_SEEDS']:
        _fp = os.path.join(R.DIR_RUNS, R.run_id(_nm, _m, _sd, None, None, VERSION), 'tau.npy')
        if not os.path.exists(_fp): continue
        _th = np.load(_fp).reshape(-1)[_mm]
        _rho = float(np.corrcoef(_th, _tt)[0, 1]) if _th.std() > 1e-12 else 0.0
        _rows4.append({'모델': _m, '시드': _sd,
                       '√PEHE': float(np.sqrt(np.mean((_th - _tt) ** 2))),
                       'level': abs(float(_th.mean() - _tt.mean())),
                       'shape': float(np.sqrt(np.var(_th - _tt))), 'ρ': _rho})
T4 = (pd.DataFrame(_rows4).groupby('모델', sort=False)
      .agg(**{'√PEHE': ('√PEHE', 'mean'), 'level': ('level', 'mean'),
              'shape': ('shape', 'mean'), 'ρ': ('ρ', 'mean')}).round(3).reset_index())
T4['shape/sd(τ)'] = (T4['shape'] / _sdT).round(3)
T4['재척도상한%'] = (100 * (1 - np.sqrt(np.maximum(0.0, 1 - T4['ρ'] ** 2)))).round(1)
T4.to_csv(os.path.join(DIR_OUT, '100_표4_분해.csv'), index=False, encoding='utf-8-sig')   # [102] out 먼저
try:
    T4.to_csv(os.path.join(TAB, 'tab4_decomp.csv'), index=False, encoding='utf-8-sig')
except Exception as _e:
    print('  ⚠ tab4 복사 실패(요약은 계속):', _e)
print('── 표 4 (sd(τ)=%.3f) — 논문 Table IV 를 이 값으로 교체' % _sdT)
print(T4.to_string(index=False))
_c4 = T4.set_index('모델')
if 'cforest' in _c4.index and 'tlearner' in _c4.index:
    if abs(_c4.loc['cforest', '√PEHE'] - _c4.loc['tlearner', '√PEHE']) < 1e-9:
        print('  ⚠ cforest ≡ tlearner — econml 폴백이 의심됩니다. 시작 게이트를 확인하십시오.')

# ── Jobs 소표 — 논문에 서술만 있고 표가 없던 구멍(S4)
if SCORES is not None and len(SCORES):
    _j = SCORES[SCORES['데이터'] == 'jobs']
    if len(_j):
        _jt = (_j.groupby('모델', sort=False)
               .agg(**{'ε_ATT': ('ε_ATT', 'mean'), 'ε_ATT_sd': ('ε_ATT', 'std'),
                       '정책위험': ('정책위험', 'mean'), '정책위험_sd': ('정책위험', 'std'),
                       '실행수': ('시드', 'count')}).round(4).reset_index())
        _jt.to_csv(os.path.join(TAB, 'tabS_jobs.csv'), index=False, encoding='utf-8-sig')
        _jt.to_csv(os.path.join(DIR_OUT, '100_Jobs표.csv'), index=False, encoding='utf-8-sig')
        print(); print('── Jobs 소표 — 논문 IX 앞에 추가하십시오 (은폐 오해 차단)')
        print(_jt.to_string(index=False))

# ── 나머지 표의 정본을 paper/tab 으로 복사
for _src9, _dst9 in [('93A_오라클상수_판별.csv', 'tab2_oracle_raw.csv'),
                     ('100A_학습가능성.csv', 'tab3_learnable.csv'),
                     ('07_모델비교_요약.csv', 'tab_all_summary.csv'),
                     ('07_앙상블.csv', 'tab7_ensemble.csv'),
                     ('07_붓스트랩.csv', 'tab7_bootstrap.csv'),
                     ('07_벽시계.csv', 'tab8_cost.csv'),
                     ('08_구조소거.csv', 'tab6_ablation.csv'),
                     ('102_정책가치.csv', 'tabS_policy_value.csv'),
                     ('104S_민감도.csv', 'tabS_sensitivity.csv'),
                     ('105N_규모곡선.csv', 'tabN_scale.csv'),
                     ('105N_판정.csv', 'tabN_scale_verdict.csv'),
                     ('109A_판정.csv', 'tabN_scale_verdict_3seed.csv'),
                     ('109B_순위재현.csv', 'tabN_draw_stability.csv'),
                     ('106X_선택이전.csv', 'tabS_selection_transfer.csv'),
                     ('106T_시간외검증.csv', 'tabS_temporal.csv'),
                     ('106F_타게팅구성.csv', 'tabS_fairness.csv'),
                     ('106R_잡음보정.csv', 'tabS_mc_noise.csv'),
                     ('107P_이론검증.csv', 'tabS_theory_check.csv'),
                     ('100D_ihdp재적합.csv', 'tab_refit.csv')]:
    _p9 = os.path.join(DIR_OUT, _src9)
    if not os.path.exists(_p9):
        continue
    try:                                    # [102] 복사 실패가 요약을 막지 않게
        shutil.copy(_p9, os.path.join(TAB, _dst9))
    except Exception as _e:
        print('  ⚠ 복사 실패 %s → %s: %s' % (_src9, _dst9, _e))
print(); print('paper/tab 저장:', sorted(os.listdir(TAB)))
REPORT['paper/tab 내보내기'] = '%d개 표' % len(os.listdir(TAB))
del _s, _o, _t; gc.collect()
step('§100T 표 내보내기')

In [ ]:
%%time
# ═══ [106X] 시뮬레이터 선택 → 실데이터 이전 — 진단 절차 4단계의 실증 (추가 학습 없음) ═══
# 질문: 기관이 보정 시뮬레이터에서 모형을 고른다면, 어떤 지표로 골라야 실데이터 순위를 맞히나?
#   √PEHE 로 고르면 → CDT (실데이터 꼴찌) · ρ(형태)로 고르면 → 숲/T-러너 (실데이터 상위)
try:
    _t4 = pd.read_csv(os.path.join(DIR_OUT, '100_표4_분해.csv')).set_index('모델')
    _sm = pd.read_csv(os.path.join(DIR_OUT, '07_모델비교_요약.csv'))
    _re = _sm[(_sm['데이터'] == 'empservicereal') & (_sm['Qini'].notna())].copy()
    _re['_q'] = _re['Qini'].astype(str).str.split('±').str[0].astype(float)
    _rowsX = []
    for _y, _g in _re.groupby('결과'):
        _g = _g.set_index('모델')
        _mods = [m for m in _t4.index if m in _g.index and m not in ('naive',)]
        import scipy.stats as _st106
        _q = _g.loc[_mods, '_q']
        _byP = -_t4.loc[_mods, '√PEHE']          # √PEHE 낮을수록 좋음 → 부호 반전
        _byR = _t4.loc[_mods, 'ρ']
        _rp = _st106.spearmanr(_byP, _q).statistic
        _rr = _st106.spearmanr(_byR, _q).statistic
        _rowsX.append({'결과': _y, 'n(모형)': len(_mods),
                       '선택지표=√PEHE 의 순위상관': round(float(_rp), 3),
                       '선택지표=ρ(형태) 의 순위상관': round(float(_rr), 3),
                       '√PEHE 1위': _t4['√PEHE'].idxmin(),
                       'ρ 1위': _t4['ρ'].idxmax(), '실데이터 Qini 1위': _q.idxmax()})
    X106 = pd.DataFrame(_rowsX)
    X106.to_csv(os.path.join(DIR_OUT, '106X_선택이전.csv'), index=False, encoding='utf-8-sig')
    import shutil as _sh106
    os.makedirs(os.path.join(CONF_ROOT, 'paper', 'tab'), exist_ok=True)
    _sh106.copy(os.path.join(DIR_OUT, '106X_선택이전.csv'),
                os.path.join(CONF_ROOT, 'paper', 'tab', 'tabS_selection_transfer.csv'))
    print(X106.to_string(index=False))
    print(); print('ρ 의 상관 > √PEHE 의 상관이면: 진단 절차 4단계(「√PEHE 로 고르지 마라」)의 실증입니다.')
    REPORT['선택지표 이전(106X)'] = ' · '.join('%s: √PEHE %.2f vs ρ %.2f' % (
        r['결과'], r['선택지표=√PEHE 의 순위상관'], r['선택지표=ρ(형태) 의 순위상관']) for r in _rowsX)
except Exception:
    import traceback; traceback.print_exc()
    REPORT['선택지표 이전(106X)'] = '실패 — 트레이스백 참조'
step('§106X 선택 이전')

In [ ]:
%%time
# ═══ [107P] 수식 절의 실증 — 닫힌형 α* · 과신지수 Ω · 이력가측 상한 검증 ═══
# 원고의 Prop.2 / Cor.1 / Lemma 1 (paper/src/math_section.tex) 이 이 파이프라인의
# 실측과 맞는지 확인합니다. 기존 산출물 재분석만 — 추가 학습 없음.
try:
    _t4 = pd.read_csv(os.path.join(DIR_OUT, '100_표4_분해.csv')).set_index('모델')
    _93 = pd.read_csv(os.path.join(DIR_OUT, '93A_오라클상수_판별.csv'))
    _sdT = float(_93.loc[_93['데이터'] == 'empservicesyntv2', '오라클상수 sd(τ)'].mean())
    _mc = pd.read_csv(os.path.join(DIR_OUT, '07_모델비교.csv'))
    _sdH = _mc[_mc['데이터'] == 'empservicesyntv2'].groupby('모델')['τ sd'].mean()
    _oa = pd.read_csv(os.path.join(DIR_OUT, '92B_최적알파.csv'))
    _oa = _oa[(_oa['데이터'] == 'empservicesyntv2') & (_oa['지표'] == '√PEHE')].set_index('모델')
    _bg = pd.read_csv(os.path.join(DIR_OUT, '92B_알파주사_원자료.csv'))
    _bg = _bg[_bg['데이터'] == 'empservicesyntv2']

    _rowsP = []
    for _m in ('tlearner', 'cforest', 'tarnet', 'cdt'):
        if _m not in _t4.index or _m not in _sdH.index:
            continue
        _rho = float(_t4.loc[_m, 'ρ']); _sh = float(_sdH[_m])
        _pe1 = float(_t4.loc[_m, '√PEHE'])
        _aT = _rho * _sdT / _sh if _sh > 1e-9 else float('nan')      # Prop.2(a) 닫힌형
        _aG = float(_oa.loc[_m, '최적 α']) if _m in _oa.index else float('nan')
        # 격자에서 α*(이론) 에 가장 가까운 점의 √PEHE (이론 α 를 실제로 적용한 값)
        _g = _bg[_bg['모델'] == _m].groupby('α')['√PEHE'].mean()
        _peT = float(_g.iloc[(np.abs(_g.index.values - _aT)).argmin()]) if len(_g) and np.isfinite(_aT) else float('nan')
        _peG = float(_oa.loc[_m, '최적 값']) if _m in _oa.index else float('nan')
        _predImp = 100 * (1 - np.sqrt(max(0.0, 1 - _rho ** 2)))       # 예측 개선율(상한, 대비 오라클)
        _omega = (_sh / (2 * _rho * _sdT)) if _rho > 0 else float('inf')   # Cor.1 과신지수
        _rowsP.append({'모델': _m, 'ρ': round(_rho, 3), 'sd(τ̂)': round(_sh, 2),
                       'α*(이론)': round(_aT, 3) if np.isfinite(_aT) else _aT,
                       'α*(격자)': _aG,
                       '√PEHE@α*(이론)': round(_peT, 3) if np.isfinite(_peT) else _peT,
                       '√PEHE@α*(격자)': _peG,
                       '예측 최대개선% (1−√(1−ρ²))': round(_predImp, 2),
                       'Ω 과신지수': round(_omega, 2) if np.isfinite(_omega) else 'inf',
                       'Ω>1 (개인화 순손실)': (_omega > 1)})
    P107 = pd.DataFrame(_rowsP)
    P107.to_csv(os.path.join(DIR_OUT, '107P_이론검증.csv'), index=False, encoding='utf-8-sig')
    import shutil as _sh107
    os.makedirs(os.path.join(CONF_ROOT, 'paper', 'tab'), exist_ok=True)
    _sh107.copy(os.path.join(DIR_OUT, '107P_이론검증.csv'),
                os.path.join(CONF_ROOT, 'paper', 'tab', 'tabS_theory_check.csv'))
    print('── Prop.2 검증: 닫힌형 α* 이 격자탐색 최적을 재현하는가')
    print(P107.to_string(index=False))

    # Lemma 1: 모든 추정량의 ρ² ≤ R²_H (관측 특징 오라클)
    _r2h = float('nan')
    _pa = os.path.join(DIR_OUT, '100A_학습가능성.csv')
    if os.path.exists(_pa):
        _A = pd.read_csv(_pa)
        _row = _A[(_A['데이터'] == 'empservicesyntv2') & (_A['비고'].str.contains('관측', na=False))]
        if len(_row):
            _r2h = float(_row['R2_oracle'].iloc[0])
    _maxr2 = float((_t4['ρ'] ** 2).max())
    print()
    print('── Lemma 1 검증: max ρ² = %.4f  vs  R²_H(관측 특징 오라클) = %.4f  →  %s'
          % (_maxr2, _r2h, '상한 준수 ✓' if (not np.isfinite(_r2h)) or _maxr2 <= _r2h + 0.02
             else '⚠ 위반처럼 보임 — 두 값의 표본이 다름(추정오차) 을 본문에 명시'))
    print()
    print('읽는 법: α*(이론)≈α*(격자) 이고 √PEHE 도 일치하면 Prop.2 가 파이프라인에서 확인된 것 —')
    print('원고에 「닫힌형이 검증 스윕을 대체한다」로 쓰십시오. Ω>1 인 추정량은 Cor.1 의 판별식이')
    print('표 2 의 「오라클 상수에 패배」를 그대로 예측함을 보입니다.')
    REPORT['이론 검증(107P)'] = ' · '.join('%s: α*이론 %.2f/격자 %.2f · Ω %s' % (
        r['모델'], r['α*(이론)'], r['α*(격자)'], r['Ω 과신지수']) for r in _rowsP)
except Exception:
    import traceback; traceback.print_exc()
    REPORT['이론 검증(107P)'] = '실패 — 트레이스백 참조'
step('§107P 이론 검증')

In [ ]:
%%time
# ═══ [100F] 논문 그림 4매 — fig_inversion · fig_shrinkage · fig_confounder · fig_samplesize ═══
# [101] 그림 하나가 죽어도 나머지 그림·요약·알림은 계속 갑니다(그림별 try/except).
#   축 범위는 데이터 적응형 — 진짜 인과숲(econml)의 ρ 는 폴백 시절 수치와 크게 다릅니다.
import matplotlib
OUT99 = DIR_OUT
FIG99 = os.path.join(CONF_ROOT, 'paper', 'fig'); os.makedirs(FIG99, exist_ok=True)
C99 = {'tarnet': '#0072B2', 'cforest': '#E69F00', 'slearner': '#009E73',
       'cdt': '#D55E00', 'tlearner': '#E69F00', 'naive': '#777777'}
EN99 = {'naive': 'naive', 'slearner': 'S-learner', 'tlearner': 'T-learner',
        'cforest': 'causal forest', 'tarnet': 'TARNet', 'cdt': 'CDT'}
_FIG_OK = []

def _fig_try(name, fn):
    try:
        fn()
        _FIG_OK.append(name)
        print('  ✓', name)
    except Exception:
        import traceback
        print('  ✗ %s — 아래 원인. 다른 그림·요약은 계속 갑니다.' % name)
        traceback.print_exc()

with plt.rc_context({'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7.5,
                     'ytick.labelsize': 7.5, 'legend.fontsize': 7,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'pdf.fonttype': 42, 'ps.fonttype': 42}):
    COL_W = 3.45
    _93a = pd.read_csv(os.path.join(OUT99, '93A_오라클상수_판별.csv'))
    SD_TAU = float(_93a.loc[_93a['데이터'] == 'empservicesyntv2', '오라클상수 sd(τ)'].mean())

    def _fig_inversion():
        T4 = pd.read_csv(os.path.join(OUT99, '100_표4_분해.csv')).set_index('모델')
        RHO = T4['ρ'].to_dict(); PEHE99 = T4['√PEHE'].to_dict()
        pts = [m for m in ('slearner', 'cdt', 'cforest', 'tarnet', 'tlearner') if m in PEHE99]
        fig, ax = plt.subplots(figsize=(COL_W, 2.7))
        for m in pts:
            ax.scatter(RHO[m], PEHE99[m], s=42, color=C99[m], zorder=3,
                       edgecolors='white', linewidths=1.2)
            ax.annotate(EN99[m], (RHO[m], PEHE99[m]), textcoords='offset points',
                        xytext=(5, 4), fontsize=7.5, color='#333333')
        ax.axhline(SD_TAU, ls='--', lw=1, color='#777777')
        xmax = max(0.2, max(RHO[m] for m in pts) * 1.35)
        ymin = min(SD_TAU, min(PEHE99[m] for m in pts)) - 2
        ymax = max(PEHE99[m] for m in pts) + 3
        ax.text(xmax, SD_TAU - 0.5, 'oracle constant sd($\\tau$) = %.2f' % SD_TAU,
                ha='right', va='top', fontsize=7, color='#555555')
        ax.set_xlim(-0.03, xmax); ax.set_ylim(ymin, ymax)
        ax.set_xlabel(r'corr($\hat\tau$, $\tau$)  (recovered individual signal)')
        ax.set_ylabel(r'$\sqrt{\mathrm{PEHE}}$  (lower is "better")')
        fig.tight_layout()
        fig.savefig(os.path.join(FIG99, 'fig_inversion.pdf'))
        fig.savefig(os.path.join(FIG99, 'fig_inversion.png'), dpi=200); plt.close(fig)

    def _fig_shrinkage():
        B92 = pd.read_csv(os.path.join(OUT99, '92B_알파주사_원자료.csv'))
        fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.6))
        sim = B92[(B92['데이터'] == 'empservicesyntv2') & (B92['α'] <= 2.0)]
        base = sim[sim['모델'] == 'slearner'].groupby('α')['√PEHE'].mean().mean()
        for m in ('tlearner', 'cforest', 'tarnet', 'cdt'):
            a = sim[sim['모델'] == m].groupby('α')['√PEHE'].mean()
            if not len(a): continue
            axes[0].plot(a.index, a.values, 'o-', ms=3, lw=1.6, color=C99[m])
            axes[0].annotate(EN99[m], (a.index[-1], a.values[-1]), textcoords='offset points',
                             xytext=(4, {'cdt': -8}.get(m, -2)), fontsize=7, color=C99[m])
        axes[0].axhline(base, ls='--', lw=1, color='#777777')
        axes[0].text(1.15, base + 0.5, 'feasible constant baseline', fontsize=6.5,
                     color='#555555', ha='center', va='bottom')
        axes[0].axvline(1.0, ls=':', lw=0.8, color='#aaaaaa')
        axes[0].set_xlabel(r'shrinkage factor $\alpha$')
        axes[0].set_ylabel(r'$\sqrt{\mathrm{PEHE}}$ (simulator)')
        # α=0 은 상수 예측(순위 정보 없음) — 오른쪽 판에서 제외
        real = B92[(B92['데이터'] == 'empservicereal') & (B92['결과'] == 'Y3') & (B92['α'] >= 0.1)]
        for m in ('tarnet', 'cforest', 'tlearner', 'cdt'):
            a = real[real['모델'] == m].groupby('α')['Qini'].mean()
            if not len(a): continue
            axes[1].plot(a.index, a.values, 'o-', ms=3, lw=1.6, color=C99[m])
            axes[1].annotate(EN99[m], (a.index[-1], a.values[-1]), textcoords='offset points',
                             xytext=(4, -2), fontsize=7, color=C99[m])
        axes[1].axvline(1.0, ls=':', lw=0.8, color='#aaaaaa')
        axes[1].set_xlabel(r'shrinkage factor $\alpha$')
        axes[1].set_ylabel(r'Qini (real cohort, $Y_3$)')
        fig.tight_layout(w_pad=2.5)
        fig.savefig(os.path.join(FIG99, 'fig_shrinkage.pdf'))
        fig.savefig(os.path.join(FIG99, 'fig_shrinkage.png'), dpi=200); plt.close(fig)

    def _fig_confounder():
        SYN9 = pd.read_csv(os.path.join(OUT99, 'syn05_대리충분성.csv'))
        fig, ax = plt.subplots(figsize=(COL_W, 2.5))
        K = SYN9.iloc[:, 0]
        ax.plot(K, SYN9.iloc[:, 1], 'o-', lw=1.8, ms=4, color=C99['tarnet'])
        ax.plot(K, SYN9.iloc[:, 2], 's--', lw=1.6, ms=4, color=C99['cdt'])
        ax.annotate('U: leaves a trace in history', (K.iloc[-1], SYN9.iloc[-1, 1]),
                    textcoords='offset points', xytext=(-2, 7), ha='right',
                    fontsize=7.5, color=C99['tarnet'])
        ax.annotate('V: leaves no trace', (K.iloc[-1], SYN9.iloc[-1, 2]),
                    textcoords='offset points', xytext=(-2, 7), ha='right',
                    fontsize=7.5, color=C99['cdt'])
        ax.set_xscale('log'); ax.set_xticks(list(K))
        ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.set_xlabel('history length K (months)')
        ax.set_ylabel(r'$R^2$ recovered from history')
        ax.set_ylim(-0.01, max(0.23, float(SYN9.iloc[:, 1].max()) * 1.15))
        fig.tight_layout()
        fig.savefig(os.path.join(FIG99, 'fig_confounder.pdf'))
        fig.savefig(os.path.join(FIG99, 'fig_confounder.png'), dpi=200); plt.close(fig)

    def _fig_samplesize():
        NSZ = pd.read_csv(os.path.join(OUT99, '07_표본크기_판정.csv'))
        fig, ax = plt.subplots(figsize=(COL_W, 2.6))
        n = NSZ['n학습']; d = NSZ['cdt−기준 차']
        ax.axhline(0, lw=1, color='#777777')
        ax.fill_between(n, NSZ['하한'], NSZ['상한'], color=C99['cdt'], alpha=0.15, lw=0)
        ax.plot(n, d, 'o-', lw=1.8, ms=4, color=C99['cdt'])
        ax.set_xscale('log'); ax.set_xticks(list(n))
        ax.set_xticklabels([format(int(v), ',') for v in n], rotation=30)
        ax.set_xlabel('training individuals $n$')
        ax.set_ylabel(r'$\sqrt{\mathrm{PEHE}}$ advantage of CDT')
        ax.text(0.02, 0.96, 'CDT better', transform=ax.transAxes, fontsize=7,
                color='#333333', va='top')
        ax.text(0.02, 0.06, 'single-equation baseline better', transform=ax.transAxes,
                fontsize=7, color='#333333', va='bottom')
        fig.tight_layout()
        fig.savefig(os.path.join(FIG99, 'fig_samplesize.pdf'))
        fig.savefig(os.path.join(FIG99, 'fig_samplesize.png'), dpi=200); plt.close(fig)

    _fig_try('fig_inversion', _fig_inversion)
    _fig_try('fig_shrinkage', _fig_shrinkage)
    _fig_try('fig_confounder', _fig_confounder)
    _fig_try('fig_samplesize', _fig_samplesize)

REPORT['그림(논문 4매)'] = '성공 %d/4: %s' % (len(_FIG_OK), ' '.join(_FIG_OK) or '없음')
print(); print('paper/fig 저장:', sorted(f for f in os.listdir(FIG99) if f.startswith('fig_')))
step('§100F 그림')

In [ ]:
_envline = '기계 %s · %s · %s   라이브러리지문 %s' % (
    ENV.get('호스트'), ENV.get('장치'), ENV.get('gpu', ENV.get('os', '')),
    ENV.get('라이브러리지문'))
import json as _js2
_js2.dump(ENV, open(os.path.join(DIR_OUT, '_환경.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
_fail_n = len(FAILS) if 'FAILS' in dir() else 0
LINES = []
if _fail_n:
    # [79] v78 에서 실패 125건이 맨 아래 한 줄에 묻혀 있었습니다. 맨 위에 크게 답니다.
    LINES += ['!' * 78,
              '!!  실패 %d건 — 이 표를 논문에 쓰지 마십시오.' % _fail_n,
              '!!  %s' % ' · '.join('%s/%s' % (a, b) for a, b, _ in FAILS[:4]),
              '!!  %s' % (FAILS[0][2][:70] if FAILS else ''),
              '!' * 78]
LINES += ['=' * 78, '%s v%d 실행 결과 요약   코드지문 %s' % (NB_PATH, VERSION, CODE_HASH),
         '실행 시각 %s   총 소요 %.0f초' % (time.strftime('%Y-%m-%d %H:%M:%S'), time.time() - T0),
         _envline,
         '=' * 78]
# [84] 어느 코호트로 돈 표인지 요약에 박습니다. 이게 없으면 나중에 표만 보고는
#      2,000명짜리인지 16,000명짜리인지 알 수 없습니다.
try:
    _rs = dio.load_std('empservicereal')
    REPORT['실데이터 코호트'] = '%s · %d명 · 상태열 %d개' % (
        CFG.get('REAL_COHORT') or '기존', _rs.N, _rs.Ds)
    del _rs
except Exception as _e:
    REPORT['실데이터 코호트'] = '(못 읽음: %s)' % _e

for k, v in REPORT.items():
    LINES.append('  %-26s %s' % (k, v))
LINES.append('=' * 78)
print(chr(10).join(LINES))

files = []
for sub in ['datastd', 'datasynt', 'out', 'paper/tab', 'paper/fig']:
    d = os.path.join(CONF_ROOT, sub)
    if not os.path.isdir(d):
        continue
    n = mb = 0
    for root, _, fs in os.walk(d):
        for f in fs:
            n += 1; mb += os.path.getsize(os.path.join(root, f)) / 1e6
    files.append({'폴더': sub, '파일수': n, 'MB': round(mb, 1)})
fdf = pd.DataFrame(files); display(fdf)

# ── ★ 이 파일 하나만 보내 주시면 됩니다
LINES += ['', '[산출물]', fdf.to_string(index=False), '']
for nm in ['empservicereal'] + SYNTH:
    _s = dio.load_std(nm, 0); _m = _s.M.astype(bool)
    LINES.append('  %-20s N=%-6d T=%-4d 상태칸=%-4d 유효%5.1f%%  처치율 %.3f  Y %.3f±%.3f'
                 % (nm, _s.N, _s.T, _s.Ds, 100 * _m.mean(),
                    float(_s.A[..., 0][_m].mean()), float(_s.Y[..., 0][_m].mean()),
                    float(_s.Y[..., 0][_m].std())))
    del _s
if 'SYN' in dir():
    LINES += ['', '[합성 v2 검증]', SYN['수정확인'].to_string(index=False), '',
              SYN['교란'].to_string(index=False), '', SYN['대리충분성'].to_string(index=False)]
if 'ref' in dir():
    LINES += ['', '[벤치마크 정본 대조]', json.dumps(ref, ensure_ascii=False, indent=1, default=str)]
RPT = os.path.join(OUT, '00_결과요약.txt')
open(RPT, 'w', encoding='utf-8').write(chr(10).join(LINES))
print(chr(10) + '★ 이 파일 하나만 보내 주시면 됩니다:')
print('   ', os.path.relpath(RPT, CONF_ROOT), '  (%.1f KB)' % (os.path.getsize(RPT) / 1024))

In [ ]:
# ═══ 알림 — 토큰은 환경변수로. 공개 파이프라인에 비밀·실명을 넣지 않습니다 ═══
# 사용법: 셸에서  export TG_TOKEN=...  TG_CHAT=...   (없으면 조용히 건너뜁니다)
def _tg100(t):
    import requests
    _tok = os.environ.get('TG_TOKEN'); _cid = os.environ.get('TG_CHAT')
    if not _tok or not _cid:
        return {'skip': 'TG_TOKEN/TG_CHAT 미설정 — 전송 생략'}
    return requests.post('https://api.telegram.org/bot%s/sendMessage' % _tok,
                         data={'chat_id': _cid, 'text': t}).json()
try:
    _msg = 'icml_run%d\n' % VERSION + (chr(10).join(LINES) if 'LINES' in dir() else '끝')
except Exception:
    _msg = 'icml_run100 실행 끝'
try:
    for _i in range(0, min(len(_msg), 12000), 3800):
        _r100 = _tg100(_msg[_i:_i + 3800])
        if isinstance(_r100, dict) and _r100.get('skip'):
            print(_r100['skip']); break
    else:
        print('텔레그램 전송 완료 (%d자)' % len(_msg))
except Exception as _e:
    print('텔레그램 전송 실패 — 실행 결과와는 무관합니다: %s' % _e)